In [ ]:

!pip -q install pandas numpy scikit-learn scipy

In [ ]:

import pandas as pd
import numpy as np

from google.colab import files

uploaded = files.upload()
file_path = next(iter(uploaded))

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

Saving lmp_final - lmp_semantic_review(1).csv to lmp_final - lmp_semantic_review(1) (1).csv
Dataset shape: (357, 8)

Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment']

First 5 rows:


,ID,Original,Language,Domain,Source,Simplified,LMP score,Human_Assessment
0,en_tos_001,"As a result, these Terms of Service help defin...",en,tos_privacy,google_tos,"Because of this, these Terms of Service help d...",10,Legal meaning and key details preserved — Key ...
1,en_tos_002,"Our services are designed to work together, ma...",en,tos_privacy,google_tos,Our services are designed to work together so ...,9,Legal meaning and key details preserved — Key ...
2,en_tos_003,"We also make various policies, help centers, a...",en,tos_privacy,google_tos,"We also provide various policies, help centers...",9,Legal meaning and key details preserved — Key ...
3,en_tos_004,"comply with applicable laws, including export ...",en,tos_privacy,google_tos,"Comply with applicable laws, including export ...",10,Legal meaning and key details preserved — Key ...
4,en_tos_005,using our services (including the content they...,en,tos_privacy,google_tos,Do not use our services (including the content...,10,Legal meaning and key details preserved — Key ...


In [ ]:
# Rename the score column to the name our experiment code expects
df = df.rename(columns={"LMP score": "LMP"})

required_columns = ["Original", "Simplified", "LMP"]
missing = [col for col in required_columns if col not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["LMP"] = pd.to_numeric(df["LMP"], errors="coerce")

print("Dataset shape:", df.shape)
print("Missing scores:", df["LMP"].isna().sum())
print("Score distribution:")
print(df["LMP"].value_counts().sort_index())

Dataset shape: (357, 8)
Missing scores: 0
Score distribution:
LMP
4       4
5       5
6      17
7      34
8      71
9     102
10    124
Name: count, dtype: int64


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr
import numpy as np
import pandas as pd

# Use the exact dataset and split used by the XLM-R experiment
baseline_df = model_df.reset_index(drop=True).copy()

all_idx = np.arange(len(baseline_df))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42
)

# Match the original two-stage split: only train_part_idx is training data
train_part_idx, val_idx = train_test_split(
    train_idx, test_size=0.15, random_state=42
)

train_df = baseline_df.iloc[train_part_idx].copy()
test_df = baseline_df.iloc[test_idx].copy()

# Fit vocabulary and IDF weights on training sentences ONLY
train_sentences = (
    train_df["Original"].astype(str).tolist()
    + train_df["Simplified"].astype(str).tolist()
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000
)
vectorizer.fit(train_sentences)

# Transform train and test pairs using the training vocabulary
def pair_cosines(frame):
    originals = vectorizer.transform(frame["Original"].astype(str))
    simplified = vectorizer.transform(frame["Simplified"].astype(str))
    return np.asarray(
        originals.multiply(simplified).sum(axis=1)
    ).ravel()

train_cosine = pair_cosines(train_df)
test_cosine = pair_cosines(test_df)

y_train = train_df["LMP"].astype(float).to_numpy()
y_test = test_df["LMP"].astype(float).to_numpy()

# Calibrate using training data only
calibrator = LinearRegression()
calibrator.fit(train_cosine.reshape(-1, 1), y_train)

predictions = np.clip(
    calibrator.predict(test_cosine.reshape(-1, 1)), 1, 10
)

print("LEAKAGE-FREE TF-IDF BASELINE")
print("-" * 45)
print(f"Training pairs: {len(train_df)}")
print(f"Validation pairs (unused by baseline): {len(val_idx)}")
print(f"Test pairs: {len(test_df)}")
print(f"Pearson (cosine vs LMP): {pearsonr(test_cosine, y_test).statistic:.4f}")
print(f"Spearman (cosine vs LMP): {spearmanr(test_cosine, y_test).statistic:.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, predictions):.4f}")

tfidf_results = test_df[
    ["ID", "Original", "Simplified", "Language", "Domain", "LMP"]
].copy()
tfidf_results["TFIDF_Cosine"] = test_cosine
tfidf_results["Predicted_LMP"] = predictions
display(tfidf_results.head(10))

tfidf_results.to_csv("/content/tfidf_clean_test_predictions.csv", index=False)

LEAKAGE-FREE TF-IDF BASELINE
---------------------------------------------
Training pairs: 242
Validation pairs (unused by baseline): 43
Test pairs: 72
Pearson (cosine vs LMP): 0.6034
Spearman (cosine vs LMP): 0.7085
Test RMSE: 1.1634
Test MAE: 0.7924


,ID,Original,Simplified,Language,Domain,LMP,TFIDF_Cosine,Predicted_LMP
222,hi_tos_018,दूसरों को या खुद को नुकसान न पहुंचाना या दूसरों के साथ या अपने साथ बुरा बर्ताव न करना (या न तो ऐसे नुकसान या बुरे बर्ताव की धमकी देना या इन्हें बढ़ावा देना).,दूसरों को या खुद को नुकसान न पहुंचाना और बुरा बर्ताव न करना।,hi,tos_privacy,6.0,0.782214,9.167916
42,en_tos_043,"If we are unable to resolve a dispute within sixty (60) days after the Notice of Dispute is received, you or we may commence arbitration.","If we cannot resolve a dispute within 60 days after receiving the Notice of Dispute, you or we can start arbitration.",en,tos_privacy,9.0,0.807597,9.276008
291,hi_rental_006,अनुबंध समाप्त करने के लिए \_\_\_\_\_\_ पूर्व लिखित सूचना देनी होगी।,अनुबंध समाप्त करने के लिए पूर्व लिखित सूचना देनी होगी।,hi,rental,10.0,1.000000,10.000000
300,hi_rental_015,बातचीत से समाधान न होने पर \_\_\_\_\_\_ के सक्षम न्यायालय में कार्यवाही की जा सकती है।,बातचीत से समाधान न होने पर कार्यवाही की जा सकती है \_\_\_\_\_\_ के सक्षम न्यायालय में।,hi,rental,8.0,0.937295,9.828324
56,en_tos_057,These business or user terms and conditions are available on the respective Participating Platforms.,These terms and conditions are available on the respective Participating Platforms.,en,tos_privacy,9.0,0.862690,9.510620
335,hi_rental_050,इसलिए अब प्रथम पक्ष व द्वितीय पक्ष उक्त किरायानामा दिनांक ————— तक के लिये निष्पादित करते हैं।,इसलिए अब प्रथम पक्ष और द्वितीय पक्ष किराया तक दिनांक ————— तक देते हैं।,hi,rental,8.0,0.612261,8.444175
15,en_tos_016,"Regardless of where your information is processed, we apply the same protections described in this policy.","Regardless of processing location, we apply the same protections as described in this policy.",en,tos_privacy,9.0,0.735272,8.968015
57,en_tos_058,"For the respective contents of the Participating Platforms, the respective affiliate of Paytm is solely responsible only for those content that it actually operates.","For the content of the Participating Platforms, the relevant Paytm affiliate is responsible only for the content it operates.",en,tos_privacy,9.0,0.606212,8.418418
245,hi_tos_041,"ऐसा होने पर या इसकी वजह से, कोई तीसरा पक्ष कानूनी कार्रवाइयां (इसमें सरकारी अधिकारियों की कार्रवाइयां शामिल हैं) कर सकता है.","ऐसा होने पर या इसकी वजह से, कोई तीसरा पक्ष कानूनी कार्रवाए कर सकता है, इसमें सरकारी अधिकारियों की कार्रवाए भी शामिल हैं.",hi,tos_privacy,9.0,0.918042,9.746337
124,en_tos_125,We decide how long we need information on a case-by-case basis. Here's what we consider when we decide:,We decide how long to keep information on a case-by-case basis. We consider these factors:,en,tos_privacy,9.0,0.729535,8.943584


In [ ]:

!pip -q install sentence-transformers scipy scikit-learn

In [ ]:

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Keep only complete pairs with valid reference scores
data = df[["Original", "Simplified", "LMP"]].dropna().copy()
data["LMP"] = pd.to_numeric(data["LMP"], errors="coerce")
data = data.dropna(subset=["LMP"])
data = data[data["LMP"].between(1, 10)].reset_index(drop=True)

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

original_embeddings = model.encode(
    data["Original"].astype(str).tolist(),
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

simplified_embeddings = model.encode(
    data["Simplified"].astype(str).tolist(),
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

data["Semantic_Cosine"] = np.sum(
    original_embeddings * simplified_embeddings,
    axis=1
)

print("Sentence pairs:", len(data))
print("\nSemantic similarity summary:")
print(data["Semantic_Cosine"].describe())
display(data[["Original", "Simplified", "LMP", "Semantic_Cosine"]].head())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (7 outputs left out)

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Sentence pairs: 357

Semantic similarity summary:
count    357.000000
mean       0.933004
std        0.069801
min        0.539643
25%        0.906406
50%        0.956029
75%        0.981200
max        1.000000
Name: Semantic_Cosine, dtype: float64


,Original,Simplified,LMP,Semantic_Cosine
0,"As a result, these Terms of Service help defin...","Because of this, these Terms of Service help d...",10,0.997033
1,"Our services are designed to work together, ma...",Our services are designed to work together so ...,9,0.958247
2,"We also make various policies, help centers, a...","We also provide various policies, help centers...",9,0.956029
3,"comply with applicable laws, including export ...","Comply with applicable laws, including export ...",10,0.971900
4,using our services (including the content they...,Do not use our services (including the content...,10,0.814186


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr
import numpy as np

indices = np.arange(len(data))
train_idx, test_idx = train_test_split(
    indices, test_size=0.20, random_state=42
)

X = data[["Semantic_Cosine"]].values
y = data["LMP"].values

calibrator = LinearRegression()
calibrator.fit(X[train_idx], y[train_idx])

predictions = np.clip(calibrator.predict(X[test_idx]), 1, 10)
y_test = y[test_idx]
cosine_test = X[test_idx, 0]

print("BASELINE 2: MULTILINGUAL SENTENCE TRANSFORMER")
print("-" * 50)
print(f"Test pairs: {len(test_idx)}")
print(f"Pearson correlation: {pearsonr(cosine_test, y_test).statistic:.4f}")
print(f"Spearman correlation: {spearmanr(cosine_test, y_test).statistic:.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, predictions):.4f}")

BASELINE 2: MULTILINGUAL SENTENCE TRANSFORMER
--------------------------------------------------
Test pairs: 72
Pearson correlation: 0.5875
Spearman correlation: 0.6573
Test RMSE: 1.1020
Test MAE: 0.8461


In [ ]:

!git clone https://github.com/GRAAL-Research/JUDGEBERT.git
%cd JUDGEBERT
!find article_src -maxdepth 3 -type f

Cloning into 'JUDGEBERT'...
remote: Enumerating objects: 1096, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 1096 (delta 24), reused 45 (delta 9), pack-reused 981 (from 1)
Receiving objects: 100% (1096/1096), 33.71 MiB | 11.32 MiB/s, done.
Resolving deltas: 100% (678/678), done.
Updating files: 100% (98/98), done.
/content/JUDGEBERT
article_src/dataset_analysis/dataset_analysis.py
article_src/dataset_analysis/__init__.py
article_src/dataset_analysis/dataset_analysis_eur-lex-sum_fr.py
article_src/dataset_analysis/dataset_analysis_fr.py
article_src/src/generate_dataset_only_requirements.txt
article_src/src/tools.py
article_src/src/sweep_eval.sh
article_src/src/few_shot_training.py
article_src/src/sweep_flaubert_base_da.sh
article_src/src/models.py
article_src/src/evaluate_tools.py
article_src/src/__init__.py
article_src/src/evaluation_llm.py
article_src/src/training_only_requirements.txt
article_src/src/metrics/metr

In [ ]:

%cd /content/JUDGEBERT
!sed -n '1,240p' article_src/src/models.py

/content/JUDGEBERT
# All LLM we want to evaluate
model_names = [
    "jpacifico/Chocolatine-14B-Instruct-DPO-v1.2",
    "jpacifico/Chocolatine-14B-Instruct-DPO-v1.2",
    "jpacifico/French-Alpaca-Llama3-8B-Instruct-v1.0",
    "OpenLLM-France/Lucie-7B",
    "OpenLLM-France/Lucie-7B-Instruct",
    "OpenLLM-France/Lucie-7B-Instruct-human-data",
    "OpenLLM-France/Claire-7B-FR-Instruct-0.1",
    "unsloth/llama-3-8B-bnb-4bit",
    "unsloth/llama-3-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-1B-bnb-4bit",
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    "bigscience/bloom-560m",
    "bigscience/bloom-1b1",
    "bigscience/bloom-1b7",
    "bigscience/bloom-7b1",
    "bigscience/bloomz-560m",
    "bigscience/bloomz-1b1",
    "unsloth/Qwen2.5-0.5B-bnb-4bit",
    "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    "unsloth/Qw

In [ ]:

!sed -n '1,240p' article_src/src/factory_model_tokenizer.py
!sed -n '1,240p' article_src/src/factory_dataset.py

from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    BitsAndBytesConfig,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
)
from transformers import logging

from unsloth import FastLanguageModel

logging.set_verbosity_warning()


def model_tokenizer_factory(model_name, device, token, seed: int = 42):
    if "gpt" in model_name:
        model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
        tokenizer = GPT2TokenizerFast.from_pretrained(model_name)

        # To handle padding when prompting
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"
    elif "bert" in model_name.lower():
        # BERT based model (along with RoBERTa).
        model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
        model.eval()
        tokenizer = AutoTokenizer.from_pretrained(model_name)
    elif (
        "bloom" in model_name.lower()
        or "moe" in model_name.

In [ ]:
%cd /content/JUDGEBERT
!sed -n '1,260p' article_src/src/few_shot_training.py

/content/JUDGEBERT
import argparse
import logging
import os
import pickle
import shutil

import wandb
from poutyne import set_seeds
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    EarlyStoppingCallback,
)

from metrics.metrics import (
    compute_metrics,
    eval_compute_metrics_identical,
    eval_compute_metrics_unrelated,
)
from factory_dataset import dataset_factory
from tools import (
    bool_parse,
)

log = logging.getLogger("pytorch_lightning")
log.propagate = False
log.setLevel(logging.ERROR)

# Boilerplate params
os.environ["WANDB_PROJECT"] = "KILL"  # To control the Wandb project name in HuggingFace

parser = argparse.ArgumentParser()
parser.add_argument(
    "--seed",
    type=int,
    default=42,
    help="The seed to use for training.",
)

parser.add_argument(
    "--root",
    type=str,
    default=".",
    help="Root directory.",
)

parser.add_argument(
    "--da

In [ ]:

!sed -n '1,240p' article_src/src/evaluation_loop.py
!sed -n '1,240p' article_src/src/tools.py

import os
from functools import partial

import numpy as np
import torch
import wandb
from dotenv import dotenv_values

from factory_dataset import dataset_factory
from evaluate_tools import evaluation_llm_prompting
from factory_model_tokenizer import model_tokenizer_factory
from metrics.metrics import (
    compute_metrics,
    eval_compute_metrics_identical,
    eval_compute_metrics_unrelated,
)

# To make Wandb silent
os.environ["WANDB_SILENT"] = "true"


def evaluation_loop(
    model_name: str,
    data_augmentation: bool,
):
    params = {"batched": True, "batch_size": 16}
    output_dir = "predictions"
    os.makedirs(output_dir, exist_ok=True)

    secrets = dotenv_values(".env")

    huggingface_token = secrets["huggingface_token"]

    device = torch.device("cuda")

    clean_model_name = model_name.split("/")[-1]

    wandb.init(
        project=f"kil_llm_eval",
        config={"model_name": model_name, "data_augmentation": data_augmentation},
        name=f"{clean_model_nam

In [ ]:

%cd /content/JUDGEBERT
!grep -n -E "checkpoint|TrainingArguments|tokenize|compute_metrics|label|num_epoch|learning_rate|def main|def train" article_src/src/few_shot_training.py

/content/JUDGEBERT
11:    TrainingArguments,
19:    compute_metrics,
20:    eval_compute_metrics_identical,
21:    eval_compute_metrics_unrelated,
68:    help="The model name to use as the starting model checkpoint.",
76:num_epoch = args.epochs
86:checkpoint = args.model_name
87:tokenizer = AutoTokenizer.from_pretrained(checkpoint)
90:def tokenize_function(example):
91:    return tokenizer(
96:tokenized_legal_annotation_dataset = legal_precision_annotation_dataset.map(
97:    tokenize_function, batched=True
99:tokenize_holdout_identical_dataset = holdout_identical_dataset.map(
100:    tokenize_function, batched=True
102:tokenize_holdout_unrelated_dataset = holdout_unrelated_dataset.map(
103:    tokenize_function, batched=True
105:data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
107:output_dir = f"legal_meaning_bert_train_{checkpoint}_WithDA_{data_augmentation}_{seed}"
108:training_args = TrainingArguments(
114:    num_train_epochs=num_epoch,
115:    save_total_limit=num_epo

In [ ]:

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nLanguage distribution:")
print(df["Language"].value_counts(dropna=False))
print("\nDomain distribution:")
print(df["Domain"].value_counts(dropna=False))
print("\nLMP score distribution:")
print(df["LMP"].value_counts(dropna=False).sort_index())
print("\nMissing values:")
print(df[["Original", "Simplified", "Language", "Domain", "LMP"]].isna().sum())

Dataset shape: (357, 8)
Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP', 'Human_Assessment']

Language distribution:
Language
en    205
hi    152
Name: count, dtype: int64

Domain distribution:
Domain
tos_privacy    227
rental         130
Name: count, dtype: int64

LMP score distribution:
LMP
4       4
5       5
6      17
7      34
8      71
9     102
10    124
Name: count, dtype: int64

Missing values:
Original      0
Simplified    0
Language      0
Domain        0
LMP           0
dtype: int64


In [ ]:

from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Keep the required columns and clean the labels
model_df = df[
    ["ID", "Original", "Simplified", "Language", "Domain", "LMP"]
].copy()

model_df["LMP"] = pd.to_numeric(model_df["LMP"], errors="coerce")
model_df = model_df.dropna(
    subset=["Original", "Simplified", "Language", "Domain", "LMP"]
).reset_index(drop=True)

model_df["LMP"] = model_df["LMP"].astype(float)

# Reproducible 80/20 split
indices = np.arange(len(model_df))
train_idx, test_idx = train_test_split(
    indices, test_size=0.20, random_state=42
)

train_df = model_df.iloc[train_idx].reset_index(drop=True)
test_df = model_df.iloc[test_idx].reset_index(drop=True)

print("Training pairs:", len(train_df))
print("Test pairs:", len(test_df))
print("\nTraining language counts:")
print(train_df["Language"].value_counts())
print("\nTest language counts:")
print(test_df["Language"].value_counts())
print("\nTraining domain counts:")
print(train_df["Domain"].value_counts())
print("\nTest domain counts:")
print(test_df["Domain"].value_counts())

Training pairs: 285
Test pairs: 72

Training language counts:
Language
en    164
hi    121
Name: count, dtype: int64

Test language counts:
Language
en    41
hi    31
Name: count, dtype: int64

Training domain counts:
Domain
tos_privacy    177
rental         108
Name: count, dtype: int64

Test domain counts:
Domain
tos_privacy    50
rental         22
Name: count, dtype: int64


In [ ]:

!pip -q install transformers accelerate

In [ ]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

CHECKPOINT = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=1
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Checkpoint:", CHECKPOINT)
print("Device:", device)
print("Regression outputs:", model.config.num_labels)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Checkpoint: xlm-roberta-base
Device: cpu
Regression outputs: 1


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in ["JUDGEBERT", ".cache"]]
    for file in files:
        if file.lower().endswith((".csv", ".xlsx", ".json", ".jsonl")):
            print(os.path.join(root, file))

/content/lmp_final - lmp_semantic_review(1).csv
/content/.config/.last_update_check.json
/content/sample_data/anscombe.json
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_test.csv


In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/lmp_final - lmp_semantic_review(1).csv"
)

if "LMP score" in df.columns and "LMP" not in df.columns:
    df = df.rename(columns={"LMP score": "LMP"})

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (357, 8)
Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP', 'Human_Assessment']


,ID,Original,Language,Domain,Source,Simplified,LMP,Human_Assessment
0,en_tos_001,"As a result, these Terms of Service help defin...",en,tos_privacy,google_tos,"Because of this, these Terms of Service help d...",10,Legal meaning and key details preserved — Key ...
1,en_tos_002,"Our services are designed to work together, ma...",en,tos_privacy,google_tos,Our services are designed to work together so ...,9,Legal meaning and key details preserved — Key ...
2,en_tos_003,"We also make various policies, help centers, a...",en,tos_privacy,google_tos,"We also provide various policies, help centers...",9,Legal meaning and key details preserved — Key ...
3,en_tos_004,"comply with applicable laws, including export ...",en,tos_privacy,google_tos,"Comply with applicable laws, including export ...",10,Legal meaning and key details preserved — Key ...
4,en_tos_005,using our services (including the content they...,en,tos_privacy,google_tos,Do not use our services (including the content...,10,Legal meaning and key details preserved — Key ...


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

model_df = df[
    ["ID", "Original", "Simplified", "Language", "Domain", "LMP"]
].copy()

model_df["LMP"] = pd.to_numeric(model_df["LMP"], errors="coerce")
model_df = model_df.dropna(
    subset=["Original", "Simplified", "Language", "Domain", "LMP"]
).reset_index(drop=True)

model_df["LMP"] = model_df["LMP"].astype(float)

train_idx, test_idx = train_test_split(
    np.arange(len(model_df)),
    test_size=0.20,
    random_state=42
)

train_df = model_df.iloc[train_idx].reset_index(drop=True)
test_df = model_df.iloc[test_idx].reset_index(drop=True)

train_part, val_part = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42
)

train_part = train_part.reset_index(drop=True)
val_part = val_part.reset_index(drop=True)

print("Total:", len(model_df))
print("Training:", len(train_part))
print("Validation:", len(val_part))
print("Test:", len(test_df))

Total: 357
Training: 242
Validation: 43
Test: 72


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CHECKPOINT = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=1
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Device:", device)
print("Model ready for LMP regression.")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda
Model ready for LMP regression.


In [ ]:
!pip -q install datasets

In [ ]:
from datasets import Dataset

MAX_LENGTH = 256

def tokenize_pairs(batch):
    return tokenizer(
        batch["Original"],
        batch["Simplified"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_data = Dataset.from_pandas(
    train_part[["Original", "Simplified", "LMP"]]
)
val_data = Dataset.from_pandas(
    val_part[["Original", "Simplified", "LMP"]]
)
test_data = Dataset.from_pandas(
    test_df[["Original", "Simplified", "LMP"]]
)

train_data = train_data.map(tokenize_pairs, batched=True)
val_data = val_data.map(tokenize_pairs, batched=True)
test_data = test_data.map(tokenize_pairs, batched=True)

train_data = train_data.rename_column("LMP", "labels")
val_data = val_data.rename_column("LMP", "labels")
test_data = test_data.rename_column("LMP", "labels")

print("Training:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))
print("Columns:", train_data.column_names)

Map:   0%|          | 0/242 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Map:   0%|          | 0/72 [00:00<?, ? examples/s]

Training: 242
Validation: 43
Test: 72
Columns: ['Original', 'Simplified', 'labels', 'input_ids', 'attention_mask']


In [ ]:
!pip -q install accelerate scikit-learn scipy

In [ ]:
import numpy as np
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.asarray(predictions).reshape(-1)
    labels = np.asarray(labels).reshape(-1)

    pearson = pearsonr(labels, predictions)[0] if len(labels) > 1 else 0.0
    spearman = spearmanr(labels, predictions)[0] if len(labels) > 1 else 0.0

    return {
        "rmse": float(np.sqrt(mean_squared_error(labels, predictions))),
        "mae": float(mean_absolute_error(labels, predictions)),
        "pearson": float(pearson),
        "spearman": float(spearman),
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="/content/judgebert_lmp_results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_rmse",
    greater_is_better=False,
    fp16=True,
    logging_strategy="epoch",
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Training configuration ready.")
print("GPU:", torch.cuda.get_device_name(0))
print("Epochs:", training_args.num_train_epochs)

Training configuration ready.
GPU: Tesla T4
Epochs: 5


In [ ]:
train_result = trainer.train()

print("\nTraining completed!")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation RMSE:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Rmse,Mae,Pearson,Spearman
1,61.921245,26.063982,5.105290,4.921239,0.113955,0.136967
2,20.119991,6.526847,2.554769,2.321039,-0.347197,-0.360353
3,5.774424,1.994349,1.412214,1.194222,-0.002032,0.059111
4,2.039555,1.588119,1.260206,0.928688,0.565164,0.333494
5,1.690267,1.505611,1.227034,0.894713,0.569618,0.358591


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed!
Best checkpoint: /content/judgebert_lmp_results/checkpoint-155
Best validation RMSE: 1.2270335368148846


In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

# Evaluate once on the held-out test set
test_output = trainer.predict(test_data)

y_true = np.asarray(test_output.label_ids).reshape(-1)
y_pred = np.asarray(test_output.predictions).reshape(-1)

# LMP scores are on a 1–10 scale; clip predictions to that range
y_pred = np.clip(y_pred, 1, 10)

test_results = {
    "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
    "MAE": mean_absolute_error(y_true, y_pred),
    "Pearson": pearsonr(y_true, y_pred).statistic,
    "Spearman": spearmanr(y_true, y_pred).statistic,
}

print("JUDGEBERT-inspired XLM-R test results")
for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

JUDGEBERT-inspired XLM-R test results
RMSE: 1.2326
MAE: 0.9753
Pearson: 0.5780
Spearman: 0.4165


In [ ]:
SAVE_PATH = "/content/judgebert_lmp_final"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved to:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/judgebert_lmp_final


In [ ]:
import numpy as np
import pandas as pd
import torch

# Use the best model selected during training
trainer.model.eval()

all_data = Dataset.from_pandas(
    model_df[["Original", "Simplified", "LMP", "Language", "Domain"]]
)

def tokenize_all(batch):
    return tokenizer(
        batch["Original"],
        batch["Simplified"],
        truncation=True,
        max_length=256,
    )

all_data = all_data.map(tokenize_all, batched=True)
all_data = all_data.rename_column("LMP", "labels")

all_predictions = trainer.predict(all_data)
predicted_scores = np.asarray(
    all_predictions.predictions
).reshape(-1)

# Keep predictions within the project's 1–10 score scale
predicted_scores = np.clip(predicted_scores, 1, 10)

results_df = model_df.copy()
results_df["Predicted_LMP"] = np.round(predicted_scores, 2)

# Provisional threshold; document and justify it in the paper
THRESHOLD = 8.0

results_df["Preservation_Decision"] = np.where(
    results_df["Predicted_LMP"] >= THRESHOLD,
    "Preserved",
    "Review: possible meaning loss",
)

display(results_df.head(10))
print("Total predictions:", len(results_df))

Map:   0%|          | 0/357 [00:00<?, ? examples/s]

,ID,Original,Simplified,Language,Domain,LMP,Predicted_LMP,Preservation_Decision
0,en_tos_001,"As a result, these Terms of Service help defin...","Because of this, these Terms of Service help d...",en,tos_privacy,10.0,8.91,Preserved
1,en_tos_002,"Our services are designed to work together, ma...",Our services are designed to work together so ...,en,tos_privacy,9.0,9.05,Preserved
2,en_tos_003,"We also make various policies, help centers, a...","We also provide various policies, help centers...",en,tos_privacy,9.0,8.92,Preserved
3,en_tos_004,"comply with applicable laws, including export ...","Comply with applicable laws, including export ...",en,tos_privacy,10.0,8.96,Preserved
4,en_tos_005,using our services (including the content they...,Do not use our services (including the content...,en,tos_privacy,10.0,8.99,Preserved
5,en_tos_006,We need your permission if your intellectual p...,We need your permission if your intellectual p...,en,tos_privacy,10.0,9.02,Preserved
6,en_tos_007,This license lasts for as long as your content...,This license lasts as long as your content is ...,en,tos_privacy,10.0,8.98,Preserved
7,en_tos_008,Google services may frequently use the Interne...,Google services may use the Internet frequentl...,en,tos_privacy,9.0,8.88,Preserved
8,en_tos_009,Some of our services include software that’s o...,Some of our services include software offered ...,en,tos_privacy,9.0,8.90,Preserved
9,en_tos_010,"The information Google collects, and how that ...",The information Google collects and how it is ...,en,tos_privacy,9.0,8.87,Preserved


Total predictions: 357


In [ ]:
OUTPUT_PATH = "/content/lmp_prediction_results.csv"

results_df.to_csv(OUTPUT_PATH, index=False)

print("Results exported:", OUTPUT_PATH)

Results exported: /content/lmp_prediction_results.csv


In [ ]:
print("Prediction distribution:")
print(results_df["Preservation_Decision"].value_counts())

print("\nPredicted LMP statistics:")
print(results_df["Predicted_LMP"].describe())

print("\nLanguage-wise results:")
display(
    results_df.groupby("Language")["Predicted_LMP"]
    .agg(["count", "mean", "std"])
    .round(2)
)

print("\nDomain-wise results:")
display(
    results_df.groupby("Domain")["Predicted_LMP"]
    .agg(["count", "mean", "std"])
    .round(2)
)

Prediction distribution:
Preservation_Decision
Preserved                        344
Review: possible meaning loss     13
Name: count, dtype: int64

Predicted LMP statistics:
count    357.000000
mean       8.863221
std        0.329144
min        7.340000
25%        8.780000
50%        8.980000
75%        9.100000
max        9.150000
Name: Predicted_LMP, dtype: float64

Language-wise results:


,count,mean,std
Language,,,
en,205,8.70,0.35
hi,152,9.09,0.06



Domain-wise results:


,count,mean,std
Domain,,,
rental,130,8.77,0.45
tos_privacy,227,8.92,0.21


In [ ]:
difficult_cases = results_df.copy()

difficult_cases["Score_Difference"] = (
    difficult_cases["Predicted_LMP"] - difficult_cases["LMP"]
).abs()

difficult_cases = difficult_cases.sort_values(
    "Score_Difference", ascending=False
)

display(
    difficult_cases[
        [
            "Original",
            "Simplified",
            "Language",
            "Domain",
            "LMP",
            "Predicted_LMP",
            "Score_Difference",
        ]
    ].head(20)
)

,Original,Simplified,Language,Domain,LMP,Predicted_LMP,Score_Difference
205,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेव...",उपयोग करते समय सहमत होने वाले नियमों का वर्णन ...,hi,tos_privacy,5.0,9.10,4.10
266,हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव क...,"हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धो...",hi,tos_privacy,5.0,8.99,3.99
191,The Landlord may rel et the Premises or any pa...,The Landlord may sublet the Premises or part o...,en,rental,4.0,7.93,3.93
202,Within 30 days of the termination of this leas...,Within 30 days after the lease ends and Tenant...,en,rental,4.0,7.61,3.61
180,It is the intent of this Lease and agreed to b...,It is agreed that the Tenant will pay the Base...,en,rental,4.0,7.60,3.60
183,Should the Tenant remain in possession of the ...,If the Tenant stays in the Premises with the L...,en,rental,4.0,7.34,3.34
280,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाण...,PhonePe आपसे विशिष्ट जानकारी का अनुरोध कर सकता...,hi,tos_privacy,6.0,9.13,3.13
265,हम व्यक्तिगत जानकारी को भारत के भीतर संग्रहीत ...,हम भारत में व्यक्तिगत जानकारी को संग्रहीत करते...,hi,tos_privacy,6.0,9.12,3.12
116,To use less information that's connected to in...,"To use less personal information, we sometimes...",en,tos_privacy,5.0,8.11,3.11
332,अचल संपति को किराये पर देने की बाबत किसी प्रका...,अचल संपत्ति को किराये पर देने की कोई रूकावट नह...,hi,rental,6.0,9.09,3.09


In [ ]:
difficult_cases = results_df.copy()

difficult_cases["Score_Difference"] = (
    difficult_cases["Predicted_LMP"] - difficult_cases["LMP"]
).abs()

difficult_cases = difficult_cases.sort_values(
    "Score_Difference",
    ascending=False
)

display(
    difficult_cases[
        [
            "ID",
            "Original",
            "Simplified",
            "Language",
            "Domain",
            "LMP",
            "Predicted_LMP",
            "Score_Difference",
        ]
    ].head(20)
)

,ID,Original,Simplified,Language,Domain,LMP,Predicted_LMP,Score_Difference
205,hi_tos_001,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेव...",उपयोग करते समय सहमत होने वाले नियमों का वर्णन ...,hi,tos_privacy,5.0,9.10,4.10
266,hi_tos_062,हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव क...,"हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धो...",hi,tos_privacy,5.0,8.99,3.99
191,en_rental_046,The Landlord may rel et the Premises or any pa...,The Landlord may sublet the Premises or part o...,en,rental,4.0,7.93,3.93
202,en_rental_057,Within 30 days of the termination of this leas...,Within 30 days after the lease ends and Tenant...,en,rental,4.0,7.61,3.61
180,en_rental_035,It is the intent of this Lease and agreed to b...,It is agreed that the Tenant will pay the Base...,en,rental,4.0,7.60,3.60
183,en_rental_038,Should the Tenant remain in possession of the ...,If the Tenant stays in the Premises with the L...,en,rental,4.0,7.34,3.34
280,hi_tos_076,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाण...,PhonePe आपसे विशिष्ट जानकारी का अनुरोध कर सकता...,hi,tos_privacy,6.0,9.13,3.13
265,hi_tos_061,हम व्यक्तिगत जानकारी को भारत के भीतर संग्रहीत ...,हम भारत में व्यक्तिगत जानकारी को संग्रहीत करते...,hi,tos_privacy,6.0,9.12,3.12
116,en_tos_117,To use less information that's connected to in...,"To use less personal information, we sometimes...",en,tos_privacy,5.0,8.11,3.11
332,hi_rental_047,अचल संपति को किराये पर देने की बाबत किसी प्रका...,अचल संपत्ति को किराये पर देने की कोई रूकावट नह...,hi,rental,6.0,9.09,3.09


In [ ]:
results_df["Prediction_Error"] = (
    results_df["Predicted_LMP"] - results_df["LMP"]
)

results_df["Absolute_Error"] = results_df["Prediction_Error"].abs()

print("Mean absolute error:", round(results_df["Absolute_Error"].mean(), 4))

print("\nLargest overestimations:")
display(
    results_df.sort_values("Prediction_Error", ascending=False)[
        ["ID", "Language", "Domain", "LMP",
         "Predicted_LMP", "Prediction_Error",
         "Original", "Simplified"]
    ].head(10)
)

print("\nLargest underestimations:")
display(
    results_df.sort_values("Prediction_Error")[
        ["ID", "Language", "Domain", "LMP",
         "Predicted_LMP", "Prediction_Error",
         "Original", "Simplified"]
    ].head(10)
)

Mean absolute error: 0.9084

Largest overestimations:


,ID,Language,Domain,LMP,Predicted_LMP,Prediction_Error,Original,Simplified
205,hi_tos_001,hi,tos_privacy,5.0,9.10,4.10,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेव...",उपयोग करते समय सहमत होने वाले नियमों का वर्णन ...
266,hi_tos_062,hi,tos_privacy,5.0,8.99,3.99,हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव क...,"हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धो..."
191,en_rental_046,en,rental,4.0,7.93,3.93,The Landlord may rel et the Premises or any pa...,The Landlord may sublet the Premises or part o...
202,en_rental_057,en,rental,4.0,7.61,3.61,Within 30 days of the termination of this leas...,Within 30 days after the lease ends and Tenant...
180,en_rental_035,en,rental,4.0,7.60,3.60,It is the intent of this Lease and agreed to b...,It is agreed that the Tenant will pay the Base...
183,en_rental_038,en,rental,4.0,7.34,3.34,Should the Tenant remain in possession of the ...,If the Tenant stays in the Premises with the L...
280,hi_tos_076,hi,tos_privacy,6.0,9.13,3.13,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाण...,PhonePe आपसे विशिष्ट जानकारी का अनुरोध कर सकता...
265,hi_tos_061,hi,tos_privacy,6.0,9.12,3.12,हम व्यक्तिगत जानकारी को भारत के भीतर संग्रहीत ...,हम भारत में व्यक्तिगत जानकारी को संग्रहीत करते...
116,en_tos_117,en,tos_privacy,5.0,8.11,3.11,To use less information that's connected to in...,"To use less personal information, we sometimes..."
222,hi_tos_018,hi,tos_privacy,6.0,9.09,3.09,दूसरों को या खुद को नुकसान न पहुंचाना या दूसरो...,दूसरों को या खुद को नुकसान न पहुंचाना और बुरा ...



Largest underestimations:


,ID,Language,Domain,LMP,Predicted_LMP,Prediction_Error,Original,Simplified
166,en_rental_021,en,rental,10.0,8.35,-1.65,Major maintenance and repair of the Property n...,"Major maintenance and repair of the Property, ..."
144,en_tos_147,en,tos_privacy,10.0,8.75,-1.25,In response to legal requests from third parti...,In response to legal requests from third parti...
102,en_tos_103,en,tos_privacy,10.0,8.76,-1.24,Not all content is available in spatial audio ...,Not all content is available in spatial audio ...
198,en_rental_053,en,rental,10.0,8.77,-1.23,If the Start Date is on a day other than the f...,If the start date is not on the first of the m...
168,en_rental_023,en,rental,10.0,8.80,-1.20,The Tenant will not engage in any illegal trad...,The Tenant will not engage in any illegal acti...
163,en_rental_018,en,rental,10.0,8.81,-1.19,"A consent by the Landlord to one assignment, s...","A consent by the Landlord to one assignment, s..."
94,en_tos_095,en,tos_privacy,10.0,8.83,-1.17,While we strive to provide a consistent and hi...,While we aim to provide a consistent and high-...
106,en_tos_107,en,tos_privacy,10.0,8.84,-1.16,"The Netflix software is developed by, or for, ...",The Netflix software is developed by Netflix a...
86,en_tos_087,en,tos_privacy,10.0,8.85,-1.15,Members of households with an existing or rece...,Members with an existing or recent Netflix acc...
97,en_tos_098,en,tos_privacy,10.0,8.85,-1.15,"Depending on the subscription plan, some Netfl...","Depending on your subscription plan, some Netf..."


In [ ]:
import pandas as pd

# Select 15 largest disagreements for manual legal review
case_review = (
    results_df.assign(
        Absolute_Error=lambda x: (x["Predicted_LMP"] - x["LMP"]).abs()
    )
    .sort_values("Absolute_Error", ascending=False)
    .head(15)
    .copy()
)

case_review["Legal_Error_Type"] = "Needs manual review"
case_review["Legal_Impact"] = ""
case_review["Reviewer_Explanation"] = ""

case_review = case_review[
    [
        "ID",
        "Language",
        "Domain",
        "Original",
        "Simplified",
        "LMP",
        "Predicted_LMP",
        "Absolute_Error",
        "Legal_Error_Type",
        "Legal_Impact",
        "Reviewer_Explanation",
    ]
]

display(case_review)

,ID,Language,Domain,Original,Simplified,LMP,Predicted_LMP,Absolute_Error,Legal_Error_Type,Legal_Impact,Reviewer_Explanation
205,hi_tos_001,hi,tos_privacy,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेव...",उपयोग करते समय सहमत होने वाले नियमों का वर्णन ...,5.0,9.10,4.10,Needs manual review,,
266,hi_tos_062,hi,tos_privacy,हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव क...,"हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धो...",5.0,8.99,3.99,Needs manual review,,
191,en_rental_046,en,rental,The Landlord may rel et the Premises or any pa...,The Landlord may sublet the Premises or part o...,4.0,7.93,3.93,Needs manual review,,
202,en_rental_057,en,rental,Within 30 days of the termination of this leas...,Within 30 days after the lease ends and Tenant...,4.0,7.61,3.61,Needs manual review,,
180,en_rental_035,en,rental,It is the intent of this Lease and agreed to b...,It is agreed that the Tenant will pay the Base...,4.0,7.60,3.60,Needs manual review,,
183,en_rental_038,en,rental,Should the Tenant remain in possession of the ...,If the Tenant stays in the Premises with the L...,4.0,7.34,3.34,Needs manual review,,
280,hi_tos_076,hi,tos_privacy,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाण...,PhonePe आपसे विशिष्ट जानकारी का अनुरोध कर सकता...,6.0,9.13,3.13,Needs manual review,,
265,hi_tos_061,hi,tos_privacy,हम व्यक्तिगत जानकारी को भारत के भीतर संग्रहीत ...,हम भारत में व्यक्तिगत जानकारी को संग्रहीत करते...,6.0,9.12,3.12,Needs manual review,,
116,en_tos_117,en,tos_privacy,To use less information that's connected to in...,"To use less personal information, we sometimes...",5.0,8.11,3.11,Needs manual review,,
332,hi_rental_047,hi,rental,अचल संपति को किराये पर देने की बाबत किसी प्रका...,अचल संपत्ति को किराये पर देने की कोई रूकावट नह...,6.0,9.09,3.09,Needs manual review,,


In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

for _, row in case_review.iterrows():
    print("=" * 100)
    print("ID:", row["ID"])
    print("Language:", row["Language"], "| Domain:", row["Domain"])
    print("Human LMP:", row["LMP"],
          "| Predicted LMP:", row["Predicted_LMP"])
    print("\nORIGINAL:\n", row["Original"])
    print("\nSIMPLIFIED:\n", row["Simplified"])
    print()

ID: hi_tos_001
Language: hi | Domain: tos_privacy
Human LMP: 5.0 | Predicted LMP: 9.100000381469727

ORIGINAL:
 उन नियमों का वर्णन करता है, जिनसे आप हमारी सेवाओं का उपयोग करते समय सहमत होते हैं.

SIMPLIFIED:
 उपयोग करते समय सहमत होने वाले नियमों का वर्णन करता है.

ID: hi_tos_062
Language: hi | Domain: tos_privacy
Human LMP: 5.0 | Predicted LMP: 8.989999771118164

ORIGINAL:
 हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव करके रख सकते हैं, अगर हमें लगता है कि यह, धोखाधड़ी या भविष्य में गलत इस्तेमाल को रोकने के लिए ज़रूरी है या किसी कानूनी/नियामक कार्यवाही को अधूरी रहने या उस बारे में कोई कानूनी और/या नियामक निर्देश मिलने की स्थिति में कानून के हिसाब से ज़रूरी हो।

SIMPLIFIED:
 हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धोखाधड़ी रोकने या कानूनी कार्यवाही को अधूरा नहीं रहने के लिए ज़रूरी हो।

ID: en_rental_046
Language: en | Domain: rental
Human LMP: 4.0 | Predicted LMP: 7.929999828338623

ORIGINAL:
 The Landlord may rel et the Premises or any part of the Premises for a term or terms which 

In [ ]:
review_notes = {
    "hi_tos_001": (
        "Scope/detail omission",
        "Low to moderate",
        "The explicit reference to using the services is omitted."
    ),
    "hi_tos_062": (
        "Condition/purpose omission",
        "High",
        "Future misuse prevention and some legal or regulatory circumstances are omitted."
    ),
    "en_rental_046": (
        "Potential terminology change",
        "Needs legal review",
        "Relet and sublet may not be legally equivalent in this lease context."
    ),
    "en_rental_057": (
        "Condition/detail omission",
        "Moderate",
        "The explicit tenant-vacating condition and some itemization detail are omitted."
    ),
    "en_rental_035": (
        "No material change apparent",
        "Low",
        "The gross-rent arrangement and exception are substantially retained."
    ),
    "en_rental_038": (
        "No material change apparent",
        "Low",
        "The tenancy type, original terms, and one-month notice are retained."
    ),
    "hi_tos_076": (
        "No material change apparent",
        "Low",
        "Identity verification and authentication purposes are retained."
    ),
    "hi_tos_061": (
        "Retention-period omission",
        "High",
        "The purpose-based limit on the retention period is omitted."
    ),
    "en_tos_117": (
        "Technical-detail omission",
        "Moderate",
        "Anonymisation and the explicit non-identifiability outcome are omitted."
    ),
    "hi_rental_047": (
        "Authority/scope omission",
        "Moderate",
        "The explicit reference to departments and courts is omitted."
    ),
    "hi_tos_018": (
        "Threat/encouragement omission",
        "Moderate",
        "Threatening or encouraging harmful conduct is omitted."
    ),
    "hi_tos_029": (
        "Example/detail omission",
        "Low",
        "The example of changing content format is omitted."
    ),
    "hi_tos_034": (
        "Response-condition omission",
        "Moderate",
        "The ability to act without replying to the suggestion is omitted."
    ),
    "en_tos_144": (
        "Mostly preserved",
        "Low",
        "The legal-obligation limitation is retained, although relevant is omitted."
    ),
    "en_tos_126": (
        "No material change apparent",
        "Low",
        "The core meaning is retained."
    ),
}

for idx, row in case_review.iterrows():
    case_id = row["ID"]
    if case_id in review_notes:
        (
            case_review.at[idx, "Legal_Error_Type"],
            case_review.at[idx, "Legal_Impact"],
            case_review.at[idx, "Reviewer_Explanation"],
        ) = review_notes[case_id]

display(
    case_review[
        [
            "ID",
            "Legal_Error_Type",
            "Legal_Impact",
            "Reviewer_Explanation",
        ]
    ]
)

,ID,Legal_Error_Type,Legal_Impact,Reviewer_Explanation
205,hi_tos_001,Scope/detail omission,Low to moderate,The explicit reference to using the services is omitted.
266,hi_tos_062,Condition/purpose omission,High,Future misuse prevention and some legal or regulatory circumstances are omitted.
191,en_rental_046,Potential terminology change,Needs legal review,Relet and sublet may not be legally equivalent in this lease context.
202,en_rental_057,Condition/detail omission,Moderate,The explicit tenant-vacating condition and some itemization detail are omitted.
180,en_rental_035,No material change apparent,Low,The gross-rent arrangement and exception are substantially retained.
183,en_rental_038,No material change apparent,Low,"The tenancy type, original terms, and one-month notice are retained."
280,hi_tos_076,No material change apparent,Low,Identity verification and authentication purposes are retained.
265,hi_tos_061,Retention-period omission,High,The purpose-based limit on the retention period is omitted.
116,en_tos_117,Technical-detail omission,Moderate,Anonymisation and the explicit non-identifiability outcome are omitted.
332,hi_rental_047,Authority/scope omission,Moderate,The explicit reference to departments and courts is omitted.


In [ ]:
case_review.to_csv(
    "/content/lmp_qualitative_case_analysis.csv",
    index=False
)

print("Saved: /content/lmp_qualitative_case_analysis.csv")

Saved: /content/lmp_qualitative_case_analysis.csv


In [ ]:

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "/content/judgebert_lmp_final"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

sanity_pairs = [
    {
        "Test": "Identical",
        "Original": "The tenant must pay rent on time.",
        "Simplified": "The tenant must pay rent on time."
    },
    {
        "Test": "Identical",
        "Original": "We store personal information in India.",
        "Simplified": "We store personal information in India."
    },
    {
        "Test": "Unrelated",
        "Original": "The tenant must pay rent on time.",
        "Simplified": "The Earth revolves around the Sun."
    },
    {
        "Test": "Unrelated",
        "Original": "We store personal information in India.",
        "Simplified": "The recipe requires two cups of flour."
    }
]

def predict_lmp(original, simplified):
    inputs = tokenizer(
        original,
        simplified,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        prediction = model(**inputs).logits.squeeze().item()

    return round(max(1.0, min(10.0, prediction)), 2)

for pair in sanity_pairs:
    pair["Predicted_LMP"] = predict_lmp(
        pair["Original"], pair["Simplified"]
    )

sanity_df = pd.DataFrame(sanity_pairs)
display(sanity_df)

sanity_df.to_csv("/content/lmp_sanity_checks.csv", index=False)
print("Saved: /content/lmp_sanity_checks.csv")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,Test,Original,Simplified,Predicted_LMP
0,Identical,The tenant must pay rent on time.,The tenant must pay rent on time.,8.95
1,Identical,We store personal information in India.,We store personal information in India.,8.91
2,Unrelated,The tenant must pay rent on time.,The Earth revolves around the Sun.,8.55
3,Unrelated,We store personal information in India.,The recipe requires two cups of flour.,8.72


Saved: /content/lmp_sanity_checks.csv


In [ ]:

# Generalization analysis using the predictions already generated
import pandas as pd

results_df["Absolute_Error"] = (
    results_df["Predicted_LMP"] - results_df["LMP"]
).abs()

print("Performance by language")
language_results = results_df.groupby("Language").apply(
    lambda g: pd.Series({
        "N": len(g),
        "MAE": g["Absolute_Error"].mean(),
        "Mean_Human_LMP": g["LMP"].mean(),
        "Mean_Predicted_LMP": g["Predicted_LMP"].mean()
    }),
    include_groups=False
)
display(language_results.round(3))

print("\nPerformance by legal domain")
domain_results = results_df.groupby("Domain").apply(
    lambda g: pd.Series({
        "N": len(g),
        "MAE": g["Absolute_Error"].mean(),
        "Mean_Human_LMP": g["LMP"].mean(),
        "Mean_Predicted_LMP": g["Predicted_LMP"].mean()
    }),
    include_groups=False
)
display(domain_results.round(3))

language_results.to_csv("/content/generalization_by_language.csv")
domain_results.to_csv("/content/generalization_by_domain.csv")

print("\nSaved both generalization analysis files.")

Performance by language


,N,MAE,Mean_Human_LMP,Mean_Predicted_LMP
Language,,,,
en,205.0,0.862,8.478,8.695
hi,152.0,0.971,9.007,9.090



Performance by legal domain


,N,MAE,Mean_Human_LMP,Mean_Predicted_LMP
Domain,,,,
rental,130.0,0.913,8.631,8.770
tos_privacy,227.0,0.906,8.744,8.917



Saved both generalization analysis files.


In [ ]:

from sklearn.model_selection import train_test_split

# Use the existing model_df containing the 357 labeled examples.
# Keep the same XLM-R checkpoint and regression objective.
domain_col = "Domain"
label_col = "LMP"

print("Domain counts:")
print(model_df[domain_col].value_counts())

print("\nAvailable columns:")
print(model_df.columns.tolist())

assert model_df[domain_col].nunique() == 2
assert model_df[label_col].notna().all()

print("\nSetup verified. Ready for held-out-domain training.")

Domain counts:
Domain
tos_privacy    227
rental         130
Name: count, dtype: int64

Available columns:
['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP']

Setup verified. Ready for held-out-domain training.


In [ ]:

import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

CHECKPOINT = "xlm-roberta-base"
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 5
SEED = 42

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def metric_fn(eval_pred):
    predictions, labels = eval_pred
    predictions = np.asarray(predictions).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    predictions = np.clip(predictions, 1, 10)

    return {
        "rmse": float(np.sqrt(mean_squared_error(labels, predictions))),
        "mae": float(mean_absolute_error(labels, predictions)),
        "pearson": float(pearsonr(labels, predictions).statistic)
            if np.std(predictions) > 0 and np.std(labels) > 0 else 0.0,
        "spearman": float(spearmanr(labels, predictions).statistic)
            if np.std(predictions) > 0 and np.std(labels) > 0 else 0.0,
    }


def run_domain_experiment(train_domain, test_domain):
    print(f"\n{'=' * 65}")
    print(f"Train domain: {train_domain} | Test domain: {test_domain}")
    print("=" * 65)

    train_df = model_df[model_df["Domain"] == train_domain].copy()
    test_df = model_df[model_df["Domain"] == test_domain].copy()

    # Validation is drawn only from the training domain.
    train_part, val_part = train_test_split(
        train_df,
        test_size=0.15,
        random_state=SEED,
        stratify=train_df["Language"]
    )

    def prepare_dataset(frame):
        ds = Dataset.from_pandas(
            frame[["Original", "Simplified", "LMP"]].reset_index(drop=True)
        )

        def tokenize_batch(batch):
            return tokenizer(
                batch["Original"],
                batch["Simplified"],
                truncation=True,
                max_length=MAX_LENGTH
            )

        ds = ds.map(tokenize_batch, batched=True)
        ds = ds.rename_column("LMP", "labels")
        ds.set_format(
            type="torch",
            columns=["input_ids", "attention_mask", "labels"]
        )
        return ds

    train_data = prepare_dataset(train_part)
    val_data = prepare_dataset(val_part)
    test_data = prepare_dataset(test_df)

    # Dynamically pad each batch to the longest sequence in that batch.
    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT,
        num_labels=1
    )

    output_dir = f"/content/domain_exp_{train_domain}_to_{test_domain}"

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_rmse",
        greater_is_better=False,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        data_seed=SEED,
        report_to="none",
        save_total_limit=1
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=val_data,
        data_collator=data_collator,
        compute_metrics=metric_fn
    )

    trainer.train()

    # Evaluate only after training, on the entirely held-out domain.
    prediction_output = trainer.predict(test_data)
    predictions = np.asarray(
        prediction_output.predictions
    ).reshape(-1)
    predictions = np.clip(predictions, 1, 10)

    labels = test_df["LMP"].to_numpy(dtype=float)

    result = {
        "Train_Domain": train_domain,
        "Test_Domain": test_domain,
        "Train_N": len(train_part),
        "Validation_N": len(val_part),
        "Test_N": len(test_df),
        "RMSE": np.sqrt(mean_squared_error(labels, predictions)),
        "MAE": mean_absolute_error(labels, predictions),
        "Pearson": pearsonr(labels, predictions).statistic
            if np.std(predictions) > 0 and np.std(labels) > 0 else np.nan,
        "Spearman": spearmanr(labels, predictions).statistic
            if np.std(predictions) > 0 and np.std(labels) > 0 else np.nan,
    }

    print(pd.Series(result).to_string())

    return result


# Run both directions.
domain_results = []

domain_results.append(
    run_domain_experiment("rental", "tos_privacy")
)

domain_results.append(
    run_domain_experiment("tos_privacy", "rental")
)

domain_results_df = pd.DataFrame(domain_results)

print("\nFINAL HELD-OUT-DOMAIN RESULTS")
display(domain_results_df.round(4))

domain_results_df.to_csv(
    "/content/held_out_domain_results.csv",
    index=False
)

print("Saved: /content/held_out_domain_results.csv")


Train domain: rental | Test domain: tos_privacy


Map:   0%|          | 0/110 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (17 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Train_Domain    tos_privacy
Test_Domain          rental
Train_N                 192
Validation_N             35
Test_N                  130
RMSE               1.761515
MAE                1.539754
Pearson            0.309955
Spearman           0.301214

FINAL HELD-OUT-DOMAIN RESULTS


,Train_Domain,Test_Domain,Train_N,Validation_N,Test_N,RMSE,MAE,Pearson,Spearman
0,rental,tos_privacy,110,20,227,3.6670,3.4596,0.0281,-0.0017
1,tos_privacy,rental,192,35,130,1.7615,1.5398,0.3100,0.3012


Saved: /content/held_out_domain_results.csv


In [ ]:

import pandas as pd
import numpy as np

df_audit = model_df.copy()

# 1. Dataset size and missing values
print("=== DATASET OVERVIEW ===")
print("Total pairs:", len(df_audit))
print("\nMissing values:")
print(df_audit[["Original", "Simplified", "Language", "Domain", "LMP"]]
      .isna().sum())

# 2. Duplicate pair check (case- and whitespace-insensitive)
df_audit["pair_key"] = (
    df_audit["Original"].fillna("").str.strip().str.lower()
    + " || "
    + df_audit["Simplified"].fillna("").str.strip().str.lower()
)

print("\n=== DUPLICATE CHECK ===")
print("Rows belonging to repeated sentence pairs:",
      int(df_audit["pair_key"].duplicated(keep=False).sum()))
print("Repeated pair occurrences beyond the first:",
      int(df_audit["pair_key"].duplicated().sum()))

# Check whether exact pairs have conflicting LMP labels
pair_label_counts = df_audit.groupby("pair_key")["LMP"].nunique()
print("Repeated pairs with conflicting LMP labels:",
      int((pair_label_counts[pair_label_counts > 1]).shape[0]))

# 3. Human LMP score distribution
print("\n=== LMP SCORE DISTRIBUTION ===")
display(
    df_audit["LMP"].value_counts().sort_index()
    .rename_axis("LMP")
    .to_frame("Count")
)

# 4. Score distribution by language
print("\n=== SCORES BY LANGUAGE ===")
display(
    df_audit.groupby("Language")["LMP"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(3)
)

# 5. Score distribution by legal domain
print("\n=== SCORES BY DOMAIN ===")
display(
    df_audit.groupby("Domain")["LMP"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(3)
)

# 6. Language x domain counts
print("\n=== LANGUAGE / DOMAIN COUNTS ===")
display(pd.crosstab(df_audit["Language"], df_audit["Domain"]))

# 7. Identical original/simplified pairs in the dataset
identical_mask = (
    df_audit["Original"].fillna("").str.strip().str.lower()
    == df_audit["Simplified"].fillna("").str.strip().str.lower()
)
print("\nIdentical original/simplified pairs:", int(identical_mask.sum()))

print("\nAudit complete. No dataset values were modified.")

=== DATASET OVERVIEW ===
Total pairs: 357

Missing values:
Original      0
Simplified    0
Language      0
Domain        0
LMP           0
dtype: int64

=== DUPLICATE CHECK ===
Rows belonging to repeated sentence pairs: 0
Repeated pair occurrences beyond the first: 0
Repeated pairs with conflicting LMP labels: 0

=== LMP SCORE DISTRIBUTION ===


,Count
LMP,
4.0,4
5.0,5
6.0,17
7.0,34
8.0,71
9.0,102
10.0,124



=== SCORES BY LANGUAGE ===


,count,mean,std,min,max
Language,,,,,
en,205,8.478,1.352,4.0,10.0
hi,152,9.007,1.247,5.0,10.0



=== SCORES BY DOMAIN ===


,count,mean,std,min,max
Domain,,,,,
rental,130,8.631,1.490,4.0,10.0
tos_privacy,227,8.744,1.236,5.0,10.0



=== LANGUAGE / DOMAIN COUNTS ===


Domain,rental,tos_privacy
Language,,
en,59,146
hi,71,81



Identical original/simplified pairs: 14

Audit complete. No dataset values were modified.


In [ ]:

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Reconstruct the original 80/20 split.
all_idx = np.arange(len(model_df))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42
)

# Reconstruct the training/validation split.
train_part_idx, val_idx = train_test_split(
    train_idx, test_size=0.15, random_state=42
)

train_df = model_df.iloc[train_part_idx].copy()
val_df = model_df.iloc[val_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("=== SPLIT SIZES ===")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\n=== LMP DISTRIBUTION BY SPLIT ===")
for split_name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"\n{split_name}:")
    print("Mean LMP:", round(split_df["LMP"].mean(), 3))
    print("Std LMP:", round(split_df["LMP"].std(), 3))
    print("Score counts:")
    print(split_df["LMP"].value_counts().sort_index().to_dict())

print("\n=== LANGUAGE / DOMAIN DISTRIBUTION ===")
for split_name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"\n{split_name}:")
    display(pd.crosstab(split_df["Language"], split_df["Domain"]))

# Check for identical normalized pairs crossing split boundaries.
def normalize_pair(frame):
    return (
        frame["Original"].fillna("").str.strip().str.lower()
        + " || "
        + frame["Simplified"].fillna("").str.strip().str.lower()
    )

train_pairs = set(normalize_pair(train_df))
val_pairs = set(normalize_pair(val_df))
test_pairs = set(normalize_pair(test_df))

print("\n=== CROSS-SPLIT IDENTICAL PAIRS ===")
print("Train ∩ Validation:", len(train_pairs & val_pairs))
print("Train ∩ Test:", len(train_pairs & test_pairs))
print("Validation ∩ Test:", len(val_pairs & test_pairs))

# Save the split IDs for reproducibility.
split_ids = pd.concat([
    pd.DataFrame({"row_index": train_part_idx, "split": "train"}),
    pd.DataFrame({"row_index": val_idx, "split": "validation"}),
    pd.DataFrame({"row_index": test_idx, "split": "test"}),
], ignore_index=True)

split_ids.to_csv("/content/lmp_split_audit.csv", index=False)
print("\nSaved /content/lmp_split_audit.csv")

=== SPLIT SIZES ===
Training: 242
Validation: 43
Test: 72

=== LMP DISTRIBUTION BY SPLIT ===

Train:
Mean LMP: 8.723
Std LMP: 1.321
Score counts:
{4.0: 3, 5.0: 4, 6.0: 8, 7.0: 22, 8.0: 54, 9.0: 65, 10.0: 86}

Validation:
Mean LMP: 8.744
Std LMP: 1.382
Score counts:
{4.0: 1, 6.0: 2, 7.0: 5, 8.0: 5, 9.0: 15, 10.0: 15}

Test:
Mean LMP: 8.611
Std LMP: 1.359
Score counts:
{5.0: 1, 6.0: 7, 7.0: 7, 8.0: 12, 9.0: 22, 10.0: 23}

=== LANGUAGE / DOMAIN DISTRIBUTION ===

Train:


Domain,rental,tos_privacy
Language,,
en,47,94
hi,48,53



Validation:


Domain,rental,tos_privacy
Language,,
en,5,18
hi,8,12



Test:


Domain,rental,tos_privacy
Language,,
en,7,34
hi,15,16



=== CROSS-SPLIT IDENTICAL PAIRS ===
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

Saved /content/lmp_split_audit.csv


In [ ]:

# Inspect whether TF-IDF was fitted before or after splitting.
# Run this in the same notebook where you trained the baselines.

print("Relevant variables currently in memory:")
for name in [
    "tfidf", "tfidf_vectorizer", "vectorizer",
    "X_train_tfidf", "X_test_tfidf",
    "tfidf_model", "lr_tfidf", "tfidf_regressor"
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")

print("\nSearch your notebook cells for these patterns:")
print("- TfidfVectorizer(")
print("- fit_transform(")
print("- train_test_split(")
print("- LinearRegression(")

Relevant variables currently in memory:

Search your notebook cells for these patterns:
- TfidfVectorizer(
- fit_transform(
- train_test_split(
- LinearRegression(


In [ ]:

# Check saved prediction files for IDs and compare their test rows.
import os
import pandas as pd

for path in [
    "/content/tfidf_clean_test_predictions.csv",
    "/content/lmp_prediction_results.csv",
    "/content/held_out_domain_results.csv"
]:
    print(f"\n{path}:")
    if os.path.exists(path):
        temp = pd.read_csv(path)
        print("Shape:", temp.shape)
        print("Columns:", temp.columns.tolist())
        if "ID" in temp.columns:
            print("First IDs:", temp["ID"].head(10).tolist())
    else:
        print("Not found")


/content/tfidf_clean_test_predictions.csv:
Shape: (72, 8)
Columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP', 'TFIDF_Cosine', 'Predicted_LMP']
First IDs: ['hi_tos_018', 'en_tos_043', 'hi_rental_006', 'hi_rental_015', 'en_tos_057', 'hi_rental_050', 'en_tos_016', 'en_tos_058', 'hi_tos_041', 'en_tos_125']

/content/lmp_prediction_results.csv:
Shape: (357, 8)
Columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP', 'Predicted_LMP', 'Preservation_Decision']
First IDs: ['en_tos_001', 'en_tos_002', 'en_tos_003', 'en_tos_004', 'en_tos_005', 'en_tos_006', 'en_tos_007', 'en_tos_008', 'en_tos_009', 'en_tos_010']

/content/held_out_domain_results.csv:
Shape: (2, 9)
Columns: ['Train_Domain', 'Test_Domain', 'Train_N', 'Validation_N', 'Test_N', 'RMSE', 'MAE', 'Pearson', 'Spearman']


In [ ]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Reconstruct the split from the original dataset
audit_df = model_df.reset_index(drop=True).copy()

all_idx = np.arange(len(audit_df))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42
)

expected_test = audit_df.iloc[test_idx].copy()

tfidf_test = pd.read_csv("/content/tfidf_clean_test_predictions.csv")
xlmr_all = pd.read_csv("/content/lmp_prediction_results.csv")

print("Expected test rows:", len(expected_test))
print("TF-IDF test rows:", len(tfidf_test))

# Check whether the TF-IDF IDs match the reconstructed test split
expected_ids = set(expected_test["ID"].astype(str))
tfidf_ids = set(tfidf_test["ID"].astype(str))

print("TF-IDF IDs match expected test IDs:", tfidf_ids == expected_ids)
print("Missing TF-IDF IDs:", sorted(expected_ids - tfidf_ids)[:10])
print("Unexpected TF-IDF IDs:", sorted(tfidf_ids - expected_ids)[:10])

# Extract the XLM-R predictions for the same test IDs
xlmr_test = xlmr_all[xlmr_all["ID"].astype(str).isin(expected_ids)].copy()

print("\nXLM-R rows matching test IDs:", len(xlmr_test))
print("XLM-R IDs match expected test IDs:",
      set(xlmr_test["ID"].astype(str)) == expected_ids)

# Save a test-only XLM-R file for subsequent comparisons
xlmr_test.to_csv("/content/xlmr_test_predictions_audit.csv", index=False)
print("Saved /content/xlmr_test_predictions_audit.csv")

Expected test rows: 72
TF-IDF test rows: 72
TF-IDF IDs match expected test IDs: True
Missing TF-IDF IDs: []
Unexpected TF-IDF IDs: []

XLM-R rows matching test IDs: 72
XLM-R IDs match expected test IDs: True
Saved /content/xlmr_test_predictions_audit.csv


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

test_ids = set(
    pd.read_csv("/content/tfidf_clean_test_predictions.csv")["ID"].astype(str)
)

xlmr_all = pd.read_csv("/content/lmp_prediction_results.csv")
xlmr_test = xlmr_all[xlmr_all["ID"].astype(str).isin(test_ids)].copy()

assert len(xlmr_test) == 72
assert xlmr_test["ID"].astype(str).nunique() == 72

y_true = pd.to_numeric(xlmr_test["LMP"]).to_numpy()
y_pred = pd.to_numeric(xlmr_test["Predicted_LMP"]).clip(1, 10).to_numpy()

print("XLM-R — SAME 72 TEST IDs")
print(f"Pearson: {pearsonr(y_pred, y_true).statistic:.4f}")
print(f"Spearman: {spearmanr(y_pred, y_true).statistic:.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_true, y_pred):.4f}")

XLM-R — SAME 72 TEST IDs
Pearson: 0.5784
Spearman: 0.4195
RMSE: 1.2325
MAE: 0.9754


In [ ]:

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr, spearmanr
import numpy as np
import pandas as pd

# Use the same dataset and row order as the TF-IDF baseline
baseline_df = model_df.reset_index(drop=True).copy()

all_idx = np.arange(len(baseline_df))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42
)
train_part_idx, val_idx = train_test_split(
    train_idx, test_size=0.15, random_state=42
)

train_df = baseline_df.iloc[train_part_idx].copy()
test_df = baseline_df.iloc[test_idx].copy()

# Pretrained multilingual sentence-embedding model
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def get_cosines(frame):
    original = model.encode(
        frame["Original"].astype(str).tolist(),
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    simplified = model.encode(
        frame["Simplified"].astype(str).tolist(),
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    return np.sum(original * simplified, axis=1)

train_cosine = get_cosines(train_df)
test_cosine = get_cosines(test_df)

y_train = train_df["LMP"].astype(float).to_numpy()
y_test = test_df["LMP"].astype(float).to_numpy()

# Fit calibration only on training data
calibrator = LinearRegression()
calibrator.fit(train_cosine.reshape(-1, 1), y_train)

predictions = np.clip(
    calibrator.predict(test_cosine.reshape(-1, 1)), 1, 10
)

print("CLEAN MINILM BASELINE")
print("-" * 40)
print("Training:", len(train_df))
print("Validation (unused):", len(val_idx))
print("Test:", len(test_df))
print(f"Pearson: {pearsonr(test_cosine, y_test).statistic:.4f}")
print(f"Spearman: {spearmanr(test_cosine, y_test).statistic:.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, predictions):.4f}")

minilm_results = test_df[
    ["ID", "Original", "Simplified", "Language", "Domain", "LMP"]
].copy()
minilm_results["Semantic_Cosine"] = test_cosine
minilm_results["Predicted_LMP"] = predictions

minilm_results.to_csv(
    "/content/minilm_clean_test_predictions.csv", index=False
)
print("Saved /content/minilm_clean_test_predictions.csv")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (8 outputs left out)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

CLEAN MINILM BASELINE
----------------------------------------
Training: 242
Validation (unused): 43
Test: 72
Pearson: 0.5875
Spearman: 0.6574
RMSE: 1.1058
MAE: 0.8477
Saved /content/minilm_clean_test_predictions.csv


In [ ]:

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

tfidf = pd.read_csv("/content/tfidf_clean_test_predictions.csv")
minilm = pd.read_csv("/content/minilm_clean_test_predictions.csv")
xlmr = pd.read_csv("/content/xlmr_test_predictions_audit.csv")

# Verify identical test examples
id_sets = [
    set(df_["ID"].astype(str)) for df_ in [tfidf, minilm, xlmr]
]
assert id_sets[0] == id_sets[1] == id_sets[2], \
    "Test IDs differ; investigate before comparing models."

# Align rows by ID and check labels agree
tfidf = tfidf.sort_values("ID").reset_index(drop=True)
minilm = minilm.sort_values("ID").reset_index(drop=True)
xlmr = xlmr.sort_values("ID").reset_index(drop=True)

assert tfidf["ID"].equals(minilm["ID"])
assert tfidf["ID"].equals(xlmr["ID"])
assert np.allclose(tfidf["LMP"], minilm["LMP"])
assert np.allclose(tfidf["LMP"], xlmr["LMP"])

y = tfidf["LMP"].astype(float).to_numpy()

predictions = {
    "TF-IDF": tfidf["Predicted_LMP"].astype(float).to_numpy(),
    "MiniLM": minilm["Predicted_LMP"].astype(float).to_numpy(),
    "XLM-R": xlmr["Predicted_LMP"].astype(float).to_numpy()
}

rows = []
for name, pred in predictions.items():
    rows.append({
        "Model": name,
        "Pearson": pearsonr(pred, y).statistic,
        "Spearman": spearmanr(pred, y).statistic,
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "MAE": mean_absolute_error(y, pred)
    })

comparison = pd.DataFrame(rows).round(4)
display(comparison)

comparison.to_csv("/content/lmp_model_comparison_clean.csv", index=False)
print("Saved /content/lmp_model_comparison_clean.csv")
print("All models evaluated on the same 72 test IDs.")

,Model,Pearson,Spearman,RMSE,MAE
0,TF-IDF,0.5976,0.7100,1.1634,0.7924
1,MiniLM,0.5875,0.6574,1.1058,0.8477
2,XLM-R,0.5784,0.4195,1.2325,0.9754


Saved /content/lmp_model_comparison_clean.csv
All models evaluated on the same 72 test IDs.


In [ ]:

# First, inspect whether the TF-IDF objects are still available.
for name in ["vectorizer", "calibrator", "model", "tokenizer", "trainer"]:
    print(f"{name}: {'available' if name in globals() else 'not in memory'}")

vectorizer: available
calibrator: available
model: available
tokenizer: available
trainer: available


In [ ]:

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# MiniLM sentence-embedding model
minilm_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Test cases
sanity_pairs = [
    {
        "Case": "Identical",
        "Original": "The tenant must pay rent on time.",
        "Simplified": "The tenant must pay rent on time."
    },
    {
        "Case": "Identical",
        "Original": "We store personal information in India.",
        "Simplified": "We store personal information in India."
    },
    {
        "Case": "Unrelated",
        "Original": "The tenant must pay rent on time.",
        "Simplified": "The Earth revolves around the Sun."
    },
    {
        "Case": "Unrelated",
        "Original": "We store personal information in India.",
        "Simplified": "The recipe requires two cups of flour."
    }
]

sanity_df = pd.DataFrame(sanity_pairs)

# 1. TF-IDF cosine similarity and calibrated LMP prediction
texts = (
    sanity_df["Original"].astype(str).tolist()
    + sanity_df["Simplified"].astype(str).tolist()
)
vectors = vectorizer.transform(texts)
n = len(sanity_df)

tfidf_cosine = np.asarray(
    vectors[:n].multiply(vectors[n:]).sum(axis=1)
).ravel()

# Existing calibrator was fitted to cosine similarity
tfidf_pred = np.clip(
    calibrator.predict(tfidf_cosine.reshape(-1, 1)), 1, 10
)

# 2. MiniLM cosine similarity and calibrated prediction
orig_emb = minilm_model.encode(
    sanity_df["Original"].tolist(), normalize_embeddings=True
)
simp_emb = minilm_model.encode(
    sanity_df["Simplified"].tolist(), normalize_embeddings=True
)
minilm_cosine = np.sum(orig_emb * simp_emb, axis=1)

# Fit a clean calibrator on the training pairs, not the test pairs
train_df = model_df.iloc[train_part_idx].copy()
orig_train = minilm_model.encode(
    train_df["Original"].astype(str).tolist(),
    normalize_embeddings=True
)
simp_train = minilm_model.encode(
    train_df["Simplified"].astype(str).tolist(),
    normalize_embeddings=True
)
train_cosine = np.sum(orig_train * simp_train, axis=1)

from sklearn.linear_model import LinearRegression
minilm_calibrator = LinearRegression().fit(
    train_cosine.reshape(-1, 1),
    train_df["LMP"].astype(float).to_numpy()
)
minilm_pred = np.clip(
    minilm_calibrator.predict(minilm_cosine.reshape(-1, 1)), 1, 10
)

# 3. Fine-tuned XLM-R regression predictions
# Uses the existing trainer and tokenizer, without retraining.
xlmr_scores = []
for _, row in sanity_df.iterrows():
    inputs = tokenizer(
        row["Original"],
        row["Simplified"],
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    device = next(trainer.model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    trainer.model.eval()
    import torch
    with torch.no_grad():
        score = trainer.model(**inputs).logits.squeeze().item()
    xlmr_scores.append(float(np.clip(score, 1, 10)))

sanity_df["TFIDF_LMP"] = np.round(tfidf_pred, 2)
sanity_df["MiniLM_Cosine"] = np.round(minilm_cosine, 4)
sanity_df["MiniLM_LMP"] = np.round(minilm_pred, 2)
sanity_df["XLMR_LMP"] = np.round(xlmr_scores, 2)

display(sanity_df)
sanity_df.to_csv("/content/lmp_sanity_check_results.csv", index=False)
print("Saved /content/lmp_sanity_check_results.csv")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,Case,Original,Simplified,TFIDF_LMP,MiniLM_Cosine,MiniLM_LMP,XLMR_LMP
0,Identical,The tenant must pay rent on time.,The tenant must pay rent on time.,9.32,1.0000,9.32,8.95
1,Identical,We store personal information in India.,We store personal information in India.,9.32,1.0000,9.32,8.91
2,Unrelated,The tenant must pay rent on time.,The Earth revolves around the Sun.,1.76,0.0338,1.00,8.55
3,Unrelated,We store personal information in India.,The recipe requires two cups of flour.,1.00,-0.1198,1.00,8.72


Saved /content/lmp_sanity_check_results.csv


In [ ]:

import pandas as pd

xlmr_test = pd.read_csv("/content/xlmr_test_predictions_audit.csv")

print("Human LMP distribution:")
print(xlmr_test["LMP"].describe())

print("\nXLM-R predicted LMP distribution:")
print(xlmr_test["Predicted_LMP"].describe())

print("\nPrediction counts rounded to nearest integer:")
print(
    xlmr_test["Predicted_LMP"]
    .round()
    .value_counts()
    .sort_index()
)

print("\nLargest prediction errors:")
xlmr_test["Absolute_Error"] = (
    xlmr_test["LMP"] - xlmr_test["Predicted_LMP"]
).abs()

display(
    xlmr_test.sort_values("Absolute_Error", ascending=False)[
        ["ID", "Language", "Domain", "LMP",
         "Predicted_LMP", "Absolute_Error"]
    ].head(10)
)

Human LMP distribution:
count    72.000000
mean      8.611111
std       1.358910
min       5.000000
25%       8.000000
50%       9.000000
75%      10.000000
max      10.000000
Name: LMP, dtype: float64

XLM-R predicted LMP distribution:
count    72.00000
mean      8.88125
std       0.29847
min       7.87000
25%       8.82500
50%       8.98000
75%       9.11000
max       9.13000
Name: Predicted_LMP, dtype: float64

Prediction counts rounded to nearest integer:
Predicted_LMP
8.0     7
9.0    65
Name: count, dtype: int64

Largest prediction errors:


,ID,Language,Domain,LMP,Predicted_LMP,Absolute_Error
53,hi_tos_061,hi,tos_privacy,6.0,9.12,3.12
28,en_tos_117,en,tos_privacy,5.0,8.11,3.11
45,hi_tos_018,hi,tos_privacy,6.0,9.09,3.09
23,en_tos_091,en,tos_privacy,6.0,8.58,2.58
40,en_rental_052,en,rental,6.0,8.30,2.30
39,en_rental_050,en,rental,6.0,8.13,2.13
67,hi_rental_059,hi,rental,7.0,9.11,2.11
42,hi_tos_005,hi,tos_privacy,7.0,9.03,2.03
36,en_rental_012,en,rental,6.0,7.91,1.91
38,en_rental_040,en,rental,6.0,7.87,1.87


In [ ]:

print("Training arguments:")
print("Epochs:", trainer.args.num_train_epochs)
print("Learning rate:", trainer.args.learning_rate)
print("Weight decay:", trainer.args.weight_decay)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)

print("\nSplit sizes:")
print("Train:", len(train_part_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

print("\nTraining and test index overlap:")
print("Train ∩ Test:", len(set(train_part_idx) & set(test_idx)))
print("Validation ∩ Test:", len(set(val_idx) & set(test_idx)))

print("\nModel output layer:")
print(trainer.model.classifier)

Training arguments:
Epochs: 5
Learning rate: 2e-05
Weight decay: 0.01
Best checkpoint: /content/judgebert_lmp_results/checkpoint-155
Best metric: 1.2270335368148846

Split sizes:
Train: 242
Validation: 43
Test: 72

Training and test index overlap:
Train ∩ Test: 0
Validation ∩ Test: 0

Model output layer:
XLMRobertaClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=1, bias=True)
)


In [ ]:

print("Model problem type:", trainer.model.config.problem_type)
print("Number of labels:", trainer.model.config.num_labels)

print("\nTrainer dataset sizes:")
print("Train:", len(trainer.train_dataset))
print("Validation:", len(trainer.eval_dataset))

print("\nTraining label dtype:")
print(trainer.train_dataset[0]["labels"].dtype)
print("Example label:", trainer.train_dataset[0]["labels"])

Model problem type: regression
Number of labels: 1

Trainer dataset sizes:
Train: 242
Validation: 43

Training label dtype:


AttributeError: 'float' object has no attribute 'dtype'

In [ ]:

import torch

label = trainer.train_dataset[0]["labels"]

print("Label value:", label)
print("Label Python type:", type(label))
print("Label is numeric:", isinstance(label, (int, float)))
print("Model loss function:", trainer.model.config.problem_type)

print("\nTraining label range:")
train_labels = [float(x["labels"]) for x in trainer.train_dataset]
print("Min:", min(train_labels))
print("Max:", max(train_labels))
print("Mean:", sum(train_labels) / len(train_labels))

print("\nValidation label range:")
val_labels = [float(x["labels"]) for x in trainer.eval_dataset]
print("Min:", min(val_labels))
print("Max:", max(val_labels))
print("Mean:", sum(val_labels) / len(val_labels))

Label value: 10.0
Label Python type: <class 'float'>
Label is numeric: True
Model loss function: regression

Training label range:
Min: 4.0
Max: 10.0
Mean: 8.723140495867769

Validation label range:
Min: 4.0
Max: 10.0
Mean: 8.744186046511627


In [ ]:

import torch

checks = [
    (
        "Identical",
        "The tenant must pay rent on time.",
        "The tenant must pay rent on time."
    ),
    (
        "Unrelated",
        "The tenant must pay rent on time.",
        "The Earth revolves around the Sun."
    ),
]

trainer.model.eval()

for name, original, simplified in checks:
    inputs = tokenizer(
        original,
        simplified,
        truncation=True,
        max_length=256,
        padding=True,
        return_tensors="pt"
    ).to(trainer.model.device)

    with torch.no_grad():
        score = trainer.model(**inputs).logits.squeeze().item()

    print(f"{name}: raw XLM-R score = {score:.4f}")

Identical: raw XLM-R score = 8.9531
Unrelated: raw XLM-R score = 8.5469


In [ ]:

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

train_output = trainer.predict(trainer.train_dataset)
train_true = np.asarray(train_output.label_ids).reshape(-1)
train_pred = np.asarray(train_output.predictions).reshape(-1)

val_output = trainer.predict(trainer.eval_dataset)
val_true = np.asarray(val_output.label_ids).reshape(-1)
val_pred = np.asarray(val_output.predictions).reshape(-1)

def report(name, y_true, y_pred):
    print(f"\n{name}")
    print("Prediction mean:", round(float(y_pred.mean()), 4))
    print("Prediction std:", round(float(y_pred.std()), 4))
    print("MAE:", round(mean_absolute_error(y_true, y_pred), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_true, y_pred)), 4))
    print("Pearson:", round(float(pearsonr(y_true, y_pred).statistic), 4))
    print("Spearman:", round(float(spearmanr(y_true, y_pred).statistic), 4))

report("TRAIN", train_true, train_pred)
report("VALIDATION", val_true, val_pred)


TRAIN
Prediction mean: 8.8494
Prediction std: 0.3402
MAE: 0.8909
RMSE: 1.1597
Pearson: 0.5845
Spearman: 0.5116

VALIDATION
Prediction mean: 8.9122
Prediction std: 0.3113
MAE: 0.8947
RMSE: 1.227
Pearson: 0.5696
Spearman: 0.3586


In [ ]:

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

# Use the labels from the existing training and validation datasets.
train_y = np.array([
    float(item["labels"]) for item in trainer.train_dataset
])
val_y = np.array([
    float(item["labels"]) for item in trainer.eval_dataset
])

# Baseline: predict the training-label mean for every validation example.
baseline_pred = np.full(len(val_y), train_y.mean())

print("Training-label mean:", round(float(train_y.mean()), 4))
print("Mean-baseline validation MAE:",
      round(mean_absolute_error(val_y, baseline_pred), 4))
print("Mean-baseline validation RMSE:",
      round(np.sqrt(mean_squared_error(val_y, baseline_pred)), 4))

print("\nXLM-R validation MAE: 0.8947")
print("XLM-R validation RMSE: 1.2270")

Training-label mean: 8.7231
Mean-baseline validation MAE: 1.0629
Mean-baseline validation RMSE: 1.3657

XLM-R validation MAE: 0.8947
XLM-R validation RMSE: 1.2270


In [ ]:

from collections import Counter

train_labels = [
    int(round(float(item["labels"])))
    for item in trainer.train_dataset
]

counts = Counter(train_labels)

print("Training-label counts:")
for score in range(1, 11):
    print(f"Score {score}: {counts.get(score, 0)}")

print("\nPercentage with scores 8–10:")
high_count = sum(counts.get(score, 0) for score in [8, 9, 10])
print(f"{100 * high_count / len(train_labels):.1f}%")

Training-label counts:
Score 1: 0
Score 2: 0
Score 3: 0
Score 4: 3
Score 5: 4
Score 6: 8
Score 7: 22
Score 8: 54
Score 9: 65
Score 10: 86

Percentage with scores 8–10:
84.7%


In [ ]:

import inspect
from transformers import Trainer

print("Trainer class:", type(trainer).__name__)
print("Custom compute_loss:",
      type(trainer).compute_loss is not Trainer.compute_loss)

print("\nModel loss configuration:")
print("Problem type:", trainer.model.config.problem_type)
print("Number of labels:", trainer.model.config.num_labels)

print("\nTraining arguments:")
print("Epochs:", trainer.args.num_train_epochs)
print("Learning rate:", trainer.args.learning_rate)
print("Weight decay:", trainer.args.weight_decay)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

Trainer class: Trainer
Custom compute_loss: False

Model loss configuration:
Problem type: regression
Number of labels: 1

Training arguments:
Epochs: 5
Learning rate: 2e-05
Weight decay: 0.01
Best checkpoint: /content/judgebert_lmp_results/checkpoint-155


In [ ]:

import numpy as np
from collections import Counter

train_labels = np.array([
    float(item["labels"]) for item in trainer.train_dataset
])

counts = Counter(train_labels.astype(int))

# Inverse-frequency weights, tempered with square root and capped.
# Rare scores receive more attention, but not extreme influence.
class_weights = {
    score: min(
        2.0,
        np.sqrt(len(train_labels) / (10 * counts.get(score, 0)))
    )
    if counts.get(score, 0) > 0 else 1.0
    for score in range(1, 11)
}

# Normalize weights to have a training-example-weighted mean of 1.
raw_weights = np.array([
    class_weights[int(score)] for score in train_labels
])
normalizer = raw_weights.mean()
class_weights = {
    score: weight / normalizer
    for score, weight in class_weights.items()
}

print("Proposed training-only weights:")
for score in range(1, 11):
    print(f"Score {score}: {class_weights[score]:.3f}")

Proposed training-only weights:
Score 1: 1.404
Score 2: 1.404
Score 3: 1.404
Score 4: 2.807
Score 5: 2.807
Score 6: 2.441
Score 7: 1.472
Score 8: 0.940
Score 9: 0.856
Score 10: 0.745


In [ ]:

import numpy as np
from collections import Counter

train_labels = np.array([
    float(item["labels"]) for item in trainer.train_dataset
])

counts = Counter(train_labels.astype(int))
n = len(train_labels)

# Conservative inverse-frequency weighting.
raw_weights = {
    score: np.sqrt(n / (10 * counts[score]))
    if counts[score] > 0 else 1.0
    for score in range(1, 11)
}

# Cap extreme weights, then normalize over training examples.
raw_weights = {
    score: np.clip(weight, 0.7, 1.5)
    for score, weight in raw_weights.items()
}

example_weights = np.array([
    raw_weights[int(score)] for score in train_labels
])
normalizer = example_weights.mean()

class_weights = {
    score: float(weight / normalizer)
    for score, weight in raw_weights.items()
}

print("Conservative weights:")
for score in range(1, 11):
    print(f"Score {score}: {class_weights[score]:.3f}")

print("Training-example-weighted average:",
      round(np.mean([class_weights[int(y)] for y in train_labels]), 4))

Conservative weights:
Score 1: 1.280
Score 2: 1.280
Score 3: 1.280
Score 4: 1.920
Score 5: 1.920
Score 6: 1.920
Score 7: 1.342
Score 8: 0.896
Score 9: 0.896
Score 10: 0.896
Training-example-weighted average: 1.0


In [ ]:

import numpy as np

train_labels = np.array([
    float(item["labels"]) for item in trainer.train_dataset
])

raw_weights = {
    score: (
        1.5 if score in [4, 5, 6]
        else 1.0 if score == 7
        else 0.85 if score in [8, 9, 10]
        else 1.0
    )
    for score in range(1, 11)
}

example_weights = np.array([
    raw_weights[int(score)] for score in train_labels
])
normalizer = example_weights.mean()

class_weights = {
    score: raw_weights[score] / normalizer
    for score in range(1, 11)
}

print("Final experimental weights:")
for score in range(1, 11):
    print(f"Score {score}: {class_weights[score]:.3f}")

print("Weighted average across training examples:",
      round(np.mean([class_weights[int(y)] for y in train_labels]), 4))

Final experimental weights:
Score 1: 1.106
Score 2: 1.106
Score 3: 1.106
Score 4: 1.659
Score 5: 1.659
Score 6: 1.659
Score 7: 1.106
Score 8: 0.940
Score 9: 0.940
Score 10: 0.940
Weighted average across training examples: 1.0


In [ ]:

from transformers import DataCollatorWithPadding, AutoTokenizer

# Tokenizer must match xlm-roberta-base.
weighted_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

weighted_collator = DataCollatorWithPadding(
    tokenizer=weighted_tokenizer,
    padding=True,
)

# Rebuild the experimental trainer with dynamic padding.
weighted_trainer = WeightedRegressionTrainer(
    model=weighted_model,
    args=weighted_args,
    train_dataset=trainer.train_dataset,
    eval_dataset=trainer.eval_dataset,
    data_collator=weighted_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

weighted_trainer.train()

Epoch,Training Loss,Validation Loss,Rmse,Mae,Pearson,Spearman
1,No log,24.851049,5.105205,4.913926,0.011366,0.141248
2,No log,8.308526,2.950412,2.711392,-0.380510,-0.330927
3,No log,2.567112,1.530844,1.342024,0.476989,0.242095
4,No log,1.972669,1.268604,1.062137,0.589986,0.283997
5,No log,1.898769,1.231034,0.996639,0.590952,0.319592


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=155, training_loss=17.32769972278226, metrics={'train_runtime': 493.8043, 'train_samples_per_second': 2.45, 'train_steps_per_second': 0.314, 'total_flos': 64891236945816.0, 'train_loss': 17.32769972278226, 'epoch': 5.0})

In [ ]:

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

# Use the same validation dataset for both models.
val_output = weighted_trainer.predict(trainer.eval_dataset)
weighted_true = np.asarray(val_output.label_ids).reshape(-1)
weighted_pred = np.asarray(val_output.predictions).reshape(-1)

print("Weighted validation prediction distribution:")
print("Mean:", round(float(weighted_pred.mean()), 4))
print("Std:", round(float(weighted_pred.std()), 4))
print("Min:", round(float(weighted_pred.min()), 4))
print("Max:", round(float(weighted_pred.max()), 4))

print("\nWeighted predictions rounded to integers:")
print(dict(zip(
    *np.unique(np.rint(weighted_pred).astype(int), return_counts=True)
)))

print("\nLargest weighted-model errors:")
errors = np.abs(weighted_true - weighted_pred)
for idx in np.argsort(errors)[-10:][::-1]:
    print(
        f"True={weighted_true[idx]:.0f}, "
        f"Pred={weighted_pred[idx]:.2f}, "
        f"Error={errors[idx]:.2f}"
    )

Weighted validation prediction distribution:
Mean: 8.6365
Std: 0.2682
Min: 7.3047
Max: 8.8281

Weighted predictions rounded to integers:
{np.int64(7): np.int64(1), np.int64(8): np.int64(4), np.int64(9): np.int64(38)}

Largest weighted-model errors:
True=4, Pred=7.30, Error=3.30
True=6, Pred=8.77, Error=2.77
True=6, Pred=8.69, Error=2.69
True=7, Pred=8.77, Error=1.77
True=7, Pred=8.72, Error=1.72
True=7, Pred=8.58, Error=1.58
True=10, Pred=8.51, Error=1.49
True=10, Pred=8.55, Error=1.45
True=10, Pred=8.59, Error=1.41
True=10, Pred=8.62, Error=1.38


In [ ]:

import os
import pandas as pd

path = "/content/lmp_qualitative_case_analysis.csv"

print("File exists:", os.path.exists(path))

if os.path.exists(path):
    cases = pd.read_csv(path)
    print("Shape:", cases.shape)
    print("Columns:", cases.columns.tolist())
    display(cases.head(3))
else:
    print("The qualitative-analysis CSV has not been saved at this path.")

File exists: True
Shape: (15, 11)
Columns: ['ID', 'Language', 'Domain', 'Original', 'Simplified', 'LMP', 'Predicted_LMP', 'Absolute_Error', 'Legal_Error_Type', 'Legal_Impact', 'Reviewer_Explanation']


,ID,Language,Domain,Original,Simplified,LMP,Predicted_LMP,Absolute_Error,Legal_Error_Type,Legal_Impact,Reviewer_Explanation
0,hi_tos_001,hi,tos_privacy,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेवाओं का उपयोग करते समय सहमत होते हैं.",उपयोग करते समय सहमत होने वाले नियमों का वर्णन करता है.,5.0,9.10,4.10,Scope/detail omission,Low to moderate,The explicit reference to using the services is omitted.
1,hi_tos_062,hi,tos_privacy,"हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव करके रख सकते हैं, अगर हमें लगता है कि यह, धोखाधड़ी या भविष्य में गलत इस्तेमाल को रोकने के लिए ज़रूरी है या किसी कानूनी/नियामक कार्यवाही को अधूरी रहने या उस बारे में कोई कानूनी और/या नियामक निर्देश मिलने की स्थिति में कानून के हिसाब से ज़रूरी हो।","हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धोखाधड़ी रोकने या कानूनी कार्यवाही को अधूरा नहीं रहने के लिए ज़रूरी हो।",5.0,8.99,3.99,Condition/purpose omission,High,Future misuse prevention and some legal or regulatory circumstances are omitted.
2,en_rental_046,en,rental,The Landlord may rel et the Premises or any part of the Premises for a term or terms which may be less or greater than the balance of the Term remaining and may grant reasonable concessions in connection with such reletting including any alterations and improvements to the Premises.,"The Landlord may sublet the Premises or part of the Premises for a term less or greater than the remaining Term and may offer reasonable concessions, including alterations and improvements to the Premises.",4.0,7.93,3.93,Potential terminology change,Needs legal review,Relet and sublet may not be legally equivalent in this lease context.


In [ ]:

import pandas as pd
import os

path = "/content/lmp_qualitative_case_analysis.csv"

if os.path.exists(path):
    cases = pd.read_csv(path)

    print("Shape:", cases.shape)
    print("\nColumns:")
    print(cases.columns.tolist())

    review_cols = [
        "Legal_Error_Type",
        "Legal_Impact",
        "Reviewer_Explanation"
    ]

    for col in review_cols:
        if col in cases.columns:
            print(f"\nMissing {col}:", cases[col].isna().sum())

    display(cases)
else:
    print("CSV not found:", path)

Shape: (15, 11)

Columns:
['ID', 'Language', 'Domain', 'Original', 'Simplified', 'LMP', 'Predicted_LMP', 'Absolute_Error', 'Legal_Error_Type', 'Legal_Impact', 'Reviewer_Explanation']

Missing Legal_Error_Type: 0

Missing Legal_Impact: 0

Missing Reviewer_Explanation: 0


,ID,Language,Domain,Original,Simplified,LMP,Predicted_LMP,Absolute_Error,Legal_Error_Type,Legal_Impact,Reviewer_Explanation
0,hi_tos_001,hi,tos_privacy,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेवाओं का उपयोग करते समय सहमत होते हैं.",उपयोग करते समय सहमत होने वाले नियमों का वर्णन करता है.,5.0,9.10,4.10,Scope/detail omission,Low to moderate,The explicit reference to using the services is omitted.
1,hi_tos_062,hi,tos_privacy,"हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव करके रख सकते हैं, अगर हमें लगता है कि यह, धोखाधड़ी या भविष्य में गलत इस्तेमाल को रोकने के लिए ज़रूरी है या किसी कानूनी/नियामक कार्यवाही को अधूरी रहने या उस बारे में कोई कानूनी और/या नियामक निर्देश मिलने की स्थिति में कानून के हिसाब से ज़रूरी हो।","हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धोखाधड़ी रोकने या कानूनी कार्यवाही को अधूरा नहीं रहने के लिए ज़रूरी हो।",5.0,8.99,3.99,Condition/purpose omission,High,Future misuse prevention and some legal or regulatory circumstances are omitted.
2,en_rental_046,en,rental,The Landlord may rel et the Premises or any part of the Premises for a term or terms which may be less or greater than the balance of the Term remaining and may grant reasonable concessions in connection with such reletting including any alterations and improvements to the Premises.,"The Landlord may sublet the Premises or part of the Premises for a term less or greater than the remaining Term and may offer reasonable concessions, including alterations and improvements to the Premises.",4.0,7.93,3.93,Potential terminology change,Needs legal review,Relet and sublet may not be legally equivalent in this lease context.
3,en_rental_057,en,rental,"Within 30 days of the termination of this lease (and Tenant vacating the Property), Landlord will provide the Tenants with an itemized statement showing any retention of any portions of the Security Deposit amount for the cost of repair or cleaning above and beyond the ordinary wear and tear on the Property or any other sums due to Landlord under this Residential Lease Agreement.","Within 30 days after the lease ends and Tenant moves out, Landlord will give Tenants a list showing any part of the Security Deposit kept for repairs or cleaning beyond normal wear and tear or other amounts owed under this agreement.",4.0,7.61,3.61,Condition/detail omission,Moderate,The explicit tenant-vacating condition and some itemization detail are omitted.
4,en_rental_035,en,rental,It is the intent of this Lease and agreed to by the Parties to this Lease that rent for this Lease will be on a gross rent basis meaning the Tenant will pay the Base Rent and any Additional Rent and the Landlord will be responsible for all other service charges related to the Premises and the operation of the Building save as specifically provided in this Lease to the contrary.,"It is agreed that the Tenant will pay the Base Rent and any Additional Rent, and the Landlord will cover all other service charges related to the Premises and the Building, except as specifically stated otherwise in this Lease.",4.0,7.60,3.60,No material change apparent,Low,The gross-rent arrangement and exception are substantially retained.
5,en_rental_038,en,rental,"Should the Tenant remain in possession of the Premises with the consent of the Landlord after the natural expiration of this Lease, a new tenancy from month to month will be created between the Landlord and the Tenant which will be subject to all the terms and conditions of this Lease but will be terminable upon either party giving one month's notice to the other party.","If the Tenant stays in the Premises with the Landlord's permission after the lease ends, a month-to-month tenancy will be created. This tenancy will follow all the terms of the original lease and can be ended by either party giving one month's notice.",4.0,7.34,3.34,No material change apparent,Low,"The tenancy type, original terms, and one-month notice are retained."
6,hi_tos_076,hi,tos_privacy,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाणीकरण सुनि

In [ ]:

import pandas as pd

path = "/content/lmp_qualitative_case_analysis.csv"
cases = pd.read_csv(path)

cols = ["ID", "LMP", "Predicted_LMP", "Absolute_Error"]

print("Missing values:")
print(cases[cols].isna().sum())

print("\nScore values:")
print(cases[cols].to_string(index=False))

print("\nData types:")
print(cases[cols].dtypes)

Missing values:
ID                0
LMP               0
Predicted_LMP     0
Absolute_Error    0
dtype: int64

Score values:
           ID  LMP  Predicted_LMP  Absolute_Error
   hi_tos_001  5.0           9.10            4.10
   hi_tos_062  5.0           8.99            3.99
en_rental_046  4.0           7.93            3.93
en_rental_057  4.0           7.61            3.61
en_rental_035  4.0           7.60            3.60
en_rental_038  4.0           7.34            3.34
   hi_tos_076  6.0           9.13            3.13
   hi_tos_061  6.0           9.12            3.12
   en_tos_117  5.0           8.11            3.11
hi_rental_047  6.0           9.09            3.09
   hi_tos_018  6.0           9.09            3.09
   hi_tos_029  6.0           9.08            3.08
   hi_tos_034  6.0           9.05            3.05
   en_tos_144  6.0           8.98            2.98
   en_tos_126  6.0           8.84            2.84

Data types:
ID                 object
LMP               float64
Predicted_L

In [ ]:

import os
import pandas as pd

files = {
    "Model comparison": "/content/lmp_model_comparison_clean.csv",
    "Held-out domain": "/content/held_out_domain_results.csv",
    "Sanity checks": "/content/lmp_sanity_check_results.csv",
    "Qualitative analysis": "/content/lmp_qualitative_case_analysis.csv",
}

for name, path in files.items():
    print(f"\n{'=' * 15} {name} {'=' * 15}")

    if os.path.exists(path):
        result = pd.read_csv(path)
        print("Shape:", result.shape)
        print("Columns:", result.columns.tolist())
        display(result)
    else:
        print("FILE NOT FOUND:", path)


=============== Model comparison ===============
Shape: (3, 5)
Columns: ['Model', 'Pearson', 'Spearman', 'RMSE', 'MAE']


,Model,Pearson,Spearman,RMSE,MAE
0,TF-IDF,0.5976,0.7100,1.1634,0.7924
1,MiniLM,0.5875,0.6574,1.1058,0.8477
2,XLM-R,0.5784,0.4195,1.2325,0.9754



=============== Held-out domain ===============
Shape: (2, 9)
Columns: ['Train_Domain', 'Test_Domain', 'Train_N', 'Validation_N', 'Test_N', 'RMSE', 'MAE', 'Pearson', 'Spearman']


,Train_Domain,Test_Domain,Train_N,Validation_N,Test_N,RMSE,MAE,Pearson,Spearman
0,rental,tos_privacy,110,20,227,3.666976,3.459630,0.028062,-0.001710
1,tos_privacy,rental,192,35,130,1.761515,1.539754,0.309955,0.301214



=============== Sanity checks ===============
Shape: (4, 7)
Columns: ['Case', 'Original', 'Simplified', 'TFIDF_LMP', 'MiniLM_Cosine', 'MiniLM_LMP', 'XLMR_LMP']


,Case,Original,Simplified,TFIDF_LMP,MiniLM_Cosine,MiniLM_LMP,XLMR_LMP
0,Identical,The tenant must pay rent on time.,The tenant must pay rent on time.,9.32,1.0000,9.32,8.95
1,Identical,We store personal information in India.,We store personal information in India.,9.32,1.0000,9.32,8.91
2,Unrelated,The tenant must pay rent on time.,The Earth revolves around the Sun.,1.76,0.0338,1.00,8.55
3,Unrelated,We store personal information in India.,The recipe requires two cups of flour.,1.00,-0.1198,1.00,8.72



=============== Qualitative analysis ===============
Shape: (15, 11)
Columns: ['ID', 'Language', 'Domain', 'Original', 'Simplified', 'LMP', 'Predicted_LMP', 'Absolute_Error', 'Legal_Error_Type', 'Legal_Impact', 'Reviewer_Explanation']


,ID,Language,Domain,Original,Simplified,LMP,Predicted_LMP,Absolute_Error,Legal_Error_Type,Legal_Impact,Reviewer_Explanation
0,hi_tos_001,hi,tos_privacy,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेवाओं का उपयोग करते समय सहमत होते हैं.",उपयोग करते समय सहमत होने वाले नियमों का वर्णन करता है.,5.0,9.10,4.10,Scope/detail omission,Low to moderate,The explicit reference to using the services is omitted.
1,hi_tos_062,hi,tos_privacy,"हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव करके रख सकते हैं, अगर हमें लगता है कि यह, धोखाधड़ी या भविष्य में गलत इस्तेमाल को रोकने के लिए ज़रूरी है या किसी कानूनी/नियामक कार्यवाही को अधूरी रहने या उस बारे में कोई कानूनी और/या नियामक निर्देश मिलने की स्थिति में कानून के हिसाब से ज़रूरी हो।","हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धोखाधड़ी रोकने या कानूनी कार्यवाही को अधूरा नहीं रहने के लिए ज़रूरी हो।",5.0,8.99,3.99,Condition/purpose omission,High,Future misuse prevention and some legal or regulatory circumstances are omitted.
2,en_rental_046,en,rental,The Landlord may rel et the Premises or any part of the Premises for a term or terms which may be less or greater than the balance of the Term remaining and may grant reasonable concessions in connection with such reletting including any alterations and improvements to the Premises.,"The Landlord may sublet the Premises or part of the Premises for a term less or greater than the remaining Term and may offer reasonable concessions, including alterations and improvements to the Premises.",4.0,7.93,3.93,Potential terminology change,Needs legal review,Relet and sublet may not be legally equivalent in this lease context.
3,en_rental_057,en,rental,"Within 30 days of the termination of this lease (and Tenant vacating the Property), Landlord will provide the Tenants with an itemized statement showing any retention of any portions of the Security Deposit amount for the cost of repair or cleaning above and beyond the ordinary wear and tear on the Property or any other sums due to Landlord under this Residential Lease Agreement.","Within 30 days after the lease ends and Tenant moves out, Landlord will give Tenants a list showing any part of the Security Deposit kept for repairs or cleaning beyond normal wear and tear or other amounts owed under this agreement.",4.0,7.61,3.61,Condition/detail omission,Moderate,The explicit tenant-vacating condition and some itemization detail are omitted.
4,en_rental_035,en,rental,It is the intent of this Lease and agreed to by the Parties to this Lease that rent for this Lease will be on a gross rent basis meaning the Tenant will pay the Base Rent and any Additional Rent and the Landlord will be responsible for all other service charges related to the Premises and the operation of the Building save as specifically provided in this Lease to the contrary.,"It is agreed that the Tenant will pay the Base Rent and any Additional Rent, and the Landlord will cover all other service charges related to the Premises and the Building, except as specifically stated otherwise in this Lease.",4.0,7.60,3.60,No material change apparent,Low,The gross-rent arrangement and exception are substantially retained.
5,en_rental_038,en,rental,"Should the Tenant remain in possession of the Premises with the consent of the Landlord after the natural expiration of this Lease, a new tenancy from month to month will be created between the Landlord and the Tenant which will be subject to all the terms and conditions of this Lease but will be terminable upon either party giving one month's notice to the other party.","If the Tenant stays in the Premises with the Landlord's permission after the lease ends, a month-to-month tenancy will be created. This tenancy will follow all the terms of the original lease and can be ended by either party giving one month's notice.",4.0,7.34,3.34,No material change apparent,Low,"The tenancy type, original terms, and one-month notice are retained."
6,hi_tos_076,hi,tos_privacy,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाणीकरण सुनि

In [ ]:

import pandas as pd

cases = pd.read_csv("/content/lmp_qualitative_case_analysis.csv")
comparison = pd.read_csv("/content/lmp_model_comparison_clean.csv")

print("QUALITATIVE REVIEW")
print("Total cases:", len(cases))
print("Missing review fields:")
print(cases[
    ["Legal_Error_Type", "Legal_Impact", "Reviewer_Explanation"]
].isna().sum())

print("\nCases with blank original or simplified text:")
print(cases[
    cases["Original"].isna() | cases["Simplified"].isna()
][["ID", "Original", "Simplified"]].to_string(index=False))

print("\nMODEL COMPARISON")
print(comparison.to_string(index=False))

print("\nSaved files:")
for path in [
    "/content/lmp_model_comparison_clean.csv",
    "/content/held_out_domain_results.csv",
    "/content/lmp_sanity_check_results.csv",
    "/content/lmp_qualitative_case_analysis.csv",
    "/content/lmp_prediction_results.csv",
]:
    import os
    print(f"{os.path.basename(path)}: {os.path.exists(path)}")

QUALITATIVE REVIEW
Total cases: 15
Missing review fields:
Legal_Error_Type        0
Legal_Impact            0
Reviewer_Explanation    0
dtype: int64

Cases with blank original or simplified text:
Empty DataFrame
Columns: [ID, Original, Simplified]
Index: []

MODEL COMPARISON
 Model  Pearson  Spearman   RMSE    MAE
TF-IDF   0.5976    0.7100 1.1634 0.7924
MiniLM   0.5875    0.6574 1.1058 0.8477
 XLM-R   0.5784    0.4195 1.2325 0.9754

Saved files:
lmp_model_comparison_clean.csv: True
held_out_domain_results.csv: True
lmp_sanity_check_results.csv: True
lmp_qualitative_case_analysis.csv: True
lmp_prediction_results.csv: True


In [ ]:

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

path = "/content/tfidf_clean_test_predictions.csv"
preds = pd.read_csv(path)

print("Shape:", preds.shape)
print("Columns:", preds.columns.tolist())
display(preds.head())

print("\nNumeric columns:")
print(preds.select_dtypes(include="number").columns.tolist())

Shape: (72, 8)
Columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP', 'TFIDF_Cosine', 'Predicted_LMP']


,ID,Original,Simplified,Language,Domain,LMP,TFIDF_Cosine,Predicted_LMP
0,hi_tos_018,दूसरों को या खुद को नुकसान न पहुंचाना या दूसरों के साथ या अपने साथ बुरा बर्ताव न करना (या न तो ऐसे नुकसान या बुरे बर्ताव की धमकी देना या इन्हें बढ़ावा देना).,दूसरों को या खुद को नुकसान न पहुंचाना और बुरा बर्ताव न करना।,hi,tos_privacy,6.0,0.782214,9.167916
1,en_tos_043,"If we are unable to resolve a dispute within sixty (60) days after the Notice of Dispute is received, you or we may commence arbitration.","If we cannot resolve a dispute within 60 days after receiving the Notice of Dispute, you or we can start arbitration.",en,tos_privacy,9.0,0.807597,9.276008
2,hi_rental_006,अनुबंध समाप्त करने के लिए \_\_\_\_\_\_ पूर्व लिखित सूचना देनी होगी।,अनुबंध समाप्त करने के लिए पूर्व लिखित सूचना देनी होगी।,hi,rental,10.0,1.000000,10.000000
3,hi_rental_015,बातचीत से समाधान न होने पर \_\_\_\_\_\_ के सक्षम न्यायालय में कार्यवाही की जा सकती है।,बातचीत से समाधान न होने पर कार्यवाही की जा सकती है \_\_\_\_\_\_ के सक्षम न्यायालय में।,hi,rental,8.0,0.937295,9.828324
4,en_tos_057,These business or user terms and conditions are available on the respective Participating Platforms.,These terms and conditions are available on the respective Participating Platforms.,en,tos_privacy,9.0,0.862690,9.510620



Numeric columns:
['LMP', 'TFIDF_Cosine', 'Predicted_LMP']


In [ ]:

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

preds = pd.read_csv("/content/tfidf_clean_test_predictions.csv")

y_true = pd.to_numeric(preds["LMP"], errors="coerce")
y_pred = pd.to_numeric(preds["Predicted_LMP"], errors="coerce")

valid = y_true.notna() & y_pred.notna()
y_true = y_true[valid].to_numpy()
y_pred = y_pred[valid].to_numpy()

print("Valid rows:", len(y_true))
print("Missing/invalid rows:", len(preds) - len(y_true))
print("Pearson:", round(float(pearsonr(y_true, y_pred).statistic), 4))
print("Spearman:", round(float(spearmanr(y_true, y_pred).statistic), 4))
print("RMSE:", round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4))
print("MAE:", round(float(mean_absolute_error(y_true, y_pred)), 4))

Valid rows: 72
Missing/invalid rows: 0
Pearson: 0.5976
Spearman: 0.71
RMSE: 1.1634
MAE: 0.7924


In [ ]:

import numpy as np

print("=== SPLIT INDEX CHECK ===")

required = ["train_idx", "val_idx", "test_idx"]

if all(name in globals() for name in required):
    train_set = set(map(int, train_idx))
    val_set = set(map(int, val_idx))
    test_set = set(map(int, test_idx))

    print("Train:", len(train_set))
    print("Validation:", len(val_set))
    print("Test:", len(test_set))
    print("Train/validation overlap:", len(train_set & val_set))
    print("Train/test overlap:", len(train_set & test_set))
    print("Validation/test overlap:", len(val_set & test_set))
    print("Total unique indexes:", len(train_set | val_set | test_set))
else:
    print("Split index variables missing:", [
        name for name in required if name not in globals()
    ])

print("\n=== TRAINER DATASETS ===")

if "trainer" in globals():
    print("Trainer class:", type(trainer).__name__)
    print("Train dataset rows:", len(trainer.train_dataset))
    print("Validation dataset rows:", len(trainer.eval_dataset))
    print("Training arguments:")
    for key in [
        "num_train_epochs",
        "learning_rate",
        "seed",
        "data_seed",
        "load_best_model_at_end",
        "metric_for_best_model",
    ]:
        print(f"{key}:", getattr(trainer.args, key, "Not available"))
else:
    print("Variable 'trainer' not found.")

print("\n=== SAVED XLM-R OUTPUT ===")

import os
path = "/content/xlmr_test_predictions_audit.csv"

if os.path.exists(path):
    import pandas as pd
    saved = pd.read_csv(path)
    print("Saved test predictions:", len(saved))
    print("Columns:", saved.columns.tolist())
    print("Unique test IDs:", saved["ID"].nunique() if "ID" in saved else "ID column missing")
else:
    print("Audit prediction file not found:", path)

=== SPLIT INDEX CHECK ===
Train: 285
Validation: 43
Test: 72
Train/validation overlap: 43
Train/test overlap: 0
Validation/test overlap: 0
Total unique indexes: 357

=== TRAINER DATASETS ===
Trainer class: Trainer
Train dataset rows: 242
Validation dataset rows: 43
Training arguments:
num_train_epochs: 5
learning_rate: 2e-05
seed: 42
data_seed: None
load_best_model_at_end: True
metric_for_best_model: eval_rmse

=== SAVED XLM-R OUTPUT ===
Saved test predictions: 72
Columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP', 'Predicted_LMP', 'Preservation_Decision']
Unique test IDs: 72


In [ ]:

import pandas as pd
import numpy as np

# Reconstruct the intended split from the full model dataset.
all_idx = np.arange(len(model_df))

from sklearn.model_selection import train_test_split

expected_train_idx, expected_test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42
)
expected_train_idx, expected_val_idx = train_test_split(
    expected_train_idx, test_size=0.15, random_state=42
)

expected_train_ids = set(model_df.iloc[expected_train_idx]["ID"].astype(str))
expected_val_ids = set(model_df.iloc[expected_val_idx]["ID"].astype(str))
expected_test_ids = set(model_df.iloc[expected_test_idx]["ID"].astype(str))

saved_test = pd.read_csv("/content/xlmr_test_predictions_audit.csv")
saved_test_ids = set(saved_test["ID"].astype(str))

print("=== RECONSTRUCTED SPLIT ===")
print("Train:", len(expected_train_ids))
print("Validation:", len(expected_val_ids))
print("Test:", len(expected_test_ids))
print("Expected test IDs match saved predictions:",
      expected_test_ids == saved_test_ids)

print("\n=== TRAINER DATASET STRUCTURE ===")
print("Train columns:", trainer.train_dataset.column_names)
print("Validation columns:", trainer.eval_dataset.column_names)

print("\n=== ORIGINAL TRAINING DATA EVIDENCE ===")
if "train_df" in globals():
    print("train_df rows:", len(train_df))
    print("train_df columns:", train_df.columns.tolist())
    if "ID" in train_df.columns:
        actual_train_ids = set(train_df["ID"].astype(str))
        print("train_df IDs match expected training IDs:",
              actual_train_ids == expected_train_ids)
else:
    print("No train_df variable found; actual training IDs are not directly available.")

=== RECONSTRUCTED SPLIT ===
Train: 242
Validation: 43
Test: 72
Expected test IDs match saved predictions: True

=== TRAINER DATASET STRUCTURE ===
Train columns: ['Original', 'Simplified', 'labels', 'input_ids', 'attention_mask']
Validation columns: ['Original', 'Simplified', 'labels', 'input_ids', 'attention_mask']

=== ORIGINAL TRAINING DATA EVIDENCE ===
train_df rows: 242
train_df columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP']
train_df IDs match expected training IDs: True


In [ ]:
print("train_df matches expected split:",
      set(train_df["ID"].astype(str)) == expected_train_ids)

print("val_df available:", "val_df" in globals())
if "val_df" in globals() and "ID" in val_df.columns:
    print("val_df rows:", len(val_df))
    print("val_df matches expected split:",
          set(val_df["ID"].astype(str)) == expected_val_ids)

print("\nTrainer train rows:", len(trainer.train_dataset))
print("Trainer validation rows:", len(trainer.eval_dataset))

print("\nTraining history:")
for item in trainer.state.log_history:
    if "epoch" in item and (
        "eval_rmse" in item or "loss" in item
    ):
        print(item)

train_df matches expected split: True
val_df available: True
val_df rows: 43
val_df matches expected split: True

Trainer train rows: 242
Trainer validation rows: 43

Training history:
{'loss': 61.921244959677416, 'grad_norm': 727.2415771484375, 'learning_rate': 1.6129032258064517e-05, 'epoch': 1.0, 'step': 31}
{'eval_loss': 26.063982009887695, 'eval_rmse': 5.105289610775054, 'eval_mae': 4.921238899230957, 'eval_pearson': 0.11395490169525146, 'eval_spearman': 0.13696668188141883, 'eval_runtime': 0.2099, 'eval_samples_per_second': 204.865, 'eval_steps_per_second': 28.586, 'epoch': 1.0, 'step': 31}
{'loss': 20.119991179435484, 'grad_norm': 138.05906677246094, 'learning_rate': 1.2129032258064518e-05, 'epoch': 2.0, 'step': 62}
{'eval_loss': 6.526846885681152, 'eval_rmse': 2.5547691588810286, 'eval_mae': 2.3210391998291016, 'eval_pearson': -0.3471970558166504, 'eval_spearman': -0.36035296616527684, 'eval_runtime': 0.3317, 'eval_samples_per_second': 129.625, 'eval_steps_per_second': 18.087, 

In [ ]:
cd /path/to/your/project
git status --short
git branch --show-current
ls -la
find . -maxdepth 2 -type f \
  -not -path './.git/*' \
  -not -path './.venv/*' \
  -not -path './venv/*' | sort | head -200

SyntaxError: invalid syntax (3334773135.py, line 2)

In [ ]:
import os

print("Current directory:", os.getcwd())
print("\nFiles in current directory:")
print(os.listdir("."))

Current directory: /content

Files in current directory:
['.config', 'domain_exp_tos_privacy_to_rental', 'lmp_final - lmp_semantic_review(1).csv', 'judgebert_lmp_results', 'lmp_qualitative_case_analysis.csv', 'xlmr_test_predictions_audit.csv', 'lmp_split_audit.csv', 'judgebert_lmp_weighted', 'generalization_by_domain.csv', 'lmp_sanity_check_results.csv', 'lmp_sanity_checks.csv', 'judgebert_lmp_final', 'minilm_clean_test_predictions.csv', 'domain_exp_rental_to_tos_privacy', 'held_out_domain_results.csv', 'lmp_model_comparison_clean.csv', 'tfidf_clean_test_predictions.csv', 'lmp_prediction_results.csv', 'generalization_by_language.csv', 'sample_data']


In [ ]:
!git status --short
!git branch --show-current
!ls -la

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
total 528
drwxr-xr-x 1 root root   4096 Oct  9 18:00  .
drwxr-xr-x 1 root root   4096 Oct  9 17:00  ..
drwxr-xr-x 4 root root   4096 Sep 30 13:28  .config
drwxr-xr-x 3 root root   4096 Oct  9 17:33  domain_exp_rental_to_tos_privacy
drwxr-xr-x 3 root root   4096 Oct  9 17:40  domain_exp_tos_privacy_to_rental
-rw-r--r-- 1 root root    187 Oct  9 17:24  generalization_by_domain.csv
-rw-r--r-- 1 root root    177 Oct  9 17:24  generalization_by_language.csv
-rw-r--r-- 1 root root    294 Oct  9 17:40  held_out_domain_results.csv
drwxr-xr-x 2 root root   4096 Oct  9 17:17  judgebert_lmp_final
drwxr-xr-x 7 root root   4096 Oct  9 17:15  judgebert_lmp_results
drwxr-xr-x 7 root root   4096 Oct  9 18:07  judgebert_lmp_weighted
-rw-r--r-- 1 root root 214669 Oct  9 17:08 'lmp_final - lmp_semantic_review(1).csv'
-rw-r--r-- 1 root root    134 Oct  9 17:51  lmp_mode

In [ ]:
import os

for folder in [
    "/content/judgebert_lmp_final",
    "/content/judgebert_lmp_results",
    "/content/judgebert_lmp_weighted",
    "/content/domain_exp_tos_privacy_to_rental",
    "/content/domain_exp_rental_to_tos_privacy",
]:
    print(f"\n--- {folder} ---")
    if os.path.isdir(folder):
        for root, dirs, files in os.walk(folder):
            dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]
            level = root.replace(folder, "").count(os.sep)
            if level > 2:
                dirs[:] = []
                continue
            print(root, "->", files[:15])
    else:
        print("Not found")


--- /content/judgebert_lmp_final ---
/content/judgebert_lmp_final -> ['model.safetensors', 'tokenizer.json', 'config.json', 'tokenizer_config.json', 'training_args.bin']

--- /content/judgebert_lmp_results ---
/content/judgebert_lmp_results -> []
/content/judgebert_lmp_results/checkpoint-62 -> ['scheduler.pt', 'trainer_state.json', 'model.safetensors', 'tokenizer.json', 'config.json', 'optimizer.pt', 'tokenizer_config.json', 'training_args.bin', 'scaler.pt', 'rng_state.pth']
/content/judgebert_lmp_results/checkpoint-93 -> ['scheduler.pt', 'trainer_state.json', 'model.safetensors', 'tokenizer.json', 'config.json', 'optimizer.pt', 'tokenizer_config.json', 'training_args.bin', 'scaler.pt', 'rng_state.pth']
/content/judgebert_lmp_results/checkpoint-155 -> ['scheduler.pt', 'trainer_state.json', 'model.safetensors', 'tokenizer.json', 'config.json', 'optimizer.pt', 'tokenizer_config.json', 'training_args.bin', 'scaler.pt', 'rng_state.pth']
/content/judgebert_lmp_results/checkpoint-124 -> ['s

In [ ]:
import os

for folder in [
    "/content/judgebert_lmp_final",
    "/content/judgebert_lmp_results",
    "/content/judgebert_lmp_weighted",
]:
    print(f"\n{'=' * 15} {folder} {'=' * 15}")

    for root, dirs, files in os.walk(folder):
        dirs[:] = [
            d for d in dirs
            if d not in {".git", "__pycache__"}
        ]

        depth = root.replace(folder, "").count(os.sep)
        if depth > 2:
            dirs[:] = []
            continue

        for filename in files:
            path = os.path.join(root, filename)
            print(
                os.path.relpath(path, folder),
                f"({os.path.getsize(path) / 1024:.1f} KB)"
            )


=============== /content/judgebert_lmp_final ===============
model.safetensors (1086134.7 KB)
tokenizer.json (16697.4 KB)
config.json (0.9 KB)
tokenizer_config.json (0.3 KB)
training_args.bin (5.1 KB)

=============== /content/judgebert_lmp_results ===============
checkpoint-62/scheduler.pt (1.4 KB)
checkpoint-62/trainer_state.json (1.8 KB)
checkpoint-62/model.safetensors (1086134.7 KB)
checkpoint-62/tokenizer.json (16697.4 KB)
checkpoint-62/config.json (0.9 KB)
checkpoint-62/optimizer.pt (2172389.4 KB)
checkpoint-62/tokenizer_config.json (0.3 KB)
checkpoint-62/training_args.bin (5.1 KB)
checkpoint-62/scaler.pt (1.4 KB)
checkpoint-62/rng_state.pth (14.3 KB)
checkpoint-93/scheduler.pt (1.4 KB)
checkpoint-93/trainer_state.json (2.4 KB)
checkpoint-93/model.safetensors (1086134.7 KB)
checkpoint-93/tokenizer.json (16697.4 KB)
checkpoint-93/config.json (0.9 KB)
checkpoint-93/optimizer.pt (2172389.4 KB)
checkpoint-93/tokenizer_config.json (0.3 KB)
checkpoint-93/training_args.bin (5.1 KB)
che

In [ ]:
import os

project_dir = "/content/judgebert_lmp_submission"

print("Submission folder exists:", os.path.isdir(project_dir))

if os.path.isdir(project_dir):
    for root, dirs, files in os.walk(project_dir):
        level = root.replace(project_dir, "").count(os.sep)
        if level > 2:
            dirs[:] = []
            continue
        print(root, "->", files)
else:
    print("The folder has not been created yet.")

Submission folder exists: False
The folder has not been created yet.


In [ ]:
import os
import shutil

project_dir = "/content/judgebert_lmp_submission"

# Create submission folders
for folder in ["results", "notebooks"]:
    os.makedirs(os.path.join(project_dir, folder), exist_ok=True)

# Copy verified experiment outputs
result_files = [
    "lmp_model_comparison_clean.csv",
    "held_out_domain_results.csv",
    "lmp_sanity_check_results.csv",
    "lmp_qualitative_case_analysis.csv",
    "lmp_split_audit.csv",
    "tfidf_clean_test_predictions.csv",
    "minilm_clean_test_predictions.csv",
    "xlmr_test_predictions_audit.csv",
    "lmp_prediction_results.csv",
    "generalization_by_domain.csv",
    "generalization_by_language.csv",
]

copied, missing = [], []

for filename in result_files:
    source = f"/content/{filename}"
    destination = os.path.join(project_dir, "results", filename)

    if os.path.isfile(source):
        shutil.copy2(source, destination)
        copied.append(filename)
    else:
        missing.append(filename)

print("Project directory:", project_dir)
print("\nCopied files:", len(copied))
for filename in copied:
    print("✅", filename)

print("\nMissing files:", missing if missing else "None")

Project directory: /content/judgebert_lmp_submission

Copied files: 11
✅ lmp_model_comparison_clean.csv
✅ held_out_domain_results.csv
✅ lmp_sanity_check_results.csv
✅ lmp_qualitative_case_analysis.csv
✅ lmp_split_audit.csv
✅ tfidf_clean_test_predictions.csv
✅ minilm_clean_test_predictions.csv
✅ xlmr_test_predictions_audit.csv
✅ lmp_prediction_results.csv
✅ generalization_by_domain.csv
✅ generalization_by_language.csv

Missing files: None


In [ ]:

from pathlib import Path

project_dir = Path("/content/judgebert_lmp_submission")

readme = r"""# Explainable Multilingual Legal Meaning Preservation

## Project Overview

This project investigates whether simplified legal text preserves the meaning of its original version in English and Hindi.

It compares two semantic similarity baselines with a JUDGEBERT-inspired XLM-RoBERTa regression model. The model predicts a Legal Meaning Preservation (LMP) score on a 1–10 scale.

**Important:** This is a JUDGEBERT-inspired approach, not an exact reproduction of the original JUDGEBERT implementation.

## Objectives

- Estimate meaning preservation between original and simplified legal sentences.
- Compare TF-IDF, Sentence Transformers (MiniLM), and XLM-RoBERTa regression.
- Evaluate generalization across legal domains.
- Test identical and unrelated sentence pairs.
- Analyze legally important changes such as omitted conditions, exceptions, scope, and retention periods.
- Produce scores, provisional preservation decisions, and qualitative explanations.

## Dataset

- Total sentence pairs: 357
- Languages: English and Hindi
- Domains: rental agreements and terms/privacy policies
- Target: supplied LMP ratings on a 1–10 scale

The dataset contains a high proportion of ratings between 8 and 10. Results should therefore be interpreted in light of label imbalance and dataset size.

Human-rating provenance and annotation agreement should be confirmed before making claims about independently validated human judgments.

## Methods

### Baselines

1. **TF-IDF:** Text representations with a linear regression calibration fitted on training data.
2. **MiniLM:** Sentence embeddings with a linear regression calibration fitted on training data.
3. **XLM-RoBERTa:** A transformer regression model trained to predict LMP ratings.

The clean baseline comparisons use the same reconstructed train/validation/test split. TF-IDF fitting and baseline calibration are restricted to training data.

### Evaluation Metrics

- Pearson correlation
- Spearman rank correlation
- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)

### Generalization Experiment

Models were evaluated on a domain excluded from their training and validation data:
- Train on rental agreements; test on terms/privacy policies.
- Train on terms/privacy policies; test on rental agreements.

### Qualitative and Sanity Checks

The project includes identical-pair and unrelated-pair checks, plus 15 manually reviewed cases involving potentially legally meaningful changes. These qualitative assessments are preliminary and are not legal advice.

## Results

### Clean Test-Set Comparison

| Model | Pearson | Spearman | RMSE | MAE |
|---|---:|---:|---:|---:|
| TF-IDF | 0.5976 | 0.7100 | 1.1634 | 0.7924 |
| MiniLM | 0.5875 | 0.6574 | 1.1058 | 0.8477 |
| XLM-RoBERTa | 0.5784 | 0.4195 | 1.2325 | 0.9754 |

TF-IDF achieved the highest Pearson and Spearman correlations and the lowest MAE. MiniLM achieved the lowest RMSE. XLM-RoBERTa did not outperform these baselines on the reported test metrics.

These are descriptive comparisons; no statistical significance testing was performed.

### Cross-Domain Generalization

| Training domain | Test domain | RMSE | MAE | Pearson | Spearman |
|---|---|---:|---:|---:|---:|
| Rental | Terms/privacy | 3.6670 | 3.4596 | 0.0281 | -0.0017 |
| Terms/privacy | Rental | 1.7615 | 1.5398 | 0.3100 | 0.3012 |

The results indicate weak cross-domain generalization, especially when training on rental agreements and testing on terms/privacy policies.

### Sanity-Check Finding

The XLM-RoBERTa model assigned high scores to some unrelated sentence pairs. This is an important limitation: a high predicted LMP score should not be treated as proof that legal meaning is preserved.

## Preservation Decisions

The prediction output uses a provisional threshold of 8.0:
- Score >= 8: `Preserved`
- Score < 8: `Review: possible meaning loss`

This threshold has not been independently validated. Decisions should be reviewed by a qualified human, especially where obligations, exceptions, conditions, dates, amounts, or legal rights may have changed.

## Repository Structure

```text
judgebert_lmp_submission/
├── README.md
├── requirements.txt
├── notebooks/
└── results/
    ├── lmp_model_comparison_clean.csv
    ├── held_out_domain_results.csv
    ├── lmp_sanity_check_results.csv
    ├── lmp_qualitative_case_analysis.csv
    ├── lmp_split_audit.csv
    ├── tfidf_clean_test_predictions.csv
    ├── minilm_clean_test_predictions.csv
    ├── xlmr_test_predictions_audit.csv
    ├── lmp_prediction_results.csv
    ├── generalization_by_domain.csv
    └── generalization_by_language.csv
```

## Reproducibility

The `notebooks/` directory is intended to contain the experiment notebook. The current results and prediction files are included for inspection.

Before claiming full reproducibility, add the complete notebook, pin the software dependencies, document the dataset source and preprocessing, and verify that the experiments can be rerun from a clean runtime.

## Limitations

- Small dataset with an imbalanced LMP label distribution.
- Performance does not establish legal correctness.
- XLM-RoBERTa produced suspiciously high scores for some unrelated pairs.
- Cross-domain generalization was weak.
- The preservation threshold is provisional.
- Qualitative reviews are preliminary and should be independently checked.
- Results are not a substitute for legal review.

## References

- GRAAL Research, JUDGEBERT: https://github.com/GRAAL-Research/JUDGEBERT
- XLM-RoBERTa: https://huggingface.co/FacebookAI/xlm-roberta-base
- Sentence Transformers: https://www.sbert.net/

## Disclaimer

This is an academic research prototype for evaluating legal meaning preservation. It is not a legal advice system and should not be used to make legal decisions without qualified human review.
"""

(project_dir / "README.md").write_text(readme, encoding="utf-8")

print("README created:", project_dir / "README.md")
print("Size:", (project_dir / "README.md").stat().st_size, "bytes")

README created: /content/judgebert_lmp_submission/README.md
Size: 5899 bytes


In [ ]:
import os

print("Possible notebooks in /content:")
for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in [".config", "sample_data"]]
    for file in files:
        if file.endswith(".ipynb"):
            print(os.path.join(root, file))

Possible notebooks in /content:


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    dirs[:] = [
        d for d in dirs
        if d not in [".trash", ".ipynb_checkpoints"]
    ]
    for file in files:
        if file.endswith(".ipynb"):
            matches.append(os.path.join(root, file))

print("Notebook files found:", len(matches))
for path in matches:
    print(path)

Notebook files found: 1
/content/drive/MyDrive/Colab Notebooks/IMD_LAB6.ipynb


In [ ]:
import json
from pathlib import Path

model_dir = Path("/content/judgebert_lmp_final")
config_path = model_dir / "config.json"

if config_path.exists():
    config = json.loads(config_path.read_text())
    for key in [
        "model_type",
        "architectures",
        "num_labels",
        "hidden_size",
        "max_position_embeddings"
    ]:
        print(f"{key}: {config.get(key)}")
else:
    print("Model config not found.")

model_type: xlm-roberta
architectures: ['XLMRobertaForSequenceClassification']
num_labels: None
hidden_size: 768
max_position_embeddings: 514


In [ ]:
from pathlib import Path

dataset_path = Path("/content/lmp_final - lmp_semantic_review(1).csv")

print("Dataset exists:", dataset_path.exists())
if dataset_path.exists():
    print("Dataset size (MB):", round(dataset_path.stat().st_size / (1024 * 1024), 2))

Dataset exists: True
Dataset size (MB): 0.2


In [ ]:
import pandas as pd

dataset_path = "/content/lmp_final - lmp_semantic_review(1).csv"
df = pd.read_csv(dataset_path)

# Normalize the target column name for inspection
df["LMP score"] = pd.to_numeric(df["LMP score"], errors="coerce")

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

print("\nLanguage counts:")
print(df["Language"].value_counts(dropna=False))

print("\nLMP score summary:")
print(df["LMP score"].describe())

print("\nMissing values:")
print(df[["Original", "Simplified", "Language", "Domain", "LMP score"]].isna().sum())

pred_path = "/content/lmp_prediction_results.csv"
predictions = pd.read_csv(pred_path)

print("\nPrediction shape:", predictions.shape)
print("Prediction columns:", predictions.columns.tolist())

Dataset shape: (357, 8)

Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment']

Language counts:
Language
en    205
hi    152
Name: count, dtype: int64

LMP score summary:
count    357.000000
mean       8.703081
std        1.332952
min        4.000000
25%        8.000000
50%        9.000000
75%       10.000000
max       10.000000
Name: LMP score, dtype: float64

Missing values:
Original      0
Simplified    0
Language      0
Domain        0
LMP score     0
dtype: int64

Prediction shape: (357, 8)
Prediction columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP', 'Predicted_LMP', 'Preservation_Decision']


In [ ]:

import json
from pathlib import Path

project_dir = Path("/content/judgebert_lmp_submission")
notebook_dir = project_dir / "notebooks"
notebook_dir.mkdir(parents=True, exist_ok=True)

def markdown_cell(source):
    return {
        "cell_type": "markdown",
        "metadata": {},
        "source": source.splitlines(keepends=True)
    }

def code_cell(source):
    return {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": source.splitlines(keepends=True)
    }

cells = [
    markdown_cell("""# Explainable Multilingual Legal Meaning Preservation

## JUDGEBERT-inspired XLM-RoBERTa Regression

This notebook documents the verified outputs of an academic project evaluating whether simplified legal text preserves the meaning of the original in English and Hindi.

**Reproducibility note:** The original experiment notebook was not found in the current Colab runtime or the inspected Drive location. This notebook is a reconstructed analysis notebook for the saved results. It does not claim to reproduce the original training run exactly.

The model is JUDGEBERT-inspired, not an exact reproduction of the original JUDGEBERT implementation.
"""),
    markdown_cell("""## 1. Dataset and evaluation setup

The supplied dataset contains 357 sentence pairs in English and Hindi. Its target column is `LMP score`, with ratings from 1 to 10. The dataset's assessment provenance should be verified before describing the ratings as independently validated human annotations.

The reported clean baseline comparison uses a common reconstructed split: 242 training, 43 validation, and 72 test rows. Baseline fitting was restricted to training data. See `results/lmp_split_audit.csv` for the split audit.
"""),
    code_cell("""from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/judgebert_lmp_submission")
RESULTS = ROOT / "results"
DATA_PATH = Path("/content/lmp_final - lmp_semantic_review(1).csv")

df = pd.read_csv(DATA_PATH)
df["LMP score"] = pd.to_numeric(df["LMP score"], errors="raise")

print("Dataset shape:", df.shape)
print("Languages:")
display(df["Language"].value_counts().rename_axis("Language").to_frame("Count"))
print("LMP score summary:")
display(df["LMP score"].describe().to_frame("LMP score"))
"""),
    markdown_cell("""## 2. Clean baseline comparison

The following metrics are read from the saved comparison CSV. They are descriptive test-set results; no statistical significance testing was performed.
"""),
    code_cell("""comparison = pd.read_csv(RESULTS / "lmp_model_comparison_clean.csv")
display(comparison)
"""),
    markdown_cell("""## 3. Cross-domain generalization

Two experiments evaluate each model on a domain not used for its training or validation data. Performance is weak in the rental-to-terms/privacy direction.
"""),
    code_cell("""held_out = pd.read_csv(RESULTS / "held_out_domain_results.csv")
display(held_out)
"""),
    markdown_cell("""## 4. Identical and unrelated sentence sanity checks

The saved checks show that XLM-RoBERTa assigns high scores to some unrelated sentence pairs. This is a major limitation: predicted scores must not be interpreted as proof of legal equivalence.
"""),
    code_cell("""sanity = pd.read_csv(RESULTS / "lmp_sanity_check_results.csv")
display(sanity)
"""),
    markdown_cell("""## 5. Qualitative legal-error analysis

Fifteen manually reviewed examples cover possible omissions, scope changes, conditions, retention periods, technical details, and terminology changes. These assessments are preliminary and are not legal advice.
"""),
    code_cell("""qualitative = pd.read_csv(RESULTS / "lmp_qualitative_case_analysis.csv")
print("Cases:", len(qualitative))
print("Missing values per column:")
display(qualitative.isna().sum().to_frame("Missing"))
display(qualitative)
"""),
    markdown_cell("""## 6. Prediction output

The full prediction file contains scores and provisional preservation decisions. The threshold of 8.0 is not independently validated and should be treated as a review heuristic.
"""),
    code_cell("""predictions = pd.read_csv(RESULTS / "lmp_prediction_results.csv")
display(predictions.head())
print("Prediction rows:", len(predictions))
print("Decision counts:")
display(predictions["Preservation_Decision"].value_counts().to_frame("Count"))
"""),
    markdown_cell("""## 7. Limitations and conclusions

- The dataset is small and its ratings are concentrated toward the high end.
- TF-IDF had the strongest Pearson/Spearman correlations and lowest MAE among the reported clean test baselines; MiniLM had the lowest RMSE.
- XLM-RoBERTa did not outperform the baselines on these metrics.
- The transformer assigned high scores to some unrelated pairs.
- Cross-domain generalization was weak.
- Qualitative review and the threshold-based decision are preliminary.
- Legal meaning preservation requires qualified human review, especially for obligations, exceptions, conditions, dates, amounts, entities, and rights.

This notebook analyzes saved experiment artifacts. It does not independently retrain the models or prove that rerunning training will reproduce the recorded metrics.
""")
]

notebook = {
    "cells": cells,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {"name": "python"}
    },
    "nbformat": 4,
    "nbformat_minor": 5
}

out = notebook_dir / "LMP_results_analysis.ipynb"
out.write_text(json.dumps(notebook, indent=2, ensure_ascii=False), encoding="utf-8")
print("Created:", out)
print("Notebook cells:", len(cells))

Created: /content/judgebert_lmp_submission/notebooks/LMP_results_analysis.ipynb
Notebook cells: 14


In [ ]:
from pathlib import Path
import json

root = Path("/content/judgebert_lmp_submission")
nb_path = root / "notebooks" / "LMP_results_analysis.ipynb"

nb = json.loads(nb_path.read_text(encoding="utf-8"))

print("Notebook exists:", nb_path.exists())
print("Notebook cells:", len(nb["cells"]))
print("Result CSVs:", len(list((root / "results").glob("*.csv"))))
print("README exists:", (root / "README.md").exists())

Notebook exists: True
Notebook cells: 14
Result CSVs: 11
README exists: True


In [ ]:
import os

print("Current working directory:", os.getcwd())

from google.colab import drive
print("\nChecking common notebook locations:")

for root in [
    "/content/drive/MyDrive/Colab Notebooks",
    "/content"
]:
    if os.path.isdir(root):
        for name in os.listdir(root):
            if "LMP_Baseline_Experiments" in name:
                print(os.path.join(root, name))

Current working directory: /content

Checking common notebook locations:
/content/drive/MyDrive/Colab Notebooks/LMP_Baseline_Experimentsipynb


In [ ]:
import os

for root in [
    "/content/drive/MyDrive/Colab Notebooks",
    "/content"
]:
    if os.path.isdir(root):
        for name in os.listdir(root):
            if "LMP_Baseline_Experiments" in name:
                print(os.path.join(root, name))

/content/drive/MyDrive/Colab Notebooks/LMP_Baseline_Experimentsipynb


In [ ]:
from pathlib import Path
import shutil

source = Path(
    "/content/drive/MyDrive/Colab Notebooks/LMP_Baseline_Experimentsipynb"
)

destination = Path(
    "/content/judgebert_lmp_submission/notebooks/LMP_Baseline_Experiments.ipynb"
)

if not source.is_file():
    raise FileNotFoundError(f"Notebook not found: {source}")

shutil.copy2(source, destination)

print("Copied successfully:", destination)
print("File size (MB):", round(destination.stat().st_size / (1024 * 1024), 2))

Copied successfully: /content/judgebert_lmp_submission/notebooks/LMP_Baseline_Experiments.ipynb
File size (MB): 1.28


In [ ]:
import shutil
from pathlib import Path

root = Path("/content/judgebert_lmp_submission")
zip_base = "/content/judgebert_lmp_submission_final"

zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir="/content",
    base_dir=root.name
)

print("Final ZIP:", zip_path)
print("Size (MB):", round(Path(zip_path).stat().st_size / (1024 * 1024), 2))

from google.colab import files
files.download(zip_path)

Final ZIP: /content/judgebert_lmp_submission_final.zip
Size (MB): 0.21


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# STEP 1: Check GPU, environment, and project files
import os
import sys
import glob
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Enable a GPU in Runtime settings.")

print("\nCurrent directory:", os.getcwd())
print("\nFiles in /content:")
for item in sorted(os.listdir("/content"))[:50]:
    print(" -", item)

print("\nSearching common locations for your project artifacts...")
patterns = [
    "/content/**/lmp_split_audit.csv",
    "/content/**/lmp_model_comparison_clean.csv",
    "/content/**/LMP_Baseline_Experiments.ipynb",
    "/content/**/tfidf_clean_test_predictions.csv",
    "/content/**/xlmr_test_predictions_audit.csv",
]

for pattern in patterns:
    matches = glob.glob(pattern, recursive=True)
    print(f"\n{os.path.basename(pattern)}:")
    print(matches[:10] if matches else "Not found")


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

Current directory: /content

Files in /content:
 - .config
 - lmp_final - lmp_semantic_review(1).csv
 - sample_data

Searching common locations for your project artifacts...

lmp_split_audit.csv:
Not found

lmp_model_comparison_clean.csv:
Not found

LMP_Baseline_Experiments.ipynb:
Not found

tfidf_clean_test_predictions.csv:
Not found

xlmr_test_predictions_audit.csv:
Not found


In [ ]:

# STEP 2: Inspect the existing LMP dataset
import pandas as pd
import os

DATA_PATH = "/content/lmp_final - lmp_semantic_review(1).csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))

print("\nPossible language columns:")
for col in df.columns:
    if any(word in col.lower() for word in ["language", "lang"]):
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False))

print("\nPossible score columns:")
for col in df.columns:
    if any(word in col.lower() for word in ["lmp", "score", "rating"]):
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False).sort_index())


Dataset shape: (357, 8)

Column names:
['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment']

First 5 rows:


,ID,Original,Language,Domain,Source,Simplified,LMP score,Human_Assessment
0,en_tos_001,"As a result, these Terms of Service help defin...",en,tos_privacy,google_tos,"Because of this, these Terms of Service help d...",10,Legal meaning and key details preserved — Key ...
1,en_tos_002,"Our services are designed to work together, ma...",en,tos_privacy,google_tos,Our services are designed to work together so ...,9,Legal meaning and key details preserved — Key ...
2,en_tos_003,"We also make various policies, help centers, a...",en,tos_privacy,google_tos,"We also provide various policies, help centers...",9,Legal meaning and key details preserved — Key ...
3,en_tos_004,"comply with applicable laws, including export ...",en,tos_privacy,google_tos,"Comply with applicable laws, including export ...",10,Legal meaning and key details preserved — Key ...
4,en_tos_005,using our services (including the content they...,en,tos_privacy,google_tos,Do not use our services (including the content...,10,Legal meaning and key details preserved — Key ...



Data types:
ID                  object
Original            object
Language            object
Domain              object
Source              object
Simplified          object
LMP score            int64
Human_Assessment    object
dtype: object

Missing values:


,missing_count
ID,0
Original,0
Language,0
Domain,0
Source,0
Simplified,0
LMP score,0
Human_Assessment,0



Possible language columns:

Language:
Language
en    205
hi    152
Name: count, dtype: int64

Possible score columns:

LMP score:
LMP score
4       4
5       5
6      17
7      34
8      71
9     102
10    124
Name: count, dtype: int64


In [ ]:

# STEP 3: Dataset audit and source-grouped split
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("/content/lmp_final - lmp_semantic_review(1).csv")

# Basic integrity checks
assert df["ID"].is_unique, "Duplicate IDs found"
assert df[["Original", "Simplified", "Language", "Domain", "Source", "LMP score"]].notna().all().all()

print("DATASET")
print("Rows:", len(df))
print("Unique sources:", df["Source"].nunique())
print("Duplicate IDs:", df["ID"].duplicated().sum())
print("Duplicate sentence pairs:",
      df.duplicated(subset=["Original", "Simplified"]).sum())

print("\nLMP distribution:")
print(df["LMP score"].value_counts().sort_index())

print("\nLanguage × Domain:")
print(pd.crosstab(df["Language"], df["Domain"]))

print("\nSource counts:")
print(df["Source"].value_counts())

# Make a source-grouped split: approximately 70% train, 15% validation, 15% test.
# This is a NEW evaluation split; it does not replace your original benchmark.
gss1 = GroupShuffleSplit(n_splits=100, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(df, groups=df["Source"]))

temp = df.iloc[temp_idx].reset_index()
gss2 = GroupShuffleSplit(n_splits=100, test_size=0.50, random_state=43)
val_rel, test_rel = next(gss2.split(temp, groups=temp["Source"]))

train = df.iloc[train_idx].copy()
val = temp.iloc[val_rel].copy()
test = temp.iloc[test_rel].copy()

# Confirm no source leakage across splits
train_sources = set(train["Source"])
val_sources = set(val["Source"])
test_sources = set(test["Source"])

print("\nNEW SOURCE-GROUPED SPLIT")
print(f"Train: {len(train)} rows, {train['Source'].nunique()} sources")
print(f"Validation: {len(val)} rows, {val['Source'].nunique()} sources")
print(f"Test: {len(test)} rows, {test['Source'].nunique()} sources")
print("Train/validation source overlap:", train_sources & val_sources)
print("Train/test source overlap:", train_sources & test_sources)
print("Validation/test source overlap:", val_sources & test_sources)

print("\nGrouped split label means:")
for name, part in [("Train", train), ("Validation", val), ("Test", test)]:
    print(name, "mean LMP =", round(part["LMP score"].mean(), 3),
          "| languages =", part["Language"].value_counts().to_dict(),
          "| domains =", part["Domain"].value_counts().to_dict())


DATASET
Rows: 357
Unique sources: 13
Duplicate IDs: 0
Duplicate sentence pairs: 0

LMP distribution:
LMP score
4       4
5       5
6      17
7      34
8      71
9     102
10    124
Name: count, dtype: int64

Language × Domain:
Domain    rental  tos_privacy
Language                     
en            59          146
hi            71           81

Source counts:
Source
google_tos              63
yojanavani_rental       38
meta_tos                36
lawdepot_residential    34
paytm_tos               31
netflix_tos             31
phonepe_tos             29
whatsapp_tos            24
adhiwakta_rental        19
lawdepot_commercial     15
vidhikInfo_rental       14
tata_tos                13
pandadoc_rental         10
Name: count, dtype: int64

NEW SOURCE-GROUPED SPLIT
Train: 272 rows, 9 sources
Validation: 43 rows, 2 sources
Test: 42 rows, 2 sources
Train/validation source overlap: set()
Train/test source overlap: set()
Validation/test source overlap: set()

Grouped split label means:
Train 

In [ ]:

# STEP 4: Search for a source-grouped split with better coverage
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("/content/lmp_final - lmp_semantic_review(1).csv")

# Each source must belong to exactly one split.
# Search many candidates and prefer reasonable sizes and coverage.
best = None
rng_seeds = range(100, 1100)

for seed in rng_seeds:
    outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    trainval_idx, test_idx = next(outer.split(df, groups=df["Source"]))

    trainval = df.iloc[trainval_idx]
    test = df.iloc[test_idx]

    inner = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed + 10000)
    train_rel, val_rel = next(
        inner.split(trainval, groups=trainval["Source"])
    )

    train = trainval.iloc[train_rel]
    val = trainval.iloc[val_rel]

    # Coverage checks
    splits = {"train": train, "val": val, "test": test}
    if any(len(s) == 0 for s in splits.values()):
        continue

    # Penalize splits missing a language or domain.
    penalty = 0
    for s in splits.values():
        if s["Language"].nunique() < 2:
            penalty += 100
        if s["Domain"].nunique() < 2:
            penalty += 100

    # Penalize sizes far from target proportions.
    n = len(df)
    size_penalty = (
        abs(len(train) / n - 0.64)
        + abs(len(val) / n - 0.16)
        + abs(len(test) / n - 0.20)
    ) * 100

    # Penalize language/domain distribution differences from full dataset.
    target_lang = df["Language"].value_counts(normalize=True)
    target_domain = df["Domain"].value_counts(normalize=True)
    distribution_penalty = 0

    for s in splits.values():
        lang = s["Language"].value_counts(normalize=True)
        domain = s["Domain"].value_counts(normalize=True)
        distribution_penalty += sum(
            abs(lang.get(k, 0) - target_lang.get(k, 0))
            for k in target_lang.index
        )
        distribution_penalty += sum(
            abs(domain.get(k, 0) - target_domain.get(k, 0))
            for k in target_domain.index
        )

    score = penalty + size_penalty + distribution_penalty * 10

    if best is None or score < best[0]:
        best = (score, train.copy(), val.copy(), test.copy())

if best is None:
    raise RuntimeError("No valid grouped split found.")

_, train, val, test = best

print("Best candidate grouped split:")
for name, s in [("Train", train), ("Validation", val), ("Test", test)]:
    print(f"\n{name}: {len(s)} rows; {s['Source'].nunique()} sources")
    print("Sources:", sorted(s["Source"].unique()))
    print("Language counts:", s["Language"].value_counts().to_dict())
    print("Domain counts:", s["Domain"].value_counts().to_dict())

print("\nSource overlaps:")
print("Train ∩ Val:", set(train.Source) & set(val.Source))
print("Train ∩ Test:", set(train.Source) & set(test.Source))
print("Val ∩ Test:", set(val.Source) & set(test.Source))


Best candidate grouped split:

Train: 228 rows; 8 sources
Sources: ['google_tos', 'lawdepot_commercial', 'lawdepot_residential', 'meta_tos', 'phonepe_tos', 'tata_tos', 'vidhikInfo_rental', 'whatsapp_tos']
Language counts: {'en': 133, 'hi': 95}
Domain counts: {'tos_privacy': 165, 'rental': 63}

Validation: 50 rows; 2 sources
Sources: ['adhiwakta_rental', 'paytm_tos']
Language counts: {'en': 31, 'hi': 19}
Domain counts: {'tos_privacy': 31, 'rental': 19}

Test: 79 rows; 3 sources
Sources: ['netflix_tos', 'pandadoc_rental', 'yojanavani_rental']
Language counts: {'en': 41, 'hi': 38}
Domain counts: {'rental': 48, 'tos_privacy': 31}

Source overlaps:
Train ∩ Val: set()
Train ∩ Test: set()
Val ∩ Test: set()


In [ ]:
import json
from pathlib import Path

# Load the dataset
df = pd.read_csv("/content/lmp_final - lmp_semantic_review(1).csv")

# Use the exact source groups from your best candidate
train_sources = [
    "google_tos", "lawdepot_commercial", "lawdepot_residential",
    "meta_tos", "phonepe_tos", "tata_tos",
    "vidhikInfo_rental", "whatsapp_tos"
]

val_sources = ["adhiwakta_rental", "paytm_tos"]

test_sources = ["netflix_tos", "pandadoc_rental", "yojanavani_rental"]

# Reconstruct the split from the source assignments
train_df = df[df["Source"].isin(train_sources)].copy()
val_df = df[df["Source"].isin(val_sources)].copy()
test_df = df[df["Source"].isin(test_sources)].copy()

# Verify source coverage and row counts before saving
assert len(train_df) == 228, f"Expected 228 training rows, got {len(train_df)}"
assert len(val_df) == 50, f"Expected 50 validation rows, got {len(val_df)}"
assert len(test_df) == 79, f"Expected 79 test rows, got {len(test_df)}"

assert set(train_sources).isdisjoint(val_sources)
assert set(train_sources).isdisjoint(test_sources)
assert set(val_sources).isdisjoint(test_sources)

assert len(train_df) + len(val_df) + len(test_df) == len(df)

split_dir = Path("/content/judgebert_lmp_results")
split_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(split_dir / "train_source_grouped.csv", index=False)
val_df.to_csv(split_dir / "validation_source_grouped.csv", index=False)
test_df.to_csv(split_dir / "test_source_grouped.csv", index=False)

audit = {
    "method": "source-disjoint grouped split",
    "train_rows": len(train_df),
    "validation_rows": len(val_df),
    "test_rows": len(test_df),
    "train_sources": train_sources,
    "validation_sources": val_sources,
    "test_sources": test_sources,
}

with open(split_dir / "source_grouped_split_audit.json", "w") as f:
    json.dump(audit, f, indent=2)

print("Split saved successfully.")
print(f"Train: {len(train_df)} rows | {train_df['Source'].nunique()} sources")
print(f"Validation: {len(val_df)} rows | {val_df['Source'].nunique()} sources")
print(f"Test: {len(test_df)} rows | {test_df['Source'].nunique()} sources")

print("\nTest language counts:")
print(test_df["Language"].value_counts())

print("\nTest domain counts:")
print(test_df["Domain"].value_counts())

print("\nSaved files to:", split_dir)

Split saved successfully.
Train: 228 rows | 8 sources
Validation: 50 rows | 2 sources
Test: 79 rows | 3 sources

Test language counts:
Language
en    41
hi    38
Name: count, dtype: int64

Test domain counts:
Domain
rental         48
tos_privacy    31
Name: count, dtype: int64

Saved files to: /content/judgebert_lmp_results


In [ ]:
import pandas as pd
from pathlib import Path

split_dir = Path("/content/judgebert_lmp_results")

train_df = pd.read_csv(split_dir / "train_source_grouped.csv")
val_df = pd.read_csv(split_dir / "validation_source_grouped.csv")
test_df = pd.read_csv(split_dir / "test_source_grouped.csv")

splits = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

# 1. Check source leakage
source_sets = {
    name: set(frame["Source"].dropna().unique())
    for name, frame in splits.items()
}

print("SOURCE OVERLAP CHECK")
for a, b in [("train", "validation"), ("train", "test"),
             ("validation", "test")]:
    overlap = source_sets[a] & source_sets[b]
    print(f"{a} ∩ {b}: {overlap}")
    assert not overlap, f"Source leakage detected: {a} and {b}"

# 2. Check labels and distributions
print("\nLABEL AND SPLIT AUDIT")
for name, frame in splits.items():
    scores = pd.to_numeric(frame["LMP score"], errors="coerce")

    print(f"\n{name.upper()} — {len(frame)} rows")
    print("Languages:", frame["Language"].value_counts().to_dict())
    print("Domains:", frame["Domain"].value_counts().to_dict())
    print(
        f"LMP: min={scores.min()}, max={scores.max()}, "
        f"mean={scores.mean():.3f}, std={scores.std():.3f}, "
        f"unique_scores={scores.nunique()}"
    )

    assert scores.notna().all(), f"Missing/non-numeric LMP labels in {name}"
    assert scores.between(1, 10).all(), f"Labels outside 1–10 in {name}"

# 3. Check rows are neither missing nor duplicated across saved splits
combined = pd.concat(splits.values(), ignore_index=True)
assert len(combined) == 357, f"Expected 357 total rows, got {len(combined)}"
assert combined["ID"].nunique() == len(combined), "Duplicate IDs across splits"

print("\nPASS: no source overlap, valid labels, and all 357 rows accounted for.")

SOURCE OVERLAP CHECK
train ∩ validation: set()
train ∩ test: set()
validation ∩ test: set()

LABEL AND SPLIT AUDIT

TRAIN — 228 rows
Languages: {'en': 133, 'hi': 95}
Domains: {'tos_privacy': 165, 'rental': 63}
LMP: min=4, max=10, mean=8.592, std=1.384, unique_scores=7

VALIDATION — 50 rows
Languages: {'en': 31, 'hi': 19}
Domains: {'tos_privacy': 31, 'rental': 19}
LMP: min=6, max=10, mean=8.880, std=1.154, unique_scores=5

TEST — 79 rows
Languages: {'en': 41, 'hi': 38}
Domains: {'rental': 48, 'tos_privacy': 31}
LMP: min=4, max=10, mean=8.911, std=1.263, unique_scores=6

PASS: no source overlap, valid labels, and all 357 rows accounted for.


In [ ]:
import importlib.util
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

packages = ["sklearn", "sentence_transformers", "scipy"]
for package in packages:
    print(f"{package}:",
          "installed" if importlib.util.find_spec(package) else "missing")

PyTorch: 2.11.0+cu130
CUDA available: True
sklearn: installed
sentence_transformers: installed
scipy: installed


In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

# Load the frozen split
split_dir = Path("/content/judgebert_lmp_results")
train_df = pd.read_csv(split_dir / "train_source_grouped.csv")
val_df = pd.read_csv(split_dir / "validation_source_grouped.csv")

TEXT_COL_1 = "Original"
TEXT_COL_2 = "Simplified"
TARGET_COL = "LMP score"

# Fit TF-IDF vocabulary on training text only
train_texts = pd.concat(
    [train_df[TEXT_COL_1], train_df[TEXT_COL_2]]
).fillna("").astype(str)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    strip_accents=None,
    max_features=50000
)
vectorizer.fit(train_texts)

def tfidf_similarity(frame):
    original = frame[TEXT_COL_1].fillna("").astype(str).tolist()
    simplified = frame[TEXT_COL_2].fillna("").astype(str).tolist()

    original_vec = vectorizer.transform(original)
    simplified_vec = vectorizer.transform(simplified)

    return cosine_similarity(original_vec, simplified_vec).diagonal()

# Similarities
train_sim = tfidf_similarity(train_df).reshape(-1, 1)
val_sim = tfidf_similarity(val_df).reshape(-1, 1)

y_train = pd.to_numeric(train_df[TARGET_COL]).to_numpy()
y_val = pd.to_numeric(val_df[TARGET_COL]).to_numpy()

# Fit calibration only on training data
calibrator = LinearRegression()
calibrator.fit(train_sim, y_train)

val_pred = np.clip(calibrator.predict(val_sim), 1, 10)

def report_metrics(y_true, y_pred, split_name):
    print(f"\n{split_name}")
    print(f"Pearson:  {pearsonr(y_true, y_pred).statistic:.4f}")
    print(f"Spearman: {spearmanr(y_true, y_pred).statistic:.4f}")
    print(f"RMSE:     {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
    print(f"MAE:      {mean_absolute_error(y_true, y_pred):.4f}")

report_metrics(y_val, val_pred, "TF-IDF — Validation")

print("\nCalibration slope:", round(float(calibrator.coef_[0]), 4))
print("Calibration intercept:", round(float(calibrator.intercept_), 4))
print("Validation prediction range:",
      round(float(val_pred.min()), 3),
      "to",
      round(float(val_pred.max()), 3))


TF-IDF — Validation
Pearson:  0.7027
Spearman: 0.7877
RMSE:     0.8862
MAE:      0.6379

Calibration slope: 4.7197
Calibration intercept: 5.4834
Validation prediction range: 5.483 to 10.0


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

split_dir = Path("/content/judgebert_lmp_results")
train_df = pd.read_csv(split_dir / "train_source_grouped.csv")
val_df = pd.read_csv(split_dir / "validation_source_grouped.csv")

# Multilingual sentence embedding model
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(model_name, device="cuda")

def embedding_similarity(frame):
    originals = frame["Original"].fillna("").astype(str).tolist()
    simplified = frame["Simplified"].fillna("").astype(str).tolist()

    original_embeddings = embedder.encode(
        originals,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    simplified_embeddings = embedder.encode(
        simplified,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return np.sum(original_embeddings * simplified_embeddings, axis=1)

train_sim = embedding_similarity(train_df).reshape(-1, 1)
val_sim = embedding_similarity(val_df).reshape(-1, 1)

y_train = pd.to_numeric(train_df["LMP score"]).to_numpy()
y_val = pd.to_numeric(val_df["LMP score"]).to_numpy()

# Fit calibration on training data only
calibrator = LinearRegression()
calibrator.fit(train_sim, y_train)

val_pred = np.clip(calibrator.predict(val_sim), 1, 10)

print("\nMultilingual MiniLM — Validation")
print(f"Pearson:  {pearsonr(y_val, val_pred).statistic:.4f}")
print(f"Spearman: {spearmanr(y_val, val_pred).statistic:.4f}")
print(f"RMSE:     {np.sqrt(mean_squared_error(y_val, val_pred)):.4f}")
print(f"MAE:      {mean_absolute_error(y_val, val_pred):.4f}")
print("Calibration slope:", round(float(calibrator.coef_[0]), 4))
print("Calibration intercept:", round(float(calibrator.intercept_), 4))
print("Prediction range:", round(float(val_pred.min()), 3),
      "to", round(float(val_pred.max()), 3))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (8 outputs left out)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Multilingual MiniLM — Validation
Pearson:  0.6375
Spearman: 0.6420
RMSE:     0.9462
MAE:      0.7437
Calibration slope: 10.0528
Calibration intercept: -0.7796
Prediction range: 6.161 to 9.273


In [ ]:
from pathlib import Path

# Check common locations for existing model checkpoints
search_roots = [
    Path("/content"),
    Path("/content/judgebert_lmp_final"),
    Path("/content/judgebert_lmp_results"),
]

print("=== Existing XLM-R files and checkpoints ===")

for root in search_roots:
    if not root.exists():
        print(f"\nNot found: {root}")
        continue

    print(f"\nSearching: {root}")
    matches = []

    for path in root.rglob("*"):
        if any(part in {".cache", "__pycache__"} for part in path.parts):
            continue

        if path.is_dir() and (
            path.name.startswith("checkpoint-")
            or path.name in {"judgebert_lmp_final", "xlm-roberta-base"}
        ):
            matches.append(path)

    if matches:
        for path in matches[:30]:
            print(path)
    else:
        print("No matching checkpoint directories found.")

print("\n=== Current split files ===")
for name in [
    "train_source_grouped.csv",
    "validation_source_grouped.csv",
    "test_source_grouped.csv",
]:
    path = Path("/content/judgebert_lmp_results") / name
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

=== Existing XLM-R files and checkpoints ===

Searching: /content
No matching checkpoint directories found.

Not found: /content/judgebert_lmp_final

Searching: /content/judgebert_lmp_results
No matching checkpoint directories found.

=== Current split files ===
train_source_grouped.csv: OK
validation_source_grouped.csv: OK
test_source_grouped.csv: OK


In [ ]:

# ============================================================
# JUDGEBERT-INSPIRED XLM-R REGRESSION
# Source-disjoint training + validation
# Dataset: 228 train / 50 validation / 79 held-out test
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

# ----------------------------
# 1. Reproducibility
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU unavailable; training will be slower.")

# ----------------------------
# 2. Load frozen data splits
# ----------------------------
split_dir = Path("/content/judgebert_lmp_results")

train_path = split_dir / "train_source_grouped.csv"
val_path = split_dir / "validation_source_grouped.csv"

assert train_path.exists(), f"Missing {train_path}"
assert val_path.exists(), f"Missing {val_path}"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)

assert len(train_df) == 228, f"Expected 228 train rows, got {len(train_df)}"
assert len(val_df) == 50, f"Expected 50 validation rows, got {len(val_df)}"
assert set(train_df["Source"]).isdisjoint(set(val_df["Source"])), \
    "Source leakage detected between training and validation!"

for frame in (train_df, val_df):
    frame["LMP score"] = pd.to_numeric(
        frame["LMP score"], errors="raise"
    ).astype(np.float32)

    assert frame["LMP score"].between(1, 10).all()

print(f"Training rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")
print("Train/validation sources are disjoint.")

# ----------------------------
# 3. Load XLM-RoBERTa
# ----------------------------
CHECKPOINT = "xlm-roberta-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=1,
    problem_type="regression",
)

# Newly initialized regression-head weights are expected here.

# ----------------------------
# 4. Tokenization and FLOAT labels
# ----------------------------
def prepare_dataset(frame):
    dataset_df = frame[
        ["Original", "Simplified", "LMP score"]
    ].rename(columns={"LMP score": "labels"}).reset_index(drop=True)

    dataset = Dataset.from_pandas(dataset_df)

    def tokenize_batch(batch):
        return tokenizer(
            batch["Original"],
            batch["Simplified"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    dataset = dataset.map(tokenize_batch, batched=True)

    # Critical regression fix: labels must be floating-point.
    dataset = dataset.map(
        lambda batch: {
            "labels": [float(x) for x in batch["labels"]]
        },
        batched=True,
    )

    dataset = dataset.remove_columns(["Original", "Simplified"])

    dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"],
    )

    return dataset

train_data = prepare_dataset(train_df)
val_data = prepare_dataset(val_df)

print("Training label dtype:", train_data[0]["labels"].dtype)
print("Validation label dtype:", val_data[0]["labels"].dtype)

assert train_data[0]["labels"].dtype == torch.float32
assert val_data[0]["labels"].dtype == torch.float32

# ----------------------------
# 5. Validation metrics
# ----------------------------
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.asarray(predictions).reshape(-1)
    labels = np.asarray(labels).reshape(-1)

    # Keep reported scores within the defined 1–10 scale.
    predictions = np.clip(predictions, 1.0, 10.0)

    pearson = (
        float(pearsonr(labels, predictions).statistic)
        if np.std(predictions) > 0 and np.std(labels) > 0
        else 0.0
    )

    spearman = (
        float(spearmanr(labels, predictions).statistic)
        if np.std(predictions) > 0 and np.std(labels) > 0
        else 0.0
    )

    return {
        "rmse": float(np.sqrt(mean_squared_error(labels, predictions))),
        "mae": float(mean_absolute_error(labels, predictions)),
        "pearson": pearson,
        "spearman": spearman,
    }

# ----------------------------
# 6. Training configuration
# ----------------------------
output_dir = split_dir / "xlmr_source_grouped"

training_args = TrainingArguments(
    output_dir=str(output_dir),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_rmse",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    logging_strategy="epoch",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    save_total_limit=2,
)

# ----------------------------
# 7. Trainer
# ----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# ----------------------------
# 8. Train
# ----------------------------
train_result = trainer.train()

print("\nTraining finished.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation RMSE:", trainer.state.best_metric)

# ----------------------------
# 9. Save best model and tokenizer
# ----------------------------
final_model_dir = split_dir / "xlmr_source_grouped_best"
final_model_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

# Save training metadata for reproducibility.
metadata = {
    "base_model": CHECKPOINT,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "train_rows": len(train_df),
    "validation_rows": len(val_df),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_rmse": float(trainer.state.best_metric),
}

import json

with open(final_model_dir / "training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\nBest model saved to:", final_model_dir)
print("Test set has NOT been evaluated.")


GPU: Tesla T4
Training rows: 228
Validation rows: 50
Train/validation sources are disjoint.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/228 [00:00<?, ? examples/s]

Map:   0%|          | 0/228 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

... (7 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training finished.
Best checkpoint: /content/judgebert_lmp_results/xlmr_source_grouped/checkpoint-145
Best validation RMSE: 1.293464081724485


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Best model saved to: /content/judgebert_lmp_results/xlmr_source_grouped_best
Test set has NOT been evaluated.


In [ ]:

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from datasets import Dataset
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

split_dir = Path("/content/judgebert_lmp_results")
model_dir = split_dir / "xlmr_source_grouped_best"

val_df = pd.read_csv(split_dir / "validation_source_grouped.csv")
val_df["LMP score"] = pd.to_numeric(val_df["LMP score"]).astype(np.float32)

tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))

def make_validation_dataset(frame):
    dataset = Dataset.from_pandas(
        frame[["Original", "Simplified"]].reset_index(drop=True)
    )

    dataset = dataset.map(
        lambda batch: tokenizer(
            batch["Original"],
            batch["Simplified"],
            truncation=True,
            max_length=256,
        ),
        batched=True,
    )

    dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask"],
    )
    return dataset

val_data = make_validation_dataset(val_df)

args = TrainingArguments(
    output_dir=str(split_dir / "xlmr_diagnostic"),
    per_device_eval_batch_size=8,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

raw_predictions = trainer.predict(val_data).predictions
predictions = np.asarray(raw_predictions).reshape(-1)
predictions = np.clip(predictions, 1, 10)

actual = val_df["LMP score"].to_numpy()

print("Validation prediction diagnostics")
print(f"Actual mean/std:       {actual.mean():.3f} / {actual.std():.3f}")
print(f"Predicted mean/std:    {predictions.mean():.3f} / {predictions.std():.3f}")
print(f"Prediction min/max:    {predictions.min():.3f} / {predictions.max():.3f}")
print(f"RMSE:                  {np.sqrt(mean_squared_error(actual, predictions)):.4f}")
print(f"MAE:                   {mean_absolute_error(actual, predictions):.4f}")
print(f"Pearson:               {pearsonr(actual, predictions).statistic:.4f}")
print(f"Spearman:              {spearmanr(actual, predictions).statistic:.4f}")

diagnostics = val_df[["ID", "Language", "Domain", "Source", "Original", "Simplified", "LMP score"]].copy()
diagnostics["prediction"] = predictions
diagnostics["absolute_error"] = np.abs(actual - predictions)

display(diagnostics.sort_values("absolute_error", ascending=False).head(10))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Validation prediction diagnostics
Actual mean/std:       8.880 / 1.143
Predicted mean/std:    8.297 / 0.066
Prediction min/max:    8.109 / 8.430
RMSE:                  1.2935
MAE:                   1.1378
Pearson:               -0.1538
Spearman:              -0.1463


,ID,Language,Domain,Source,Original,Simplified,LMP score,prediction,absolute_error
5,en_tos_054,en,tos_privacy,paytm_tos,Paytm may make changes to any Paytm Services o...,Paytm may change its services or terms at any ...,6.0,8.382812,2.382812
20,en_tos_069,en,tos_privacy,paytm_tos,"Pending resolution of any dispute, the User sh...","While a dispute is unresolved, the User must s...",6.0,8.296875,2.296875
40,hi_rental_010,hi,rental,adhiwakta_rental,"सामान्य टूट-फूट को छोड़कर, संपत्ति को उसी स्थि...","सामान्य टूट-फूट को छोड़कर, संपत्ति को उसी स्थि...",10.0,8.109375,1.890625
6,en_tos_055,en,tos_privacy,paytm_tos,After successfully registering for the SSOID S...,After successfully registering for the SSOID S...,10.0,8.203125,1.796875
18,en_tos_067,en,tos_privacy,paytm_tos,The User may not assign or transfer any rights...,The User may not assign or transfer any rights...,10.0,8.210938,1.789062
38,hi_rental_008,hi,rental,adhiwakta_rental,बिना अनुमति संरचनात्मक परिवर्तन नहीं होगा।,बिना अनुमति कोई संरचनात्मक परिवर्तन नहीं होगा।,10.0,8.226562,1.773438
43,hi_rental_013,hi,rental,adhiwakta_rental,संपत्ति को हुई क्षति (सामान्य टूट-फूट को छोड़क...,संपत्ति को हुई क्षति (सामान्य टूट-फूट को छोड़क...,10.0,8.226562,1.773438
49,hi_rental_019,hi,rental,adhiwakta_rental,शर्तों का उल्लंघन होने पर।,शर्तों का उल्लंघन होने पर।,10.0,8.242188,1.757812
46,hi_rental_016,hi,rental,adhiwakta_rental,यह अनुबंध उत्तर प्रदेश के किराया नियंत्रण अधिन...,यह अनुबंध उत्तर प्रदेश के किराया नियंत्रण अधिन...,10.0,8.265625,1.734375
44,hi_rental_014,hi,rental,adhiwakta_rental,किसी भी विवाद की स्थिति में दोनों पक्ष सबसे पह...,किसी भी विवाद की स्थिति में दोनों पक्ष सबसे पह...,10.0,8.273438,1.726562


In [ ]:
import os

BASE = "/content/judgebert_lmp_results"

for path in [
    f"{BASE}/train_source_grouped.csv",
    f"{BASE}/validation_source_grouped.csv",
    f"{BASE}/test_source_grouped.csv",
    f"{BASE}/xlmr_source_grouped_best",
]:
    print(f"{'OK' if os.path.exists(path) else 'MISSING'}: {path}")

OK: /content/judgebert_lmp_results/train_source_grouped.csv
OK: /content/judgebert_lmp_results/validation_source_grouped.csv
OK: /content/judgebert_lmp_results/test_source_grouped.csv
OK: /content/judgebert_lmp_results/xlmr_source_grouped_best


In [ ]:
import pandas as pd

train_df = pd.read_csv(f"{BASE}/train_source_grouped.csv")
val_df   = pd.read_csv(f"{BASE}/validation_source_grouped.csv")
test_df  = pd.read_csv(f"{BASE}/test_source_grouped.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTest languages:")
print(test_df["Language"].value_counts())

print("\nTest domains:")
print(test_df["Domain"].value_counts())

print("\nTest sources:")
print(test_df["Source"].value_counts())

assert set(train_df["Source"]).isdisjoint(set(val_df["Source"]))
assert set(train_df["Source"]).isdisjoint(set(test_df["Source"]))
assert set(val_df["Source"]).isdisjoint(set(test_df["Source"]))

print("\nSource-disjoint split verified.")

Train: (228, 8)
Validation: (50, 8)
Test: (79, 8)

Test languages:
Language
en    41
hi    38
Name: count, dtype: int64

Test domains:
Domain
rental         48
tos_privacy    31
Name: count, dtype: int64

Test sources:
Source
yojanavani_rental    38
netflix_tos          31
pandadoc_rental      10
Name: count, dtype: int64

Source-disjoint split verified.


In [ ]:
!pip -q install transformers sentence-transformers scikit-learn scipy pandas numpy

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE = "/content/judgebert_lmp_results"
MODEL_PATH = f"{BASE}/xlmr_source_grouped_best"
OUT_DIR = f"{BASE}/test_evaluation"
os.makedirs(OUT_DIR, exist_ok=True)

# Load frozen split
test_df = pd.read_csv(f"{BASE}/test_source_grouped.csv").reset_index(drop=True)

# Validate columns
required = {"Original", "Simplified", "LMP score", "Language", "Domain", "Source"}
missing = required - set(test_df.columns)
assert not missing, f"Missing columns: {missing}"

# Load saved model and tokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

# Predict without using test labels during inference
predictions = []
batch_size = 8

with torch.no_grad():
    for start in range(0, len(test_df), batch_size):
        batch = test_df.iloc[start:start + batch_size]

        texts_a = batch["Original"].fillna("").astype(str).tolist()
        texts_b = batch["Simplified"].fillna("").astype(str).tolist()

        encoded = tokenizer(
            texts_a,
            texts_b,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        logits = model(**encoded).logits.reshape(-1)
        predictions.extend(logits.detach().cpu().numpy().tolist())

test_df["XLMR_prediction_raw"] = predictions

# Keep the original model outputs for honest reporting.
# Clip only a separate column to the required 1–10 decision scale.
test_df["XLMR_prediction_1_10"] = test_df["XLMR_prediction_raw"].clip(1, 10)

y_true = test_df["LMP score"].astype(float).to_numpy()
y_pred = test_df["XLMR_prediction_1_10"].to_numpy()

def safe_corr(func, a, b):
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(func(a, b).statistic)

metrics = {
    "model": "XLM-R cross-encoder regression",
    "split": "source-disjoint test",
    "n": int(len(test_df)),
    "pearson": safe_corr(pearsonr, y_true, y_pred),
    "spearman": safe_corr(spearmanr, y_true, y_pred),
    "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    "mae": float(mean_absolute_error(y_true, y_pred)),
    "prediction_mean": float(np.mean(y_pred)),
    "prediction_std": float(np.std(y_pred)),
    "prediction_min": float(np.min(y_pred)),
    "prediction_max": float(np.max(y_pred)),
}

with open(f"{OUT_DIR}/xlmr_test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

test_df.to_csv(f"{OUT_DIR}/xlmr_test_predictions.csv", index=False)

print("\nXLM-R TEST RESULTS")
for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nSaved files:")
print(f"{OUT_DIR}/xlmr_test_metrics.json")
print(f"{OUT_DIR}/xlmr_test_predictions.csv")

Device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


XLM-R TEST RESULTS
model: XLM-R cross-encoder regression
split: source-disjoint test
n: 79
pearson: -0.18832237412375877
spearman: -0.17529290425805208
rmse: 1.422767338545036
mae: 1.2185144665875012
prediction_mean: 8.267174431040317
prediction_std: 0.064799074360138
prediction_min: 8.10445499420166
prediction_max: 8.39054012298584

Saved files:
/content/judgebert_lmp_results/test_evaluation/xlmr_test_metrics.json
/content/judgebert_lmp_results/test_evaluation/xlmr_test_predictions.csv


In [ ]:
import os
import json
import pandas as pd
import numpy as np

BASE = "/content/judgebert_lmp_results"
OUT = f"{BASE}/test_evaluation"

# Load saved XLM-R test results
with open(f"{OUT}/xlmr_test_metrics.json") as f:
    xlmr_metrics = json.load(f)

print("XLM-R held-out test metrics")
print(pd.Series(xlmr_metrics).to_string())

# Check prediction collapse
preds = pd.read_csv(f"{OUT}/xlmr_test_predictions.csv")
print("\nPrediction distribution:")
print(preds["XLMR_prediction_1_10"].describe().to_string())

# Prepare a project-results summary for the baselines and XLM-R.
# Baseline test metrics will be added after evaluating their predictions.
summary = pd.DataFrame([
    {
        "Model": "XLM-R cross-encoder",
        "Split": "Source-disjoint test",
        "N": xlmr_metrics["n"],
        "Pearson": xlmr_metrics["pearson"],
        "Spearman": xlmr_metrics["spearman"],
        "RMSE": xlmr_metrics["rmse"],
        "MAE": xlmr_metrics["mae"],
        "Prediction SD": xlmr_metrics["prediction_std"],
    }
])

summary.to_csv(f"{OUT}/model_comparison_test.csv", index=False)
print("\nSaved:", f"{OUT}/model_comparison_test.csv")

XLM-R held-out test metrics
model              XLM-R cross-encoder regression
split                        source-disjoint test
n                                              79
pearson                                 -0.188322
spearman                                -0.175293
rmse                                     1.422767
mae                                      1.218514
prediction_mean                          8.267174
prediction_std                           0.064799
prediction_min                           8.104455
prediction_max                            8.39054

Prediction distribution:
count    79.000000
mean      8.267174
std       0.065213
min       8.104455
25%       8.225651
50%       8.268749
75%       8.312114
max       8.390540

Saved: /content/judgebert_lmp_results/test_evaluation/model_comparison_test.csv


In [ ]:
!pip -q install sentence-transformers scikit-learn scipy pandas

import os, json
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sentence_transformers import SentenceTransformer

BASE = "/content/judgebert_lmp_results"
OUT = f"{BASE}/test_evaluation"
os.makedirs(OUT, exist_ok=True)

train_df = pd.read_csv(f"{BASE}/train_source_grouped.csv")
test_df  = pd.read_csv(f"{BASE}/test_source_grouped.csv")

TEXT_A, TEXT_B, TARGET = "Original", "Simplified", "LMP score"

# Basic checks
assert set(train_df["Source"]).isdisjoint(set(test_df["Source"]))
assert train_df[TARGET].notna().all() and test_df[TARGET].notna().all()

y_train = train_df[TARGET].astype(float).to_numpy()
y_test  = test_df[TARGET].astype(float).to_numpy()

def metrics(y, p):
    return {
        "pearson": float(pearsonr(y, p).statistic) if np.std(p) > 0 else float("nan"),
        "spearman": float(spearmanr(y, p).statistic) if np.std(p) > 0 else float("nan"),
        "rmse": float(np.sqrt(mean_squared_error(y, p))),
        "mae": float(mean_absolute_error(y, p)),
        "prediction_mean": float(np.mean(p)),
        "prediction_std": float(np.std(p)),
        "n": int(len(y)),
    }

# --------------------------
# 1) TF-IDF cosine baseline
# --------------------------
all_train_texts = (
    train_df[TEXT_A].fillna("").astype(str).tolist()
    + train_df[TEXT_B].fillna("").astype(str).tolist()
)

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=100_000,
    sublinear_tf=True,
)
tfidf.fit(all_train_texts)

A_train = tfidf.transform(train_df[TEXT_A].fillna("").astype(str))
B_train = tfidf.transform(train_df[TEXT_B].fillna("").astype(str))
A_test  = tfidf.transform(test_df[TEXT_A].fillna("").astype(str))
B_test  = tfidf.transform(test_df[TEXT_B].fillna("").astype(str))

def cosine_rows(A, B):
    from sklearn.preprocessing import normalize
    A = normalize(A)
    B = normalize(B)
    return np.asarray(A.multiply(B).sum(axis=1)).ravel()

sim_train = cosine_rows(A_train, B_train).reshape(-1, 1)
sim_test  = cosine_rows(A_test, B_test).reshape(-1, 1)

tfidf_calibrator = LinearRegression()
tfidf_calibrator.fit(sim_train, y_train)
tfidf_pred = np.clip(tfidf_calibrator.predict(sim_test), 1, 10)

# --------------------------
# 2) Multilingual MiniLM
# --------------------------
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

train_a = train_df[TEXT_A].fillna("").astype(str).tolist()
train_b = train_df[TEXT_B].fillna("").astype(str).tolist()
test_a  = test_df[TEXT_A].fillna("").astype(str).tolist()
test_b  = test_df[TEXT_B].fillna("").astype(str).tolist()

emb_train_a = embedder.encode(train_a, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
emb_train_b = embedder.encode(train_b, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
emb_test_a  = embedder.encode(test_a, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
emb_test_b  = embedder.encode(test_b, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)

sim_train_mini = np.sum(emb_train_a * emb_train_b, axis=1).reshape(-1, 1)
sim_test_mini  = np.sum(emb_test_a * emb_test_b, axis=1).reshape(-1, 1)

mini_calibrator = LinearRegression()
mini_calibrator.fit(sim_train_mini, y_train)
mini_pred = np.clip(mini_calibrator.predict(sim_test_mini), 1, 10)

# --------------------------
# Save predictions + metrics
# --------------------------
results = []
prediction_df = test_df[["ID", "Source", "Language", "Domain", TARGET]].copy()

for name, pred in [
    ("TF-IDF cosine + linear calibration", tfidf_pred),
    ("Multilingual MiniLM cosine + linear calibration", mini_pred),
]:
    m = metrics(y_test, pred)
    m["model"] = name
    results.append(m)
    prediction_df[name] = pred

prediction_df.to_csv(f"{OUT}/baseline_test_predictions.csv", index=False)

xlmr_metrics_path = f"{OUT}/xlmr_test_metrics.json"
with open(xlmr_metrics_path) as f:
    xlmr_metrics = json.load(f)

xlmr_row = {
    "model": "XLM-R cross-encoder regression",
    "n": xlmr_metrics["n"],
    "pearson": xlmr_metrics["pearson"],
    "spearman": xlmr_metrics["spearman"],
    "rmse": xlmr_metrics["rmse"],
    "mae": xlmr_metrics["mae"],
    "prediction_mean": xlmr_metrics["prediction_mean"],
    "prediction_std": xlmr_metrics["prediction_std"],
}

results.append(xlmr_row)
results_df = pd.DataFrame(results)
results_df = results_df[
    ["model", "n", "pearson", "spearman", "rmse", "mae", "prediction_mean", "prediction_std"]
]
results_df.to_csv(f"{OUT}/model_comparison_test.csv", index=False)

print("\nHELD-OUT TEST COMPARISON")
display(results_df.round(4))
print("\nSaved predictions:", f"{OUT}/baseline_test_predictions.csv")
print("Saved comparison:", f"{OUT}/model_comparison_test.csv")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]


HELD-OUT TEST COMPARISON


,model,n,pearson,spearman,rmse,mae,prediction_mean,prediction_std
0,TF-IDF cosine + linear calibration,79,0.6143,0.6766,1.0315,0.6830,9.2007,0.7612
1,Multilingual MiniLM cosine + linear calibration,79,0.4502,0.5648,1.1530,0.9236,8.6619,0.6743
2,XLM-R cross-encoder regression,79,-0.1883,-0.1753,1.4228,1.2185,8.2672,0.0648



Saved predictions: /content/judgebert_lmp_results/test_evaluation/baseline_test_predictions.csv
Saved comparison: /content/judgebert_lmp_results/test_evaluation/model_comparison_test.csv


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

BASE = "/content/judgebert_lmp_results"
OUT = f"{BASE}/test_evaluation"

df = pd.read_csv(f"{OUT}/baseline_test_predictions.csv")
xlmr = pd.read_csv(f"{OUT}/xlmr_test_predictions.csv")

# Join XLM-R predictions by ID
df = df.merge(
    xlmr[["ID", "XLMR_prediction_1_10"]],
    on="ID",
    how="left",
    validate="one_to_one"
)

target = "LMP score"
models = {
    "TF-IDF": "TF-IDF cosine + linear calibration",
    "MiniLM": "Multilingual MiniLM cosine + linear calibration",
    "XLM-R": "XLMR_prediction_1_10"
}

def calculate_metrics(group, pred_col):
    y = group[target].astype(float).to_numpy()
    p = group[pred_col].astype(float).to_numpy()

    return {
        "n": len(group),
        "Pearson": pearsonr(y, p).statistic if len(group) >= 2 and np.std(y) > 0 and np.std(p) > 0 else np.nan,
        "Spearman": spearmanr(y, p).statistic if len(group) >= 2 and np.std(y) > 0 and np.std(p) > 0 else np.nan,
        "RMSE": np.sqrt(mean_squared_error(y, p)),
        "MAE": mean_absolute_error(y, p),
    }

rows = []
for dimension in ["Language", "Domain"]:
    for group_name, group in df.groupby(dimension):
        for model_name, pred_col in models.items():
            result = calculate_metrics(group, pred_col)
            rows.append({
                "Dimension": dimension,
                "Group": group_name,
                "Model": model_name,
                **result
            })

breakdown = pd.DataFrame(rows)
breakdown.to_csv(f"{OUT}/language_domain_breakdown.csv", index=False)

display(breakdown.round(4))
print("Saved:", f"{OUT}/language_domain_breakdown.csv")

,Dimension,Group,Model,n,Pearson,Spearman,RMSE,MAE
0,Language,en,TF-IDF,41,0.5928,0.6277,1.1968,0.8670
1,Language,en,MiniLM,41,0.5340,0.6892,1.2253,0.9166
2,Language,en,XLM-R,41,-0.3521,-0.2857,1.4413,1.1621
3,Language,hi,TF-IDF,38,0.6253,0.7366,0.8165,0.4844
4,Language,hi,MiniLM,38,0.6610,0.6039,1.0696,0.9311
5,Language,hi,XLM-R,38,0.1044,-0.0225,1.4025,1.2794
6,Domain,rental,TF-IDF,48,0.6169,0.7387,1.1109,0.6706
7,Domain,rental,MiniLM,48,0.5195,0.6224,1.2422,0.9952
8,Domain,rental,XLM-R,48,-0.1615,-0.1480,1.5310,1.3305
9,Domain,tos_privacy,TF-IDF,31,0.6080,0.5866,0.8949,0.7020


Saved: /content/judgebert_lmp_results/test_evaluation/language_domain_breakdown.csv


In [ ]:
import os
import numpy as np
import pandas as pd

BASE = "/content/judgebert_lmp_results"
OUT = f"{BASE}/test_evaluation"

pred = pd.read_csv(f"{OUT}/baseline_test_predictions.csv")
xlmr = pd.read_csv(f"{OUT}/xlmr_test_predictions.csv")

# Merge XLM-R predictions with the other model outputs
df = pred.merge(
    xlmr[["ID", "XLMR_prediction_1_10"]],
    on="ID",
    how="left",
    validate="one_to_one"
)

# Recover the original text fields and human assessment from the frozen test split
test = pd.read_csv(f"{BASE}/test_source_grouped.csv")

needed = ["ID", "Original", "Simplified", "Human_Assessment"]
available = [c for c in needed if c in test.columns]
df = df.merge(
    test[available],
    on="ID",
    how="left",
    validate="one_to_one"
)

target = "LMP score"
tfidf_col = "TF-IDF cosine + linear calibration"
mini_col = "Multilingual MiniLM cosine + linear calibration"
xlmr_col = "XLMR_prediction_1_10"

# Error and disagreement columns
df["TFIDF_absolute_error"] = (df[target] - df[tfidf_col]).abs()
df["MiniLM_absolute_error"] = (df[target] - df[mini_col]).abs()
df["XLMR_absolute_error"] = (df[target] - df[xlmr_col]).abs()

df["TFIDF_minus_human"] = df[tfidf_col] - df[target]
df["MiniLM_minus_human"] = df[mini_col] - df[target]
df["XLMR_minus_human"] = df[xlmr_col] - df[target]

# Choose 15 informative cases: high-error examples across models.
# This is an error-analysis sample, not a manually verified legal gold set.
df["mean_absolute_error"] = df[
    ["TFIDF_absolute_error", "MiniLM_absolute_error", "XLMR_absolute_error"]
].mean(axis=1)

hard_cases = df.sort_values("mean_absolute_error", ascending=False).head(15)

# Save full predictions and selected error cases
df.to_csv(f"{OUT}/all_models_test_error_analysis.csv", index=False)
hard_cases.to_csv(f"{OUT}/15_high_error_cases.csv", index=False)

show_cols = [
    "ID", "Language", "Domain", target,
    tfidf_col, "TFIDF_absolute_error",
    mini_col, "MiniLM_absolute_error",
    xlmr_col, "XLMR_absolute_error"
]
show_cols = [c for c in show_cols if c in hard_cases.columns]

print("15 high-error cases (inspect these; do not call them verified legal violations):")
display(hard_cases[show_cols].round(3))

print("\nSaved:")
print(f"{OUT}/all_models_test_error_analysis.csv")
print(f"{OUT}/15_high_error_cases.csv")

15 high-error cases (inspect these; do not call them verified legal violations):


,ID,Language,Domain,LMP score,TF-IDF cosine + linear calibration,TFIDF_absolute_error,Multilingual MiniLM cosine + linear calibration,MiniLM_absolute_error,XLMR_prediction_1_10,XLMR_absolute_error
38,en_rental_057,en,rental,4,7.800,3.800,8.243,4.243,8.391,4.391
11,en_tos_091,en,tos_privacy,6,8.231,2.231,8.989,2.989,8.300,2.300
31,en_rental_050,en,rental,6,8.818,2.818,8.373,2.373,8.322,2.322
33,en_rental_052,en,rental,6,8.662,2.662,8.090,2.090,8.283,2.283
54,hi_rental_047,hi,rental,6,8.215,2.215,4.924,1.076,8.152,2.152
66,hi_rental_059,hi,rental,7,9.825,2.825,7.432,0.432,8.202,1.202
22,en_tos_102,en,tos_privacy,10,8.385,1.615,9.020,0.980,8.254,1.746
12,en_tos_092,en,tos_privacy,7,8.428,1.428,8.639,1.639,8.257,1.257
7,en_tos_087,en,tos_privacy,10,8.790,1.210,8.739,1.261,8.300,1.700
51,hi_rental_044,hi,rental,10,9.311,0.689,8.406,1.594,8.199,1.801



Saved:
/content/judgebert_lmp_results/test_evaluation/all_models_test_error_analysis.csv
/content/judgebert_lmp_results/test_evaluation/15_high_error_cases.csv


In [ ]:
import pandas as pd

BASE = "/content/judgebert_lmp_results"
OUT = f"{BASE}/test_evaluation"

cases = pd.read_csv(f"{OUT}/15_high_error_cases.csv")

# Load full source-disjoint test data to retrieve original text
test = pd.read_csv(f"{BASE}/test_source_grouped.csv")

# Keep the original and simplified clauses, human score, and metadata
text_cols = [
    "ID", "Original", "Simplified", "LMP score",
    "Language", "Domain", "Source"
]
text_cols = [c for c in text_cols if c in test.columns]

review = cases[["ID"]].merge(
    test[text_cols],
    on="ID",
    how="left",
    validate="one_to_one"
)

# Show all 15 cases with full text
for _, row in review.iterrows():
    print("\n" + "=" * 100)
    print(
        f"ID: {row['ID']} | Language: {row['Language']} | "
        f"Domain: {row['Domain']} | Human LMP: {row['LMP score']}"
    )
    print("\nORIGINAL:\n", row["Original"])
    print("\nSIMPLIFIED:\n", row["Simplified"])


ID: en_rental_057 | Language: en | Domain: rental | Human LMP: 4

ORIGINAL:
 Within 30 days of the termination of this lease (and Tenant vacating the Property), Landlord will provide the Tenants with an itemized statement showing any retention of any portions of the Security Deposit amount for the cost of repair or cleaning above and beyond the ordinary wear and tear on the Property or any other sums due to Landlord under this Residential Lease Agreement.

SIMPLIFIED:
 Within 30 days after the lease ends and Tenant moves out, Landlord will give Tenants a list showing any part of the Security Deposit kept for repairs or cleaning beyond normal wear and tear or other amounts owed under this agreement.

ID: en_tos_091 | Language: en | Domain: tos_privacy | Human LMP: 6

ORIGINAL:
 For the purpose of determining the taxes that will be charged over your subscription fee, we will use the IP address that is used when you sign up to the Netflix service.

SIMPLIFIED:
 For tax purposes, we will 

In [ ]:

# STEP 1 — Diagnose the existing LegalDelta-BERT starting point
import os
import json
import pandas as pd
import torch

BASE = "/content/judgebert_lmp_results"
MODEL_DIR = f"{BASE}/xlmr_source_grouped_best"

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("\n--- Dataset splits ---")

for name in ["train_source_grouped.csv",
             "validation_source_grouped.csv",
             "test_source_grouped.csv"]:
    path = f"{BASE}/{name}"
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"{name}: {len(df)} rows")
        print("  Columns:", list(df.columns))
        print("  Score distribution:",
              df["LMP score"].value_counts().sort_index().to_dict())
    else:
        print("MISSING:", path)

print("\n--- Previous model metadata ---")
metadata_path = f"{MODEL_DIR}/training_metadata.json"

if os.path.exists(metadata_path):
    with open(metadata_path) as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("Training metadata not found:", metadata_path)

print("\n--- Saved checkpoint files ---")
if os.path.isdir(MODEL_DIR):
    print(sorted(os.listdir(MODEL_DIR)))
else:
    print("Model directory not found:", MODEL_DIR)


GPU: Tesla T4

--- Dataset splits ---
train_source_grouped.csv: 228 rows
  Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment']
  Score distribution: {4: 3, 5: 5, 6: 11, 7: 25, 8: 47, 9: 65, 10: 72}
validation_source_grouped.csv: 50 rows
  Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment']
  Score distribution: {6: 2, 7: 5, 8: 9, 9: 15, 10: 19}
test_source_grouped.csv: 79 rows
  Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment']
  Score distribution: {4: 1, 6: 4, 7: 4, 8: 15, 9: 22, 10: 33}

--- Previous model metadata ---
{
  "base_model": "xlm-roberta-base",
  "max_length": 256,
  "seed": 42,
  "train_rows": 228,
  "validation_rows": 50,
  "best_checkpoint": "/content/judgebert_lmp_results/xlmr_source_grouped/checkpoint-145",
  "best_validation_rmse": 1.293464081724485
}

--- Saved checkpoint files ---
['config.json', 'm

In [ ]:

# STEP 2 — Check prediction collapse and score learning
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

BASE = "/content/judgebert_lmp_results"
MODEL_DIR = f"{BASE}/xlmr_source_grouped_best"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

def predict_scores(df, batch_size=16):
    predictions = []
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start + batch_size]
        enc = tokenizer(
            batch["Original"].fillna("").astype(str).tolist(),
            batch["Simplified"].fillna("").astype(str).tolist(),
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            output = model(**enc).logits.squeeze(-1)
        predictions.extend(output.detach().float().cpu().numpy().tolist())
    return np.asarray(predictions)

for split_name in ["train", "validation"]:
    df = pd.read_csv(f"{BASE}/{split_name}_source_grouped.csv")
    y = df["LMP score"].astype(float).to_numpy()
    pred = predict_scores(df)

    print(f"\n--- {split_name.upper()} ---")
    print("Actual score:      mean =", round(y.mean(), 3),
          "| std =", round(y.std(), 3),
          "| min/max =", (y.min(), y.max()))
    print("Predicted score:   mean =", round(pred.mean(), 3),
          "| std =", round(pred.std(), 3),
          "| min/max =", (round(pred.min(), 3), round(pred.max(), 3)))
    print("MAE:", round(mean_absolute_error(y, pred), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(y, pred)), 4))

    if np.std(pred) > 1e-8 and np.std(y) > 1e-8:
        print("Pearson:", round(pearsonr(y, pred).statistic, 4))
        print("Spearman:", round(spearmanr(y, pred).statistic, 4))
    else:
        print("Correlation unavailable: predictions or labels are constant.")

    print("Predictions rounded:")
    print(pd.Series(np.round(pred, 1)).value_counts().sort_index().to_dict())

print("\nCheckpoint loaded successfully. No training or files changed.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- TRAIN ---
Actual score:      mean = 8.592 | std = 1.381 | min/max = (np.float64(4.0), np.float64(10.0))
Predicted score:   mean = 8.254 | std = 0.082 | min/max = (np.float64(8.038), np.float64(8.456))
MAE: 1.2052
RMSE: 1.4449
Pearson: -0.2593
Spearman: -0.1751
Predictions rounded:
{8.0: 1, 8.1: 21, 8.2: 87, 8.3: 92, 8.4: 25, 8.5: 2}

--- VALIDATION ---
Actual score:      mean = 8.88 | std = 1.143 | min/max = (np.float64(6.0), np.float64(10.0))
Predicted score:   mean = 8.298 | std = 0.066 | min/max = (np.float64(8.108), np.float64(8.426))
MAE: 1.1376
RMSE: 1.2934
Pearson: -0.1554
Spearman: -0.1504
Predictions rounded:
{8.1: 1, 8.2: 11, 8.3: 27, 8.4: 11}

Checkpoint loaded successfully. No training or files changed.


In [ ]:

import json
from transformers import AutoConfig

MODEL_DIR = "/content/judgebert_lmp_results/xlmr_source_grouped_best"
cfg = AutoConfig.from_pretrained(MODEL_DIR)

print("num_labels:", cfg.num_labels)
print("problem_type:", cfg.problem_type)
print("classifier_dropout:", getattr(cfg, "classifier_dropout", None))
print("hidden_dropout_prob:", getattr(cfg, "hidden_dropout_prob", None))

with open(f"{MODEL_DIR}/training_metadata.json") as f:
    print("\nMetadata:")
    print(json.dumps(json.load(f), indent=2))


num_labels: 1
problem_type: regression
classifier_dropout: None
hidden_dropout_prob: 0.1

Metadata:
{
  "base_model": "xlm-roberta-base",
  "max_length": 256,
  "seed": 42,
  "train_rows": 228,
  "validation_rows": 50,
  "best_checkpoint": "/content/judgebert_lmp_results/xlmr_source_grouped/checkpoint-145",
  "best_validation_rmse": 1.293464081724485
}


In [ ]:

# LegalDelta-BERT v2 — standardized regression + pairwise ranking
# Uses training + validation only. Does NOT access the test set.

!pip -q install -U transformers accelerate scikit-learn scipy

import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BASE = "/content/judgebert_lmp_results"
OUT = f"{BASE}/legaldelta_v2"
os.makedirs(OUT, exist_ok=True)

TRAIN = pd.read_csv(f"{BASE}/train_source_grouped.csv")
VAL = pd.read_csv(f"{BASE}/validation_source_grouped.csv")

MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 6
LR = 1e-5
RANK_WEIGHT = 0.25
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Standardize targets using training labels only.
y_mean = float(TRAIN["LMP score"].mean())
y_std = float(TRAIN["LMP score"].std())
assert y_std > 0

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PairDataset(Dataset):
    def __init__(self, df):
        self.original = df["Original"].fillna("").astype(str).tolist()
        self.simplified = df["Simplified"].fillna("").astype(str).tolist()
        self.y = (df["LMP score"].astype(float).to_numpy() - y_mean) / y_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        enc = tokenizer(
            self.original[i], self.simplified[i],
            max_length=MAX_LEN, truncation=True,
            padding="max_length", return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.y[i], dtype=torch.float32),
            "raw_score": torch.tensor(
                float(self.y[i] * y_std + y_mean), dtype=torch.float32
            )
        }

train_ds = PairDataset(TRAIN)
val_ds = PairDataset(VAL)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

class LegalDeltaModel(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.regressor = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask
        )
        # CLS representation
        pooled = out.last_hidden_state[:, 0]
        return self.regressor(self.dropout(pooled)).squeeze(-1)

model = LegalDeltaModel(MODEL_NAME).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.2, total_iters=max(1, len(train_loader))
)

def evaluate(loader):
    model.eval()
    pred_z, actual_z = [], []
    with torch.no_grad():
        for b in loader:
            ids = b["input_ids"].to(device)
            mask = b["attention_mask"].to(device)
            y = b["labels"].to(device)
            p = model(ids, mask)
            pred_z.extend(p.cpu().numpy())
            actual_z.extend(y.cpu().numpy())
    pred = np.asarray(pred_z) * y_std + y_mean
    actual = np.asarray(actual_z) * y_std + y_mean
    pred = np.clip(pred, 1, 10)
    metrics = {
        "mae": float(mean_absolute_error(actual, pred)),
        "rmse": float(np.sqrt(mean_squared_error(actual, pred))),
        "prediction_mean": float(pred.mean()),
        "prediction_std": float(pred.std()),
        "actual_std": float(actual.std())
    }
    if pred.std() > 1e-8:
        metrics["pearson"] = float(pearsonr(actual, pred).statistic)
        metrics["spearman"] = float(spearmanr(actual, pred).statistic)
    else:
        metrics["pearson"] = None
        metrics["spearman"] = None
    return metrics, pred

best_rmse = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses = []

    for b in train_loader:
        ids = b["input_ids"].to(device)
        mask = b["attention_mask"].to(device)
        y = b["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)
        pred = model(ids, mask)

        # Regression loss on standardized labels
        reg_loss = F.smooth_l1_loss(pred, y)

        # Pairwise ranking: only compare pairs with different target scores
        if len(y) > 1:
            diff_y = y[:, None] - y[None, :]
            diff_p = pred[:, None] - pred[None, :]
            upper = torch.triu(
                torch.ones_like(diff_y, dtype=torch.bool), diagonal=1
            )
            valid = upper & (diff_y.abs() > 1e-6)
            if valid.any():
                rank_loss = F.softplus(
                    -torch.sign(diff_y[valid]) * diff_p[valid]
                ).mean()
            else:
                rank_loss = torch.tensor(0.0, device=device)
        else:
            rank_loss = torch.tensor(0.0, device=device)

        loss = reg_loss + RANK_WEIGHT * rank_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_losses.append(float(loss.detach().cpu()))

    val_metrics, val_pred = evaluate(val_loader)
    history.append({
        "epoch": epoch,
        "train_loss": float(np.mean(epoch_losses)),
        **val_metrics
    })

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"loss={np.mean(epoch_losses):.4f} | "
        f"val_RMSE={val_metrics['rmse']:.4f} | "
        f"val_MAE={val_metrics['mae']:.4f} | "
        f"val_Pearson={val_metrics['pearson']} | "
        f"pred_SD={val_metrics['prediction_std']:.3f}"
    )

    # Select checkpoint using validation RMSE only
    if val_metrics["rmse"] < best_rmse:
        best_rmse = val_metrics["rmse"]
        torch.save(model.state_dict(), f"{OUT}/best_model.pt")
        tokenizer.save_pretrained(OUT)
        model.encoder.config.save_pretrained(OUT)
        with open(f"{OUT}/training_metadata.json", "w") as f:
            json.dump({
                "base_model": MODEL_NAME,
                "target_mean": y_mean,
                "target_std": y_std,
                "max_length": MAX_LEN,
                "rank_weight": RANK_WEIGHT,
                "best_validation_rmse": best_rmse,
                "best_epoch": epoch,
                "seed": SEED
            }, f, indent=2)

pd.DataFrame(history).to_csv(f"{OUT}/training_history.csv", index=False)
print("\nBest validation RMSE:", round(best_rmse, 4))
print("Saved to:", OUT)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/6 | loss=0.5786 | val_RMSE=1.1398 | val_MAE=0.9484 | val_Pearson=0.3081704080104828 | pred_SD=0.019
Epoch 2/6 | loss=0.5658 | val_RMSE=1.0873 | val_MAE=0.8625 | val_Pearson=0.4319225251674652 | pred_SD=0.154
Epoch 3/6 | loss=0.5068 | val_RMSE=0.9505 | val_MAE=0.7346 | val_Pearson=0.6812424063682556 | pred_SD=0.364
Epoch 4/6 | loss=0.3923 | val_RMSE=0.8465 | val_MAE=0.6288 | val_Pearson=0.6973046660423279 | pred_SD=1.009
Epoch 5/6 | loss=0.3430 | val_RMSE=0.8895 | val_MAE=0.6454 | val_Pearson=0.7281147837638855 | pred_SD=1.161
Epoch 6/6 | loss=0.3077 | val_RMSE=0.8129 | val_MAE=0.5994 | val_Pearson=0.7554454803466797 | pred_SD=0.969

Best validation RMSE: 0.8129
Saved to: /content/judgebert_lmp_results/legaldelta_v2


In [ ]:

# STEP 5 — Evaluate LegalDelta-BERT v2 on the held-out test set
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

BASE = "/content/judgebert_lmp_results"
OUT = f"{BASE}/legaldelta_v2"
TEST_PATH = f"{BASE}/test_source_grouped.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(f"{OUT}/training_metadata.json") as f:
    meta = json.load(f)

test = pd.read_csv(TEST_PATH)
tokenizer = AutoTokenizer.from_pretrained(OUT)

class TestDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        enc = tokenizer(
            str(r["Original"]) if pd.notna(r["Original"]) else "",
            str(r["Simplified"]) if pd.notna(r["Simplified"]) else "",
            max_length=meta["max_length"],
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0)
        }

class LegalDeltaModel(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.regressor = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask)
        return self.regressor(
            self.dropout(out.last_hidden_state[:, 0])
        ).squeeze(-1)

model = LegalDeltaModel(meta["base_model"]).to(device)
state = torch.load(f"{OUT}/best_model.pt", map_location=device)
model.load_state_dict(state)
model.eval()

loader = DataLoader(TestDataset(test), batch_size=16, shuffle=False)
pred_z = []
with torch.no_grad():
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        pred_z.extend(model(ids, mask).cpu().numpy())

pred = np.clip(np.asarray(pred_z) * meta["target_std"] +
               meta["target_mean"], 1, 10)
actual = test["LMP score"].astype(float).to_numpy()

metrics = {
    "n": len(test),
    "pearson": float(pearsonr(actual, pred).statistic),
    "spearman": float(spearmanr(actual, pred).statistic),
    "rmse": float(np.sqrt(mean_squared_error(actual, pred))),
    "mae": float(mean_absolute_error(actual, pred)),
    "prediction_mean": float(pred.mean()),
    "prediction_std": float(pred.std())
}

eval_dir = f"{BASE}/test_evaluation"
os.makedirs(eval_dir, exist_ok=True)
pd.DataFrame({
    "ID": test["ID"],
    "Language": test["Language"],
    "Domain": test["Domain"],
    "Human_LMP": actual,
    "LegalDelta_v2_prediction": pred
}).to_csv(f"{eval_dir}/legaldelta_v2_test_predictions.csv", index=False)

with open(f"{eval_dir}/legaldelta_v2_test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))
print("\nSaved predictions and metrics under:", eval_dir)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{
  "n": 79,
  "pearson": 0.7386194533237392,
  "spearman": 0.6545029178381251,
  "rmse": 1.0039310776569195,
  "mae": 0.6893574014494691,
  "prediction_mean": 9.44021224975586,
  "prediction_std": 0.8139106631278992
}

Saved predictions and metrics under: /content/judgebert_lmp_results/test_evaluation


In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

OUT = "/content/judgebert_lmp_results/test_evaluation"

baseline_path = f"{OUT}/baseline_test_predictions.csv"
new_path = f"{OUT}/legaldelta_v2_test_predictions.csv"

base = pd.read_csv(baseline_path)
new = pd.read_csv(new_path)

print("Baseline columns:", base.columns.tolist())
print("LegalDelta columns:", new.columns.tolist())
print("\nBaseline preview:")
display(base.head(2))
print("\nLegalDelta preview:")
display(new.head(2))


Baseline columns: ['ID', 'Source', 'Language', 'Domain', 'LMP score', 'TF-IDF cosine + linear calibration', 'Multilingual MiniLM cosine + linear calibration']
LegalDelta columns: ['ID', 'Language', 'Domain', 'Human_LMP', 'LegalDelta_v2_prediction']

Baseline preview:


,ID,Source,Language,Domain,LMP score,TF-IDF cosine + linear calibration,Multilingual MiniLM cosine + linear calibration
0,en_tos_080,netflix_tos,en,tos_privacy,9,9.428138,8.681295
1,en_tos_081,netflix_tos,en,tos_privacy,8,9.616701,8.765947



LegalDelta preview:


,ID,Language,Domain,Human_LMP,LegalDelta_v2_prediction
0,en_tos_080,en,tos_privacy,9.0,8.723798
1,en_tos_081,en,tos_privacy,8.0,9.585258


In [ ]:

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

OUT = "/content/judgebert_lmp_results/test_evaluation"

base = pd.read_csv(f"{OUT}/baseline_test_predictions.csv")
new = pd.read_csv(f"{OUT}/legaldelta_v2_test_predictions.csv")

# Match predictions by ID and verify that all models use the same examples
df = base.merge(
    new[["ID", "Human_LMP", "LegalDelta_v2_prediction"]],
    on="ID",
    how="inner",
    validate="one_to_one"
)

assert len(df) == 79, f"Expected 79 matched test rows, got {len(df)}"
assert np.allclose(df["LMP score"], df["Human_LMP"]), \
    "Human scores differ between prediction files"

y = df["LMP score"].astype(float).to_numpy()

models = {
    "TF-IDF baseline": "TF-IDF cosine + linear calibration",
    "Multilingual MiniLM baseline": "Multilingual MiniLM cosine + linear calibration",
    "LegalDelta-BERT v2": "LegalDelta_v2_prediction",
}

results = []
for name, col in models.items():
    pred = df[col].astype(float).to_numpy()
    results.append({
        "Model": name,
        "N": len(y),
        "Pearson": pearsonr(y, pred).statistic,
        "Spearman": spearmanr(y, pred).statistic,
        "RMSE": mean_squared_error(y, pred) ** 0.5,
        "MAE": mean_absolute_error(y, pred),
        "Prediction mean": pred.mean(),
        "Prediction SD": pred.std()
    })

comparison = pd.DataFrame(results)
display(comparison.round(4))

comparison.to_csv(f"{OUT}/final_three_model_comparison.csv", index=False)

# Inspect LegalDelta's largest errors
df["LegalDelta_abs_error"] = (
    df["LMP score"] - df["LegalDelta_v2_prediction"]
).abs()

hard_cases = df.sort_values(
    "LegalDelta_abs_error", ascending=False
)[[
    "ID", "Language", "Domain", "LMP score",
    "LegalDelta_v2_prediction", "LegalDelta_abs_error",
    "TF-IDF cosine + linear calibration",
    "Multilingual MiniLM cosine + linear calibration"
]].head(15)

display(hard_cases.round(3))
hard_cases.to_csv(f"{OUT}/legaldelta_v2_hard_cases.csv", index=False)

print("Saved final comparison and hard cases to:", OUT)


,Model,N,Pearson,Spearman,RMSE,MAE,Prediction mean,Prediction SD
0,TF-IDF baseline,79,0.6143,0.6766,1.0315,0.6830,9.2007,0.7612
1,Multilingual MiniLM baseline,79,0.4502,0.5648,1.1530,0.9236,8.6619,0.6743
2,LegalDelta-BERT v2,79,0.7386,0.6545,1.0039,0.6894,9.4402,0.8139


,ID,Language,Domain,LMP score,LegalDelta_v2_prediction,LegalDelta_abs_error,TF-IDF cosine + linear calibration,Multilingual MiniLM cosine + linear calibration
38,en_rental_057,en,rental,4,7.556,3.556,7.800,8.243
16,en_tos_096,en,tos_privacy,7,9.675,2.675,7.952,8.861
3,en_tos_083,en,tos_privacy,8,10.000,2.000,9.068,8.724
58,hi_rental_051,hi,rental,8,10.000,2.000,9.587,8.666
19,en_tos_099,en,tos_privacy,8,9.999,1.999,9.423,9.036
31,en_rental_050,en,rental,6,7.978,1.978,8.818,8.373
10,en_tos_090,en,tos_privacy,8,9.953,1.953,8.921,8.637
54,hi_rental_047,hi,rental,6,7.850,1.850,8.215,4.924
33,en_rental_052,en,rental,6,7.690,1.690,8.662,8.090
1,en_tos_081,en,tos_privacy,8,9.585,1.585,9.617,8.766


Saved final comparison and hard cases to: /content/judgebert_lmp_results/test_evaluation


In [ ]:

import os
import re
import numpy as np
import pandas as pd

DATA_PATH = "/content/lmp_final - lmp_semantic_review(1).csv"
OUT = "/content/judgebert_lmp_results/test_evaluation"

data = pd.read_csv(DATA_PATH)
preds = pd.read_csv(f"{OUT}/legaldelta_v2_test_predictions.csv")

# Confirm the expected columns
required = {"ID", "Original", "Simplified"}
missing = required - set(data.columns)
assert not missing, f"Missing columns in dataset: {missing}"

df = preds.merge(
    data[["ID", "Original", "Simplified"]],
    on="ID",
    how="left",
    validate="one_to_one"
)
assert df["Original"].notna().all(), "Some prediction IDs did not match source data."

# Transparent rule-based indicators; not model-generated explanations
patterns = {
    "negation": r"\b(not|no|never|neither|nor|cannot|can't|won't|unless)\b",
    "obligation_permission": r"\b(must|shall|required|may|permitted|prohibited| обязан|должен)\b",
    "exception_condition": r"\b(except|unless|only if|provided that|subject to|otherwise)\b",
    "money_number_deadline": r"[$₹€£]?\s*\d+(?:[.,]\d+)?\s*(?:%|days?|months?|years?|hours?)?",
    "rights_responsibility": r"\b(right|consent|liable|liability|responsible|terminate|refund|breach|claim|waive)\b"
}

def find_flags(original, simplified):
    a = str(original).lower()
    b = str(simplified).lower()
    flags = []
    for name, pattern in patterns.items():
        if bool(re.search(pattern, a)) != bool(re.search(pattern, b)):
            flags.append(name)
    return ", ".join(flags) if flags else "No rule-triggered difference"

df["identical_text"] = (
    df["Original"].astype(str).str.strip().str.casefold()
    == df["Simplified"].astype(str).str.strip().str.casefold()
)
df["rule_based_flags"] = [
    find_flags(a, b) for a, b in zip(df["Original"], df["Simplified"])
]

print("Held-out examples:", len(df))
print("Identical text pairs:", int(df["identical_text"].sum()))
print("Pairs with potential rule-triggered differences:",
      int((df["rule_based_flags"] != "No rule-triggered difference").sum()))

display(df[[
    "ID", "Language", "Domain", "Human_LMP",
    "LegalDelta_v2_prediction", "identical_text", "rule_based_flags"
]].head(10))

df[[
    "ID", "Language", "Domain", "Human_LMP",
    "LegalDelta_v2_prediction", "Original", "Simplified",
    "identical_text", "rule_based_flags"
]].to_csv(f"{OUT}/legaldelta_explainability_diagnostics.csv", index=False)

print("Saved explainability diagnostics.")


Held-out examples: 79
Identical text pairs: 1
Pairs with potential rule-triggered differences: 8


,ID,Language,Domain,Human_LMP,LegalDelta_v2_prediction,identical_text,rule_based_flags
0,en_tos_080,en,tos_privacy,9.0,8.723798,False,No rule-triggered difference
1,en_tos_081,en,tos_privacy,8.0,9.585258,False,No rule-triggered difference
2,en_tos_082,en,tos_privacy,8.0,8.960516,False,No rule-triggered difference
3,en_tos_083,en,tos_privacy,8.0,10.000000,False,No rule-triggered difference
4,en_tos_084,en,tos_privacy,9.0,10.000000,False,No rule-triggered difference
5,en_tos_085,en,tos_privacy,10.0,9.838533,False,No rule-triggered difference
6,en_tos_086,en,tos_privacy,8.0,9.121532,False,negation
7,en_tos_087,en,tos_privacy,10.0,9.889037,False,No rule-triggered difference
8,en_tos_088,en,tos_privacy,9.0,9.655184,False,No rule-triggered difference
9,en_tos_089,en,tos_privacy,10.0,9.914682,False,No rule-triggered difference


Saved explainability diagnostics.


In [ ]:

import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import json, os

MODEL_DIR = "/content/judgebert_lmp_results/legaldelta_v2"
OUT = "/content/judgebert_lmp_results/test_evaluation"

with open(f"{MODEL_DIR}/training_metadata.json") as f:
    meta = json.load(f)

print("Metadata keys:", list(meta.keys()))
print("Model directory:", MODEL_DIR)


Metadata keys: ['base_model', 'target_mean', 'target_std', 'max_length', 'rank_weight', 'best_validation_rmse', 'best_epoch', 'seed']
Model directory: /content/judgebert_lmp_results/legaldelta_v2


In [ ]:

import os, torch

MODEL_DIR = "/content/judgebert_lmp_results/legaldelta_v2"
ckpt_path = f"{MODEL_DIR}/best_model.pt"

print("Checkpoint exists:", os.path.exists(ckpt_path))
print("Checkpoint size (MB):", round(os.path.getsize(ckpt_path) / (1024**2), 2))

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

if isinstance(ckpt, dict):
    print("Top-level keys:", list(ckpt.keys())[:30])
    state = ckpt.get("model_state_dict", ckpt)
    if isinstance(state, dict):
        print("Number of state entries:", len(state))
        print("First 20 parameter names:")
        for k in list(state.keys())[:20]:
            print(k, tuple(state[k].shape) if hasattr(state[k], "shape") else type(state[k]))
else:
    print("Checkpoint type:", type(ckpt))


Checkpoint exists: True
Checkpoint size (MB): 1061.49
Top-level keys: ['encoder.embeddings.word_embeddings.weight', 'encoder.embeddings.token_type_embeddings.weight', 'encoder.embeddings.LayerNorm.weight', 'encoder.embeddings.LayerNorm.bias', 'encoder.embeddings.position_embeddings.weight', 'encoder.encoder.layer.0.attention.self.query.weight', 'encoder.encoder.layer.0.attention.self.query.bias', 'encoder.encoder.layer.0.attention.self.key.weight', 'encoder.encoder.layer.0.attention.self.key.bias', 'encoder.encoder.layer.0.attention.self.value.weight', 'encoder.encoder.layer.0.attention.self.value.bias', 'encoder.encoder.layer.0.attention.output.dense.weight', 'encoder.encoder.layer.0.attention.output.dense.bias', 'encoder.encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.encoder.layer.0.intermediate.dense.weight', 'encoder.encoder.layer.0.intermediate.dense.bias', 'encoder.encoder.layer.0.output.dense.weight', 'enco

In [ ]:

import os, json, torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel

MODEL_DIR = "/content/judgebert_lmp_results/legaldelta_v2"
OUT = "/content/judgebert_lmp_results/test_evaluation"

with open(f"{MODEL_DIR}/training_metadata.json") as f:
    meta = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
encoder = AutoModel.from_pretrained(meta["base_model"])

class LegalDeltaRegressor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.regressor = nn.Sequential(
            nn.Linear(hidden, 256),  # regressor.0
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)        # regressor.3
        )

    def forward(self, input_ids, attention_mask):
        output = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = output.last_hidden_state[:, 0, :]
        return self.regressor(self.dropout(cls)).squeeze(-1)

model = LegalDeltaRegressor(encoder)
state = torch.load(
    f"{MODEL_DIR}/best_model.pt",
    map_location="cpu",
    weights_only=False
)
model.load_state_dict(state, strict=True)
model.to(device).eval()
print("Checkpoint loaded successfully with strict=True.")

def predict_lmp(pairs):
    batch = tokenizer(
        [p[0] for p in pairs],
        [p[1] for p in pairs],
        padding=True,
        truncation=True,
        max_length=meta["max_length"],
        return_tensors="pt"
    )
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        z = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        ).cpu().numpy()

    return np.clip(
        z * meta["target_std"] + meta["target_mean"], 1, 10
    )

tests = [
    ("Identical text",
     "The tenant must pay rent by the fifth day of each month.",
     "The tenant must pay rent by the fifth day of each month."),
    ("Negation changed",
     "The tenant may terminate the agreement with written notice.",
     "The tenant may not terminate the agreement with written notice."),
    ("Unrelated pair",
     "The tenant must pay rent by the fifth day of each month.",
     "Your personal data may be shared with advertising partners.")
]

scores = predict_lmp([(a, b) for _, a, b in tests])
sanity = pd.DataFrame([
    {"Test": name, "Original": a, "Simplified": b,
     "Predicted_LMP": float(score)}
    for (name, a, b), score in zip(tests, scores)
])

display(sanity.round(3))
sanity.to_csv(f"{OUT}/legaldelta_sanity_checks.csv", index=False)
print("Saved:", f"{OUT}/legaldelta_sanity_checks.csv")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Checkpoint loaded successfully with strict=True.


,Test,Original,Simplified,Predicted_LMP
0,Identical text,The tenant must pay rent by the fifth day of e...,The tenant must pay rent by the fifth day of e...,10.000
1,Negation changed,The tenant may terminate the agreement with wr...,The tenant may not terminate the agreement wit...,9.747
2,Unrelated pair,The tenant must pay rent by the fifth day of e...,Your personal data may be shared with advertis...,7.712


Saved: /content/judgebert_lmp_results/test_evaluation/legaldelta_sanity_checks.csv


In [ ]:

import re
import pandas as pd
import numpy as np

DATA_PATH = "/content/lmp_final - lmp_semantic_review(1).csv"
OUT = "/content/judgebert_lmp_results/test_evaluation"

pred_path = f"{OUT}/legaldelta_v2_test_predictions.csv"
data = pd.read_csv(DATA_PATH)
preds = pd.read_csv(pred_path)

df = preds.merge(
    data[["ID", "Original", "Simplified"]],
    on="ID",
    how="left",
    validate="one_to_one"
)
assert len(df) == 79 and df["Original"].notna().all()

# Transparent lexical indicators, not learned explanations.
RULES = {
    "Negation": r"\b(not|no|never|neither|nor|cannot|can't|won't|don't|doesn't|unless)\b",
    "Obligation": r"\b(must|shall|required|obligated| обяз|д\u0947ना|चाहिए)\b",
    "Permission/prohibition": r"\b(may|permitted|allowed|prohibited|forbidden|नहीं कर सकता|अनुमति)\b",
    "Exception/condition": r"\b(except|unless|only if|provided that|subject to|otherwise|जब तक|सिवाय|यदि)\b",
    "Numbers/deadlines": r"[$₹€£]?\s*\d+(?:[.,]\d+)?\s*(?:%|days?|months?|years?|hours?|दिन|महीने|वर्ष)?",
    "Rights/responsibility": r"\b(right|consent|liable|liability|responsible|terminate|refund|breach|claim|waive|अधिकार|जिम्मेदार|समाप्त)\b"
}

def flags_for(original, simplified):
    a, b = str(original).casefold(), str(simplified).casefold()
    flags = []
    for label, pattern in RULES.items():
        # Flag presence changes; this does not understand full legal semantics.
        if bool(re.search(pattern, a)) != bool(re.search(pattern, b)):
            flags.append(label)
    return flags

def review_recommendation(score, flags):
    if flags:
        return "Manual legal review: potential wording change"
    if score < 7:
        return "Manual review: low predicted preservation"
    if score < 9:
        return "Review recommended: moderate predicted preservation"
    return "No lexical warning detected; not a legal guarantee"

df["Rule_based_warnings"] = [
    "; ".join(flags_for(a, b))
    for a, b in zip(df["Original"], df["Simplified"])
]
df["Rule_based_warnings"] = df["Rule_based_warnings"].replace(
    "", "No rule-triggered difference detected"
)

df["Review_recommendation"] = [
    review_recommendation(score, flags_for(a, b))
    for score, a, b in zip(
        df["LegalDelta_v2_prediction"], df["Original"], df["Simplified"]
    )
]

df["LegalDelta_v2_prediction"] = df["LegalDelta_v2_prediction"].clip(1, 10).round(3)

final = df[[
    "ID", "Language", "Domain", "Human_LMP",
    "LegalDelta_v2_prediction", "Original", "Simplified",
    "Rule_based_warnings", "Review_recommendation"
]].copy()

path = f"{OUT}/legaldelta_explainable_test_results.csv"
final.to_csv(path, index=False)

print("Rows saved:", len(final))
print("Rows with potential rule warnings:",
      int((final["Rule_based_warnings"] !=
           "No rule-triggered difference detected").sum()))
print("Saved:", path)
display(final.head(10))


Rows saved: 79
Rows with potential rule warnings: 10
Saved: /content/judgebert_lmp_results/test_evaluation/legaldelta_explainable_test_results.csv


,ID,Language,Domain,Human_LMP,LegalDelta_v2_prediction,Original,Simplified,Rule_based_warnings,Review_recommendation
0,en_tos_080,en,tos_privacy,9.0,8.724,Your contractual partner and consumer point of...,Your contract contact for these Terms of Use i...,No rule-triggered difference detected,Review recommended: moderate predicted preserv...
1,en_tos_081,en,tos_privacy,8.0,9.585,These Terms of Use govern your use of the Netf...,These Terms govern your use of Netflix.,No rule-triggered difference detected,No lexical warning detected; not a legal guara...
2,en_tos_082,en,tos_privacy,8.0,8.961,"As used in these Terms of Use, ""Netflix servic...",Netflix service or the service means the Netfl...,No rule-triggered difference detected,Review recommended: moderate predicted preserv...
3,en_tos_083,en,tos_privacy,8.0,10.000,"This includes all personalization, features an...","This includes all personalization, features, f...",No rule-triggered difference detected,No lexical warning detected; not a legal guara...
4,en_tos_084,en,tos_privacy,9.0,10.000,You must be at least 18 years of age to create...,You must be at least 18 years old to create a ...,Permission/prohibition,Manual legal review: potential wording change
5,en_tos_085,en,tos_privacy,10.0,9.839,We may from time to time provide special promo...,We may from time to time provide special promo...,No rule-triggered difference detected,No lexical warning detected; not a legal guara...
6,en_tos_086,en,tos_privacy,8.0,9.122,Offer eligibility is determined by Netflix at ...,Netflix determines offer eligibility at its so...,Negation,Manual legal review: potential wording change
7,en_tos_087,en,tos_privacy,10.0,9.889,Members of households with an existing or rece...,Members with an existing or recent Netflix acc...,No rule-triggered difference detected,No lexical warning detected; not a legal guara...
8,en_tos_088,en,tos_privacy,9.0,9.655,"We may use information such as device ID, meth...","We may use device ID, payment method, or conta...",No rule-triggered difference detected,No lexical warning detected; not a legal guara...
9,en_tos_089,en,tos_privacy,10.0,9.915,The eligibility requirements and other limitat...,The eligibility requirements and other limitat...,No rule-triggered difference detected,No lexical warning detected; not a legal guara...


In [ ]:

import os
import json
import pandas as pd

ROOT = "/content/judgebert_lmp_results"
MODEL_DIR = f"{ROOT}/legaldelta_v2"

required = {
    "best checkpoint": f"{MODEL_DIR}/best_model.pt",
    "training metadata": f"{MODEL_DIR}/training_metadata.json",
    "training data": f"{ROOT}/train_source_grouped.csv",
    "validation data": f"{ROOT}/validation_source_grouped.csv",
}

for name, path in required.items():
    print(f"{'OK' if os.path.exists(path) else 'MISSING'}: {name} — {path}")

if all(os.path.exists(p) for p in required.values()):
    with open(required["training metadata"]) as f:
        metadata = json.load(f)

    history_path = f"{MODEL_DIR}/training_history.csv"
    print("\nTraining metadata:")
    print(json.dumps(metadata, indent=2))

    if os.path.exists(history_path):
        print("\nTraining history:")
        display(pd.read_csv(history_path))

    train = pd.read_csv(required["training data"])
    val = pd.read_csv(required["validation data"])
    print("\nTrain rows:", len(train), "| Validation rows:", len(val))
    print("Train columns:", train.columns.tolist())
    print("Validation columns:", val.columns.tolist())


MISSING: best checkpoint — /content/judgebert_lmp_results/legaldelta_v2/best_model.pt
MISSING: training metadata — /content/judgebert_lmp_results/legaldelta_v2/training_metadata.json
MISSING: training data — /content/judgebert_lmp_results/train_source_grouped.csv
MISSING: validation data — /content/judgebert_lmp_results/validation_source_grouped.csv


In [ ]:

import os

search_roots = ["/content", "/content/drive/MyDrive"]

targets = {
    "best_model.pt",
    "training_metadata.json",
    "train_source_grouped.csv",
    "validation_source_grouped.csv",
    "legaldelta_v2_test_predictions.csv",
}

found = {name: [] for name in targets}

for root in search_roots:
    if not os.path.exists(root):
        continue

    for current, dirs, files in os.walk(root):
        # Skip large unrelated cache folders.
        dirs[:] = [
            d for d in dirs
            if d not in {
                ".cache", "node_modules", ".git",
                "__pycache__"
            }
        ]

        for name in targets.intersection(files):
            found[name].append(os.path.join(current, name))

for name, paths in found.items():
    print(f"\n{name}:")
    if paths:
        for path in paths:
            print(" ", path)
    else:
        print("  Not found")



validation_source_grouped.csv:
  Not found

legaldelta_v2_test_predictions.csv:
  Not found

training_metadata.json:
  Not found

best_model.pt:
  Not found

train_source_grouped.csv:
  Not found


In [ ]:

import os

print("Current runtime contents:")
print(os.listdir("/content"))

drive = "/content/drive/MyDrive"
print("\nGoogle Drive mounted:", os.path.exists(drive))

if os.path.exists(drive):
    print("Drive folders:", os.listdir(drive)[:30])


Current runtime contents:
['.config', 'lmp_final - lmp_semantic_review(1).csv', 'sample_data']

Google Drive mounted: False


In [ ]:

import pandas as pd
import os

DATA_PATH = "/content/lmp_final - lmp_semantic_review(1).csv"

df = pd.read_csv(DATA_PATH)

print("Dataset recovered:", os.path.exists(DATA_PATH))
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nLanguage distribution:")
print(df["Language"].value_counts() if "Language" in df.columns else "Column missing")

print("\nSource column available:", "Source" in df.columns)
print("\nFirst 3 rows:")
display(df.head(3))


Dataset recovered: True
Rows and columns: (357, 8)

Column names:
['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment']

Missing values:
ID                  0
Original            0
Language            0
Domain              0
Source              0
Simplified          0
LMP score           0
Human_Assessment    0
dtype: int64

Language distribution:
Language
en    205
hi    152
Name: count, dtype: int64

Source column available: True

First 3 rows:


,ID,Original,Language,Domain,Source,Simplified,LMP score,Human_Assessment
0,en_tos_001,"As a result, these Terms of Service help defin...",en,tos_privacy,google_tos,"Because of this, these Terms of Service help d...",10,Legal meaning and key details preserved — Key ...
1,en_tos_002,"Our services are designed to work together, ma...",en,tos_privacy,google_tos,Our services are designed to work together so ...,9,Legal meaning and key details preserved — Key ...
2,en_tos_003,"We also make various policies, help centers, a...",en,tos_privacy,google_tos,"We also provide various policies, help centers...",9,Legal meaning and key details preserved — Key ...


In [ ]:

import os, json
import pandas as pd

DATA_PATH = "/content/lmp_final - lmp_semantic_review(1).csv"
ROOT = "/content/judgebert_lmp_results"
os.makedirs(ROOT, exist_ok=True)

df = pd.read_csv(DATA_PATH)

# Reuse the exact source assignments from our previous experiment.
train_sources = [
    "google_tos", "lawdepot_commercial", "lawdepot_residential",
    "meta_tos", "phonepe_tos", "tata_tos",
    "vidhikInfo_rental", "whatsapp_tos"
]
val_sources = ["adhiwakta_rental", "paytm_tos"]
test_sources = ["netflix_tos", "pandadoc_rental", "yojanavani_rental"]

assignments = {
    "train": train_sources,
    "validation": val_sources,
    "test": test_sources
}

# Check every source is assigned exactly once.
all_assigned = train_sources + val_sources + test_sources
assert len(all_assigned) == len(set(all_assigned)), "Source overlap in assignments"
assert set(df["Source"].unique()) == set(all_assigned), (
    "Source list differs from the recorded split; stop and inspect."
)

for split, sources in assignments.items():
    part = df[df["Source"].isin(sources)].copy()
    path = f"{ROOT}/{split}_source_grouped.csv"
    part.to_csv(path, index=False)
    print(f"{split}: {len(part)} rows, {part['Source'].nunique()} sources")
    print("  Saved:", path)

# Save the split manifest so it can be reproduced.
manifest = {
    "method": "source-disjoint; reusing previous source assignments",
    "train_sources": train_sources,
    "validation_sources": val_sources,
    "test_sources": test_sources,
    "total_rows": len(df)
}
with open(f"{ROOT}/source_grouped_split_audit.json", "w") as f:
    json.dump(manifest, f, indent=2)

# Confirm there is no source overlap.
for a, sa in assignments.items():
    for b, sb in assignments.items():
        if a < b:
            assert set(sa).isdisjoint(sb)

print("\nSplit manifest saved. Source overlap: none.")
print("\nSplit sizes:")
print(pd.DataFrame([
    {
        "split": name,
        "rows": len(df[df["Source"].isin(sources)]),
        "English": int((df[df["Source"].isin(sources)]["Language"] == "en").sum()),
        "Hindi": int((df[df["Source"].isin(sources)]["Language"] == "hi").sum())
    }
    for name, sources in assignments.items()
]).to_string(index=False))


train: 228 rows, 8 sources
  Saved: /content/judgebert_lmp_results/train_source_grouped.csv
validation: 50 rows, 2 sources
  Saved: /content/judgebert_lmp_results/validation_source_grouped.csv
test: 79 rows, 3 sources
  Saved: /content/judgebert_lmp_results/test_source_grouped.csv

Split manifest saved. Source overlap: none.

Split sizes:
     split  rows  English  Hindi
     train   228      133     95
validation    50       31     19
      test    79       41     38


In [ ]:

from google.colab import drive
drive.mount("/content/drive")

import os
import shutil

SOURCE_DIR = "/content/judgebert_lmp_results"
BACKUP_DIR = "/content/drive/MyDrive/LegalDelta_LMP"

os.makedirs(BACKUP_DIR, exist_ok=True)

files_to_backup = [
    "/content/lmp_final - lmp_semantic_review(1).csv",
    f"{SOURCE_DIR}/train_source_grouped.csv",
    f"{SOURCE_DIR}/validation_source_grouped.csv",
    f"{SOURCE_DIR}/test_source_grouped.csv",
    f"{SOURCE_DIR}/source_grouped_split_audit.json",
]

for src in files_to_backup:
    assert os.path.exists(src), f"Missing file: {src}"
    dst = os.path.join(BACKUP_DIR, os.path.basename(src))
    shutil.copy2(src, dst)
    print("Backed up:", dst)

print("\nPersistent backup ready:", BACKUP_DIR)


Mounted at /content/drive
Backed up: /content/drive/MyDrive/LegalDelta_LMP/lmp_final - lmp_semantic_review(1).csv
Backed up: /content/drive/MyDrive/LegalDelta_LMP/train_source_grouped.csv
Backed up: /content/drive/MyDrive/LegalDelta_LMP/validation_source_grouped.csv
Backed up: /content/drive/MyDrive/LegalDelta_LMP/test_source_grouped.csv
Backed up: /content/drive/MyDrive/LegalDelta_LMP/source_grouped_split_audit.json

Persistent backup ready: /content/drive/MyDrive/LegalDelta_LMP


In [ ]:
!nvidia-smi
!pip -q install transformers sentencepiece scikit-learn scipy pandas

Sat Oct 24 17:48:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
import torch
import pandas as pd
import transformers
from pathlib import Path

BASE = Path("/content/drive/MyDrive/LegalDelta_LMP")

required_files = [
    "lmp_final - lmp_semantic_review(1).csv",
    "train_source_grouped.csv",
    "validation_source_grouped.csv",
    "test_source_grouped.csv",
    "source_grouped_split_audit.json",
]

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

for filename in required_files:
    path = BASE / filename
    print(f"{'OK' if path.exists() else 'MISSING'}: {filename}")

train = pd.read_csv(BASE / "train_source_grouped.csv")
validation = pd.read_csv(BASE / "validation_source_grouped.csv")
test = pd.read_csv(BASE / "test_source_grouped.csv")

print("\nSplit sizes:")
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

assert (len(train), len(validation), len(test)) == (228, 50, 79), \
    "Split sizes differ from the frozen experiment."
assert all((BASE / f).exists() for f in required_files), \
    "A required backup file is missing."

print("\nEnvironment and split checks passed.")

GPU: Tesla T4
PyTorch: 2.11.0+cu130
Transformers: 5.20.2
OK: lmp_final - lmp_semantic_review(1).csv
OK: train_source_grouped.csv
OK: validation_source_grouped.csv
OK: test_source_grouped.csv
OK: source_grouped_split_audit.json

Split sizes:
Train: 228
Validation: 50
Test: 79

Environment and split checks passed.


In [ ]:
import os, json, random, shutil
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)
from torch import nn

# Reproducibility
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE = "/content/drive/MyDrive/LegalDelta_LMP"
OUT = os.path.join(BASE, "LegalDelta_v2_reference")
os.makedirs(OUT, exist_ok=True)

train_df = pd.read_csv(os.path.join(BASE, "train_source_grouped.csv"))
val_df = pd.read_csv(os.path.join(BASE, "validation_source_grouped.csv"))

for frame in (train_df, val_df):
    frame["LMP"] = pd.to_numeric(frame["LMP"], errors="raise").astype(float)

CHECKPOINT = "xlm-roberta-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def tokenize(batch):
    return tokenizer(
        batch["Original"],
        batch["Simplified"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_ds = Dataset.from_pandas(
    train_df[["Original", "Simplified", "LMP"]], preserve_index=False
).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(
    val_df[["Original", "Simplified", "LMP"]], preserve_index=False
).map(tokenize, batched=True)

train_ds = train_ds.rename_column("LMP", "labels")
val_ds = val_ds.rename_column("LMP", "labels")

class LegalDeltaRegressor(nn.Module):
    def __init__(self, checkpoint):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(checkpoint)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.hidden = nn.Linear(hidden, 256)
        self.activation = nn.GELU()
        self.dropout2 = nn.Dropout(0.2)
        self.output = nn.Linear(256, 1)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]
        pred = self.output(self.dropout2(self.activation(
            self.hidden(self.dropout(pooled))
        ))).squeeze(-1)

        loss = None
        if labels is not None:
            labels = labels.float()
            loss = nn.functional.smooth_l1_loss(pred, labels)
        return {"loss": loss, "logits": pred.unsqueeze(-1)}

model = LegalDeltaRegressor(CHECKPOINT)

def metrics(eval_pred):
    from sklearn.metrics import mean_squared_error, mean_absolute_error
    from scipy.stats import pearsonr, spearmanr

    pred, y = eval_pred
    pred = np.asarray(pred).reshape(-1)
    y = np.asarray(y).reshape(-1)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y, pred))),
        "mae": float(mean_absolute_error(y, pred)),
        "pearson": float(pearsonr(y, pred).statistic),
        "spearman": float(spearmanr(y, pred).statistic),
    }

args = TrainingArguments(
    output_dir="/content/LegalDelta_v2_training",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_rmse",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    logging_strategy="epoch",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    processing_class=tokenizer,
    compute_metrics=metrics,
)

print("Training pairs:", len(train_ds))
print("Validation pairs:", len(val_ds))
print("GPU:", torch.cuda.get_device_name(0))
print("Starting training; the test split is not being used.")

train_result = trainer.train()

# Save the best validation-selected checkpoint and supporting artifacts.
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)

history = pd.DataFrame(trainer.state.log_history)
history.to_csv(os.path.join(OUT, "training_history.csv"), index=False)

with open(os.path.join(OUT, "training_metadata.json"), "w") as f:
    json.dump({
        "checkpoint": CHECKPOINT,
        "seed": SEED,
        "max_length": MAX_LENGTH,
        "train_size": len(train_ds),
        "validation_size": len(val_ds),
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_validation_rmse": trainer.state.best_metric,
        "test_evaluated": False,
    }, f, indent=2)

print("\nTraining complete.")
print("Best validation RMSE:", trainer.state.best_metric)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Saved model and history to:", OUT)

KeyError: 'LMP'

In [ ]:
import pandas as pd

BASE = "/content/drive/MyDrive/LegalDelta_LMP"

train_df = pd.read_csv(f"{BASE}/train_source_grouped.csv")
val_df = pd.read_csv(f"{BASE}/validation_source_grouped.csv")

print("Train columns:", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())

Train columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment']
Validation columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment']


In [ ]:
import os

paths = [
    "/content/drive/MyDrive/LegalDelta_LMP/LegalDelta_v2_reference",
    "/content/drive/MyDrive/LegalDelta_LMP",
    "/content/LegalDelta_v2_training",
    "/content/judgebert_lmp_results",
]

for path in paths:
    print(f"\n{path}")
    if os.path.exists(path):
        print(os.listdir(path)[:30])
    else:
        print("Not found")


/content/drive/MyDrive/LegalDelta_LMP/LegalDelta_v2_reference
[]

/content/drive/MyDrive/LegalDelta_LMP
['lmp_final - lmp_semantic_review(1).csv', 'train_source_grouped.csv', 'validation_source_grouped.csv', 'test_source_grouped.csv', 'source_grouped_split_audit.json', 'LegalDelta_v2_reference']

/content/LegalDelta_v2_training
['checkpoint-29']

/content/judgebert_lmp_results
['train_source_grouped.csv', 'source_grouped_split_audit.json', 'test_source_grouped.csv', 'validation_source_grouped.csv']


In [ ]:
import os

checkpoint = "/content/LegalDelta_v2_training/checkpoint-29"

for root, dirs, files in os.walk(checkpoint):
    level = root.replace(checkpoint, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for filename in files:
        path = os.path.join(root, filename)
        print("  " * (level + 1) + f"{filename} ({os.path.getsize(path):,} bytes)")

checkpoint-29/
  model.safetensors (1,112,987,516 bytes)


In [ ]:
import os
from safetensors import safe_open

checkpoint = "/content/LegalDelta_v2_training/checkpoint-29"
weights = os.path.join(checkpoint, "model.safetensors")

with safe_open(weights, framework="pt", device="cpu") as f:
    print("Number of tensors:", len(list(f.keys())))
    print("\nFirst 20 tensor names:")
    for key in list(f.keys())[:20]:
        print(key, tuple(f.get_tensor(key).shape))

print("\nOther checkpoint files:")
print(os.listdir(checkpoint))

Number of tensors: 203

First 20 tensor names:
encoder.embeddings.LayerNorm.bias (768,)
encoder.embeddings.LayerNorm.weight (768,)
encoder.embeddings.position_embeddings.weight (514, 768)
encoder.embeddings.token_type_embeddings.weight (1, 768)
encoder.embeddings.word_embeddings.weight (250002, 768)
encoder.encoder.layer.0.attention.output.LayerNorm.bias (768,)
encoder.encoder.layer.0.attention.output.LayerNorm.weight (768,)
encoder.encoder.layer.0.attention.output.dense.bias (768,)
encoder.encoder.layer.0.attention.output.dense.weight (768, 768)
encoder.encoder.layer.0.attention.self.key.bias (768,)
encoder.encoder.layer.0.attention.self.key.weight (768, 768)
encoder.encoder.layer.0.attention.self.query.bias (768,)
encoder.encoder.layer.0.attention.self.query.weight (768, 768)
encoder.encoder.layer.0.attention.self.value.bias (768,)
encoder.encoder.layer.0.attention.self.value.weight (768, 768)
encoder.encoder.layer.0.intermediate.dense.bias (3072,)
encoder.encoder.layer.0.intermediat

In [ ]:
from safetensors import safe_open

weights = "/content/LegalDelta_v2_training/checkpoint-29/model.safetensors"

with safe_open(weights, framework="pt", device="cpu") as f:
    for key in f.keys():
        if not key.startswith("encoder."):
            print(key, tuple(f.get_tensor(key).shape))

hidden.bias (256,)
hidden.weight (256, 768)
regressor.bias (1,)
regressor.weight (1, 256)


In [ ]:
import os
import shutil

source = "/content/LegalDelta_v2_training/checkpoint-29/model.safetensors"
backup_dir = "/content/drive/MyDrive/LegalDelta_LMP/recovered_checkpoint"
destination = os.path.join(backup_dir, "model.safetensors")

os.makedirs(backup_dir, exist_ok=True)

assert os.path.isfile(source), "Checkpoint weights not found."

shutil.copy2(source, destination)

assert os.path.getsize(source) == os.path.getsize(destination), \
    "Backup size verification failed."

print("Checkpoint weights backed up successfully.")
print("Size:", round(os.path.getsize(destination) / (1024**3), 3), "GiB")
print("Location:", destination)

Checkpoint weights backed up successfully.
Size: 1.037 GiB
Location: /content/drive/MyDrive/LegalDelta_LMP/recovered_checkpoint/model.safetensors


In [ ]:
from pathlib import Path
from transformers import AutoConfig, AutoTokenizer

checkpoint = "xlm-roberta-base"

try:
    config = AutoConfig.from_pretrained(checkpoint)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    print("Model configuration loaded:", config.model_type)
    print("Hidden size:", config.hidden_size)
    print("Vocabulary size:", config.vocab_size)
    print("Tokenizer:", tokenizer.__class__.__name__)
    print("Tokenizer vocabulary size:", len(tokenizer))
    print("Special tokens:", tokenizer.special_tokens_map)

except Exception as e:
    print("Could not load configuration/tokenizer:", repr(e))

Model configuration loaded: xlm-roberta
Hidden size: 768
Vocabulary size: 250002
Tokenizer: XLMRobertaTokenizer
Tokenizer vocabulary size: 250002
Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


In [ ]:
import os
import torch
from torch import nn
from transformers import AutoModel, AutoTokenizer, AutoConfig
from safetensors.torch import load_file

BASE = "/content/drive/MyDrive/LegalDelta_LMP"
WEIGHTS = f"{BASE}/recovered_checkpoint/model.safetensors"
OUT = f"{BASE}/recovered_checkpoint/LegalDelta_model"

class LegalDeltaRegressor(nn.Module):
    def __init__(self, checkpoint="xlm-roberta-base"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(checkpoint)
        self.dropout1 = nn.Dropout(0.2)
        self.hidden = nn.Linear(768, 256)
        self.activation = nn.GELU()
        self.dropout2 = nn.Dropout(0.2)
        self.regressor = nn.Linear(256, 1)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        pooled = outputs.last_hidden_state[:, 0, :]
        x = self.dropout1(pooled)
        x = self.activation(self.hidden(x))
        x = self.dropout2(x)
        predictions = self.regressor(x).squeeze(-1)

        loss = None
        if labels is not None:
            loss = nn.functional.smooth_l1_loss(predictions, labels.float())

        return {
            "loss": loss,
            "logits": predictions.unsqueeze(-1),
        }

model = LegalDeltaRegressor("xlm-roberta-base")

state_dict = load_file(WEIGHTS, device="cpu")
missing, unexpected = model.load_state_dict(state_dict, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

assert not missing and not unexpected, "Checkpoint keys do not match the model."

model.save_pretrained if hasattr(model, "save_pretrained") else None
os.makedirs(OUT, exist_ok=True)

# Save the PyTorch state dict and tokenizer for future recovery.
torch.save(model.state_dict(), f"{OUT}/pytorch_model.bin")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
tokenizer.save_pretrained(OUT)
AutoConfig.from_pretrained("xlm-roberta-base").save_pretrained(OUT)

print("\nCheckpoint reconstructed successfully.")
print("Saved model artifacts to:", OUT)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Missing keys: []
Unexpected keys: []

Checkpoint reconstructed successfully.
Saved model artifacts to: /content/drive/MyDrive/LegalDelta_LMP/recovered_checkpoint/LegalDelta_model


In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer

BASE = "/content/drive/MyDrive/LegalDelta_LMP"
df = pd.read_csv(f"{BASE}/train_source_grouped.csv")

tokenizer = AutoTokenizer.from_pretrained(
    f"{BASE}/recovered_checkpoint/LegalDelta_model"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

samples = df.sample(n=min(5, len(df)), random_state=42)

results = []

for _, row in samples.iterrows():
    inputs = tokenizer(
        row["Original"],
        row["Simplified"],
        truncation=True,
        max_length=256,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        prediction = model(**inputs)["logits"].item()

    results.append({
        "ID": row["ID"],
        "Human LMP score": row["LMP score"],
        "Predicted score": round(prediction, 3),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

preds = results_df["Predicted score"].to_numpy()
print("\nAll predictions finite:", bool(np.isfinite(preds).all()))
print("Prediction standard deviation:", round(float(preds.std()), 4))

           ID  Human LMP score  Predicted score
hi_rental_021               10            1.441
   en_tos_129                9            1.232
   en_tos_010                9            1.235
   hi_tos_068                9            1.495
   en_tos_016                9            1.083

All predictions finite: True
Prediction standard deviation: 0.1509


In [ ]:

# ============================================================
# LegalDelta-BERT — One Improved Training Run
# XLM-RoBERTa + standardized regression + pairwise ranking loss
#
# Frozen split: 228 train / 50 validation / 79 test
# Test set is NOT loaded or evaluated in this cell.
# ============================================================

import os
import json
import random
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -------------------- 1. Configuration --------------------

SEED = 42
BASE_MODEL = "xlm-roberta-base"
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 6
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
RANK_WEIGHT = 0.25

BASE = "/content/drive/MyDrive/LegalDelta_LMP"
OUT = os.path.join(BASE, "LegalDelta_improved_candidate")
os.makedirs(OUT, exist_ok=True)

# Keep the recovered checkpoint untouched.
assert torch.cuda.is_available(), (
    "GPU not available. Select Runtime > Change runtime type > T4 GPU."
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda")

# -------------------- 2. Load only train and validation --------------------

train_df = pd.read_csv(os.path.join(BASE, "train_source_grouped.csv"))
val_df = pd.read_csv(os.path.join(BASE, "validation_source_grouped.csv"))

for df in (train_df, val_df):
    df["LMP score"] = pd.to_numeric(df["LMP score"], errors="raise")
    assert df["LMP score"].between(1, 10).all()

assert len(train_df) == 228, f"Unexpected training size: {len(train_df)}"
assert len(val_df) == 50, f"Unexpected validation size: {len(val_df)}"
assert set(train_df["Source"]).isdisjoint(set(val_df["Source"])), (
    "Train/validation source overlap detected."
)

# Standardization parameters are learned from TRAINING data only.
target_mean = float(train_df["LMP score"].mean())
target_std = float(train_df["LMP score"].std(ddof=0))
assert target_std > 0

train_df = train_df.copy()
val_df = val_df.copy()

train_df["target"] = (train_df["LMP score"] - target_mean) / target_std
val_df["target"] = (val_df["LMP score"] - target_mean) / target_std

print("GPU:", torch.cuda.get_device_name(0))
print(f"Train: {len(train_df)} | Validation: {len(val_df)}")
print(f"Training target mean/std: {target_mean:.4f} / {target_std:.4f}")
print("Test set remains untouched.")

# -------------------- 3. Tokenization and data loaders --------------------

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

class PairDataset(Dataset):
    def __init__(self, frame):
        self.original = frame["Original"].astype(str).tolist()
        self.simplified = frame["Simplified"].astype(str).tolist()
        self.targets = frame["target"].astype(float).tolist()

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = tokenizer(
            self.original[index],
            self.simplified[index],
            truncation=True,
            max_length=MAX_LENGTH,
        )
        item["labels"] = self.targets[index]
        return item

def collate_fn(features):
    labels = torch.tensor(
        [f["labels"] for f in features], dtype=torch.float32
    )
    inputs = [{k: v for k, v in f.items() if k != "labels"}
              for f in features]
    batch = tokenizer.pad(inputs, padding=True, return_tensors="pt")
    batch["labels"] = labels
    return batch

train_loader = DataLoader(
    PairDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    generator=torch.Generator().manual_seed(SEED),
)

val_loader = DataLoader(
    PairDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

# -------------------- 4. Model --------------------

class LegalDeltaRegressor(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model)
        hidden_size = self.encoder.config.hidden_size
        self.dropout1 = nn.Dropout(0.2)
        self.hidden = nn.Linear(hidden_size, 256)
        self.activation = nn.GELU()
        self.dropout2 = nn.Dropout(0.2)
        self.regressor = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls = outputs.last_hidden_state[:, 0, :]
        x = self.dropout1(cls)
        x = self.activation(self.hidden(x))
        x = self.dropout2(x)
        return self.regressor(x).squeeze(-1)

model = LegalDeltaRegressor(BASE_MODEL).to(device)

# -------------------- 5. Combined regression + ranking loss --------------------

def combined_loss(predictions, targets):
    regression_loss = F.smooth_l1_loss(predictions, targets)

    # Compare each unique pair within the batch whose target scores differ.
    target_diff = targets[:, None] - targets[None, :]
    pred_diff = predictions[:, None] - predictions[None, :]

    upper_triangle = torch.triu(
        torch.ones_like(target_diff, dtype=torch.bool), diagonal=1
    )
    valid = upper_triangle & (target_diff != 0)

    if valid.any():
        direction = torch.sign(target_diff[valid])
        ranking_loss = F.softplus(-direction * pred_diff[valid]).mean()
    else:
        ranking_loss = predictions.new_tensor(0.0)

    total = regression_loss + RANK_WEIGHT * ranking_loss
    return total, regression_loss.detach(), ranking_loss.detach()

# -------------------- 6. Optimizer and scheduler --------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

scaler = torch.amp.GradScaler("cuda", enabled=True)

# -------------------- 7. Validation --------------------

def evaluate_validation():
    model.eval()
    predictions, targets = [], []

    with torch.no_grad():
        for batch in val_loader:
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                preds = model(**batch)

            predictions.extend(preds.float().cpu().numpy().tolist())
            targets.extend(labels.cpu().numpy().tolist())

    # Convert standardized values back to the original LMP scale.
    predictions = np.asarray(predictions) * target_std + target_mean
    targets = np.asarray(targets) * target_std + target_mean

    return {
        "rmse": float(np.sqrt(mean_squared_error(targets, predictions))),
        "mae": float(mean_absolute_error(targets, predictions)),
        "pearson": float(pearsonr(targets, predictions).statistic)
            if np.std(predictions) > 0 else 0.0,
        "spearman": float(spearmanr(targets, predictions).statistic)
            if np.std(predictions) > 0 else 0.0,
        "prediction_mean": float(predictions.mean()),
        "prediction_std": float(predictions.std()),
    }

# -------------------- 8. Train once; select by validation RMSE --------------------

best_rmse = float("inf")
best_state = None
best_epoch = None
history = []

print("\nStarting the single improved training run...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_regression = 0.0
    running_ranking = 0.0

    for batch in train_loader:
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            predictions = model(**batch)
            loss, reg_loss, rank_loss = combined_loss(predictions, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        running_regression += reg_loss.item()
        running_ranking += rank_loss.item()

    val_metrics = evaluate_validation()

    record = {
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "train_regression_loss": running_regression / len(train_loader),
        "train_ranking_loss": running_ranking / len(train_loader),
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(record)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss {record['train_loss']:.4f} | "
        f"Val RMSE {val_metrics['rmse']:.4f} | "
        f"Val MAE {val_metrics['mae']:.4f} | "
        f"Val Pearson {val_metrics['pearson']:.4f}"
    )

    if np.isfinite(val_metrics["rmse"]) and val_metrics["rmse"] < best_rmse:
        best_rmse = val_metrics["rmse"]
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

        # Save the best candidate as training proceeds.
        torch.save(best_state, os.path.join(OUT, "best_model_state.pt"))

        tokenizer.save_pretrained(os.path.join(OUT, "tokenizer"))
        model.encoder.config.save_pretrained(os.path.join(OUT, "encoder_config"))

# -------------------- 9. Restore best epoch and save artifacts --------------------

assert best_state is not None, "No valid checkpoint was selected."

model.load_state_dict(best_state)
model.to(device)

final_val_metrics = evaluate_validation()

pd.DataFrame(history).to_csv(
    os.path.join(OUT, "training_history.csv"), index=False
)

metadata = {
    "project": "LegalDelta-BERT",
    "base_model": BASE_MODEL,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "epochs_requested": EPOCHS,
    "best_epoch": best_epoch,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "ranking_loss_weight": RANK_WEIGHT,
    "regression_loss": "SmoothL1",
    "target_mean_train": target_mean,
    "target_std_train": target_std,
    "train_size": len(train_df),
    "validation_size": len(val_df),
    "best_validation_metrics": final_val_metrics,
    "test_evaluated": False,
}

with open(os.path.join(OUT, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("\n========== RUN COMPLETE ==========")
print("Best epoch:", best_epoch)
print("Best validation metrics:")
for key, value in final_val_metrics.items():
    print(f"{key}: {value:.4f}")

print("\nSaved candidate to:", OUT)
print("Best weights: best_model_state.pt")
print("Training history: training_history.csv")
print("Metadata: metadata.json")
print("Test set was not loaded or evaluated.")


GPU: Tesla T4
Train: 228 | Validation: 50
Training target mean/std: 8.5921 / 1.3814
Test set remains untouched.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Starting the single improved training run...
Epoch 1/6 | Train loss 0.5773 | Val RMSE 1.1420 | Val MAE 0.9590 | Val Pearson 0.3698
Epoch 2/6 | Train loss 0.5726 | Val RMSE 1.0413 | Val MAE 0.8481 | Val Pearson 0.5120
Epoch 3/6 | Train loss 0.5135 | Val RMSE 0.9324 | Val MAE 0.7632 | Val Pearson 0.6206
Epoch 4/6 | Train loss 0.4137 | Val RMSE 0.8194 | Val MAE 0.6265 | Val Pearson 0.6987
Epoch 5/6 | Train loss 0.3610 | Val RMSE 0.7817 | Val MAE 0.5960 | Val Pearson 0.7385
Epoch 6/6 | Train loss 0.3259 | Val RMSE 0.7526 | Val MAE 0.5760 | Val Pearson 0.7573

========== RUN COMPLETE ==========
Best epoch: 6
Best validation metrics:
rmse: 0.7526
mae: 0.5760
pearson: 0.7573
spearman: 0.7399
prediction_mean: 8.9752
prediction_std: 0.8435

Saved candidate to: /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate
Best weights: best_model_state.pt
Training history: training_history.csv
Metadata: metadata.json
Test set was not loaded or evaluated.


In [ ]:

import os
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

BASE = "/content/drive/MyDrive/LegalDelta_LMP"
OUT = f"{BASE}/LegalDelta_improved_candidate"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 256
BATCH_SIZE = 8

with open(f"{OUT}/metadata.json") as f:
    meta = json.load(f)

target_mean = meta["target_mean_train"]
target_std = meta["target_std_train"]

test_df = pd.read_csv(f"{BASE}/test_source_grouped.csv")
assert len(test_df) == 79
test_df["LMP score"] = pd.to_numeric(test_df["LMP score"], errors="raise")

train_sources = set(pd.read_csv(
    f"{BASE}/train_source_grouped.csv"
)["Source"])
val_sources = set(pd.read_csv(
    f"{BASE}/validation_source_grouped.csv"
)["Source"])
assert set(test_df["Source"]).isdisjoint(train_sources | val_sources)

tokenizer = AutoTokenizer.from_pretrained(f"{OUT}/tokenizer")

class LegalDeltaRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained("xlm-roberta-base")
        self.dropout1 = nn.Dropout(0.2)
        self.hidden = nn.Linear(768, 256)
        self.activation = nn.GELU()
        self.dropout2 = nn.Dropout(0.2)
        self.regressor = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask):
        x = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]
        x = self.dropout1(x)
        x = self.activation(self.hidden(x))
        x = self.dropout2(x)
        return self.regressor(x).squeeze(-1)

class TestDataset(Dataset):
    def __init__(self, df):
        self.original = df["Original"].astype(str).tolist()
        self.simplified = df["Simplified"].astype(str).tolist()

    def __len__(self):
        return len(self.original)

    def __getitem__(self, i):
        return tokenizer(
            self.original[i],
            self.simplified[i],
            truncation=True,
            max_length=MAX_LENGTH
        )

def collate_fn(features):
    return tokenizer.pad(features, padding=True, return_tensors="pt")

loader = DataLoader(
    TestDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

model = LegalDeltaRegressor()
state = torch.load(
    f"{OUT}/best_model_state.pt",
    map_location="cpu",
    weights_only=True
)
model.load_state_dict(state, strict=True)
model.to(DEVICE).eval()

predictions = []
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            pred = model(**batch)
        predictions.extend(pred.float().cpu().numpy())

predictions = np.asarray(predictions) * target_std + target_mean
# LMP scores are on a 1–10 scale; clip predictions to that scale.
predictions = np.clip(predictions, 1.0, 10.0)
targets = test_df["LMP score"].to_numpy(dtype=float)

metrics = {
    "test_size": int(len(targets)),
    "rmse": float(np.sqrt(mean_squared_error(targets, predictions))),
    "mae": float(mean_absolute_error(targets, predictions)),
    "pearson": float(pearsonr(targets, predictions).statistic),
    "spearman": float(spearmanr(targets, predictions).statistic),
    "prediction_mean": float(predictions.mean()),
    "prediction_std": float(predictions.std()),
    "test_evaluated_once": True,
}

results = test_df.copy()
results["Predicted LMP score"] = predictions.round(4)
results["Absolute error"] = np.abs(targets - predictions).round(4)
results.to_csv(f"{OUT}/test_predictions.csv", index=False)

with open(f"{OUT}/test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Final held-out test metrics:")
for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nSaved test predictions and metrics to:", OUT)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Final held-out test metrics:
test_size: 79
rmse: 0.9062215267970672
mae: 0.6597336757032177
pearson: 0.7283554457150372
spearman: 0.6489894925885573
prediction_mean: 9.124247550964355
prediction_std: 0.722238302230835
test_evaluated_once: True

Saved test predictions and metrics to: /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate


In [ ]:

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

ROOT = "/content/drive/MyDrive/LegalDelta_LMP"
OUT = f"{ROOT}/LegalDelta_improved_candidate"

# Load the already-saved predictions from the one-time test evaluation
pred_path = f"{OUT}/test_predictions.csv"
pred_df = pd.read_csv(pred_path)

print("Columns:", pred_df.columns.tolist())
print("Rows:", len(pred_df))
display(pred_df.head())


Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment', 'Predicted LMP score', 'Absolute error']
Rows: 79


,ID,Original,Language,Domain,Source,Simplified,LMP score,Human_Assessment,Predicted LMP score,Absolute error
0,en_tos_080,Your contractual partner and consumer point of...,en,tos_privacy,netflix_tos,Your contract contact for these Terms of Use i...,9,Legal meaning and key details preserved — Key ...,8.4142,0.5858
1,en_tos_081,These Terms of Use govern your use of the Netf...,en,tos_privacy,netflix_tos,These Terms govern your use of Netflix.,8,"Mostly preserved, with minor concerns — Core m...",9.4069,1.4069
2,en_tos_082,"As used in these Terms of Use, ""Netflix servic...",en,tos_privacy,netflix_tos,Netflix service or the service means the Netfl...,8,"Mostly preserved, with minor concerns — Core m...",8.3609,0.3609
3,en_tos_083,"This includes all personalization, features an...",en,tos_privacy,netflix_tos,"This includes all personalization, features, f...",8,"Mostly preserved, with minor concerns — Core m...",9.8049,1.8049
4,en_tos_084,You must be at least 18 years of age to create...,en,tos_privacy,netflix_tos,You must be at least 18 years old to create a ...,9,Legal meaning and key details preserved — Key ...,9.6146,0.6146


In [ ]:

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

ROOT = "/content/drive/MyDrive/LegalDelta_LMP"
OUT = f"{ROOT}/LegalDelta_improved_candidate"

df = pd.read_csv(f"{OUT}/test_predictions.csv")

results = []

for language, group in df.groupby("Language"):
    actual = group["LMP score"].astype(float).to_numpy()
    predicted = group["Predicted LMP score"].astype(float).to_numpy()

    results.append({
        "Language": language,
        "Test examples": len(group),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAE": mean_absolute_error(actual, predicted),
        "Pearson": pearsonr(actual, predicted).statistic
            if len(group) >= 2 and np.std(actual) > 0 and np.std(predicted) > 0
            else np.nan,
        "Spearman": spearmanr(actual, predicted).statistic
            if len(group) >= 2 and np.std(actual) > 0 and np.std(predicted) > 0
            else np.nan,
        "Mean human score": actual.mean(),
        "Mean predicted score": predicted.mean()
    })

language_results = pd.DataFrame(results).round(4)

display(language_results)

language_results.to_csv(
    f"{OUT}/language_wise_test_metrics.csv",
    index=False
)

print("Saved to:", f"{OUT}/language_wise_test_metrics.csv")


,Language,Test examples,RMSE,MAE,Pearson,Spearman,Mean human score,Mean predicted score
0,en,41,1.0259,0.7823,0.7435,0.6425,8.6098,8.9917
1,hi,38,0.7561,0.5275,0.6670,0.6498,9.2368,9.2673


Saved to: /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate/language_wise_test_metrics.csv


In [ ]:

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

ROOT = "/content/drive/MyDrive/LegalDelta_LMP"
OUT = f"{ROOT}/LegalDelta_improved_candidate"

df = pd.read_csv(f"{OUT}/test_predictions.csv")

results = []

for domain, group in df.groupby("Domain", dropna=False):
    actual = group["LMP score"].astype(float).to_numpy()
    predicted = group["Predicted LMP score"].astype(float).to_numpy()

    results.append({
        "Domain": domain,
        "Test examples": len(group),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAE": mean_absolute_error(actual, predicted),
        "Pearson": (
            pearsonr(actual, predicted).statistic
            if len(group) >= 2 and np.std(actual) > 0
            and np.std(predicted) > 0 else np.nan
        ),
        "Spearman": (
            spearmanr(actual, predicted).statistic
            if len(group) >= 2 and np.std(actual) > 0
            and np.std(predicted) > 0 else np.nan
        ),
        "Mean human score": actual.mean(),
        "Mean predicted score": predicted.mean()
    })

domain_results = pd.DataFrame(results).round(4)
display(domain_results)

domain_results.to_csv(
    f"{OUT}/domain_wise_test_metrics.csv",
    index=False
)

print("Saved to:", f"{OUT}/domain_wise_test_metrics.csv")


,Domain,Test examples,RMSE,MAE,Pearson,Spearman,Mean human score,Mean predicted score
0,rental,48,0.9078,0.6037,0.7802,0.7244,8.9583,9.0786
1,tos_privacy,31,0.9038,0.7465,0.6399,0.5560,8.8387,9.1950


Saved to: /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate/domain_wise_test_metrics.csv


In [ ]:

import os, json
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

ROOT = "/content/drive/MyDrive/LegalDelta_LMP"
MODEL_DIR = f"{ROOT}/LegalDelta_improved_candidate"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(f"{MODEL_DIR}/metadata.json") as f:
    meta = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(f"{MODEL_DIR}/tokenizer")

class LegalDeltaRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained("xlm-roberta-base")
        self.dropout1 = nn.Dropout(0.2)
        self.hidden = nn.Linear(self.encoder.config.hidden_size, 256)
        self.activation = nn.GELU()
        self.dropout2 = nn.Dropout(0.2)
        self.regressor = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask):
        x = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]
        x = self.dropout1(x)
        x = self.hidden(x)
        x = self.activation(x)
        x = self.dropout2(x)
        return self.regressor(x).squeeze(-1)

model = LegalDeltaRegressor().to(DEVICE)
state = torch.load(
    f"{MODEL_DIR}/best_model_state.pt",
    map_location=DEVICE,
    weights_only=True
)
model.load_state_dict(state, strict=True)
model.eval()

pairs = [
    (
        "Identical",
        "The agreement starts on 1 January.",
        "The agreement starts on 1 January."
    ),
    (
        "Minor wording change",
        "The user must notify the company within 30 days.",
        "The user is required to notify the company within 30 days."
    ),
    (
        "Negation changed",
        "The user may disclose confidential information.",
        "The user may not disclose confidential information."
    ),
    (
        "Unrelated",
        "The tenant must pay rent by the fifth day of each month.",
        "Photosynthesis converts light energy into chemical energy in plants."
    )
]

@torch.no_grad()
def score_pair(original, simplified):
    encoded = tokenizer(
        original, simplified,
        max_length=256,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    prediction = model(**encoded).float().item()
    score = prediction * meta["target_std_train"] + meta["target_mean_train"]
    return float(np.clip(score, 1, 10))

rows = []
for name, original, simplified in pairs:
    rows.append({
        "Case": name,
        "Predicted LMP score": round(score_pair(original, simplified), 4)
    })

import pandas as pd
display(pd.DataFrame(rows))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Case,Predicted LMP score
0,Identical,9.2075
1,Minor wording change,9.2327
2,Negation changed,9.4575
3,Unrelated,8.4006


In [ ]:

import pandas as pd
import numpy as np

ROOT = "/content/drive/MyDrive/LegalDelta_LMP"
OUT = f"{ROOT}/LegalDelta_improved_candidate"

df = pd.read_csv(f"{OUT}/test_predictions.csv")

# Select examples with the largest differences from human scores
hard_cases = df.copy()
hard_cases["Absolute error"] = (
    hard_cases["LMP score"] - hard_cases["Predicted LMP score"]
).abs()

hard_cases = hard_cases.sort_values(
    "Absolute error", ascending=False
).head(15)

columns = [
    "ID", "Language", "Domain", "Source",
    "Original", "Simplified",
    "LMP score", "Predicted LMP score",
    "Absolute error", "Human_Assessment"
]

display(hard_cases[columns].reset_index(drop=True))

hard_cases[columns].to_csv(
    f"{OUT}/15_high_error_cases.csv",
    index=False
)

print("Saved to:", f"{OUT}/15_high_error_cases.csv")


,ID,Language,Domain,Source,Original,Simplified,LMP score,Predicted LMP score,Absolute error,Human_Assessment
0,en_rental_057,en,rental,pandadoc_rental,Within 30 days of the termination of this leas...,Within 30 days after the lease ends and Tenant...,4,7.2687,3.2687,"Mostly preserved, with minor concerns — The or..."
1,hi_rental_047,hi,rental,yojanavani_rental,अचल संपति को किराये पर देने की बाबत किसी प्रका...,अचल संपत्ति को किराये पर देने की कोई रूकावट नह...,6,8.3213,2.3213,Partially preserved; meaningful details may be...
2,en_tos_091,en,tos_privacy,netflix_tos,For the purpose of determining the taxes that ...,"For tax purposes, we will use the IP address y...",6,8.1577,2.1577,Significant change in legal meaning — The orig...
3,hi_rental_044,hi,rental,yojanavani_rental,यह है कि जब भी किसी पक्ष को उपरोक्त अचल सम्पति...,यदि किसी पक्ष को उपरोक्त अचल सम्पत्ति को खाली ...,10,7.9679,2.0321,Legal meaning and key details preserved — Key ...
4,en_rental_050,en,rental,pandadoc_rental,The rental term begins on [Date] (“Start Date”...,"The rental term starts on [Date] (""Start Date""...",6,7.8933,1.8933,Partially preserved; meaningful details may be...
5,en_tos_083,en,tos_privacy,netflix_tos,"This includes all personalization, features an...","This includes all personalization, features, f...",8,9.8049,1.8049,"Mostly preserved, with minor concerns — Core m..."
6,en_tos_096,en,tos_privacy,netflix_tos,"In addition, various aspects of the service ar...","The service, including websites, user interfac...",7,8.7843,1.7843,"Mostly preserved, with minor concerns — Core m..."
7,hi_rental_051,hi,rental,yojanavani_rental,प्रथम पक्ष ने अपनी उक्त राशि ————— रुपये प्रति...,प्रथम पक्ष ने द्वितीय पक्ष को रुपये ————— प्रत...,8,9.7630,1.7630,"Mostly preserved, with minor concerns — Core m..."
8,en_tos_099,en,tos_privacy,netflix_tos,Subject to these Terms and only to the extent ...,"Subject to these Terms, we grant you a limited...",8,9.6619,1.6619,"Mostly preserved, with minor concerns — Core m..."
9,en_tos_081,en,tos_privacy,netflix_tos,These Terms of Use govern your use of the Netf...,These Terms govern your use of Netflix.,8,9.4069,1.4069,"Mostly preserved, with minor concerns — Core m..."


Saved to: /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate/15_high_error_cases.csv


In [ ]:
import os
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoConfig, AutoModel

# --------------------------------------------------
# 1. Paths and device
# --------------------------------------------------
MODEL_DIR = (
    "/content/drive/MyDrive/LegalDelta_LMP/"
    "LegalDelta_improved_candidate"
)

CHECKPOINT_PATH = os.path.join(MODEL_DIR, "best_model_state.pt")
TOKENIZER_DIR = os.path.join(MODEL_DIR, "tokenizer")
CONFIG_DIR = os.path.join(MODEL_DIR, "encoder_config")

OUTPUT_PATH = os.path.join(
    MODEL_DIR, "sanity_check_results.csv"
)

assert os.path.isfile(CHECKPOINT_PATH), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------------
# 2. Load tokenizer and encoder configuration
# --------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
encoder_config = AutoConfig.from_pretrained(CONFIG_DIR)

# --------------------------------------------------
# 3. Recreate model architecture
# --------------------------------------------------
class LegalLMPModel(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.encoder = AutoModel.from_config(config)
        self.dropout = torch.nn.Dropout(0.2)
        self.hidden = torch.nn.Linear(config.hidden_size, 256)
        self.regressor = torch.nn.Linear(256, 1)
        self.activation = torch.nn.GELU()

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls)
        x = self.hidden(x)
        x = self.activation(x)
        x = self.dropout(x)
        return self.regressor(x).squeeze(-1)

model = LegalLMPModel(encoder_config)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

# Handle either a direct state_dict or a wrapped checkpoint
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    checkpoint = checkpoint["state_dict"]

model.load_state_dict(checkpoint)
model.to(device)
model.eval()

print("Checkpoint loaded successfully.")

# --------------------------------------------------
# 4. Read target scaling metadata
# --------------------------------------------------
with open(os.path.join(MODEL_DIR, "metadata.json"), "r") as f:
    metadata = json.load(f)

print("Metadata keys:", list(metadata.keys()))

# Find the saved train target mean and standard deviation.
# Do not silently assume the wrong scale.
def find_metadata_value(obj, candidate_names):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key.lower() in candidate_names:
                return value
        for value in obj.values():
            result = find_metadata_value(value, candidate_names)
            if result is not None:
                return result
    return None

target_mean = find_metadata_value(
    metadata, {"target_mean", "y_mean", "train_target_mean"}
)
target_std = find_metadata_value(
    metadata, {"target_std", "y_std", "train_target_std"}
)

if target_mean is None or target_std is None:
    raise ValueError(
        "Could not identify target mean/std in metadata.json. "
        "Inspect the printed metadata keys and use the exact "
        "scaling values from training before continuing."
    )

target_mean = float(target_mean)
target_std = float(target_std)

print(f"Target mean: {target_mean}")
print(f"Target std:  {target_std}")

# --------------------------------------------------
# 5. Controlled sanity-check examples
# --------------------------------------------------
pairs = [
    {
        "case": "Identical",
        "original": "The agreement starts on 1 January.",
        "simplified": "The agreement starts on 1 January.",
        "expected": "Very high"
    },
    {
        "case": "Minor paraphrase",
        "original": "The user must notify the company within 30 days.",
        "simplified": "The user is required to notify the company within 30 days.",
        "expected": "Very high"
    },
    {
        "case": "Negation reversal",
        "original": "The user may disclose confidential information.",
        "simplified": "The user may not disclose confidential information.",
        "expected": "Lower"
    },
    {
        "case": "Unrelated sentences",
        "original": "The tenant must pay rent by the fifth day of each month.",
        "simplified": "Photosynthesis converts light energy into chemical energy in plants.",
        "expected": "Very low"
    },
]

# --------------------------------------------------
# 6. Predict scores
# --------------------------------------------------
results = []

for pair in pairs:
    encoded = tokenizer(
        pair["original"],
        pair["simplified"],
        truncation=True,
        max_length=256,
        padding="max_length",
        return_tensors="pt"
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        scaled_prediction = model(
            input_ids=encoded["input_ids"],
            attention_mask=encoded["attention_mask"]
        ).item()

    # Convert standardized model output back to the 1–10 score scale
    score = scaled_prediction * target_std + target_mean
    score = max(1.0, min(10.0, score))

    results.append({
        "case": pair["case"],
        "original": pair["original"],
        "simplified": pair["simplified"],
        "expected_behavior": pair["expected"],
        "predicted_LMP_score": round(score, 3)
    })

results_df = pd.DataFrame(results)

# --------------------------------------------------
# 7. Display and save
# --------------------------------------------------
display(results_df)

results_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nResults saved to: {OUTPUT_PATH}")

Device: cuda
Checkpoint loaded successfully.
Metadata keys: ['project', 'base_model', 'seed', 'max_length', 'epochs_requested', 'best_epoch', 'learning_rate', 'batch_size', 'ranking_loss_weight', 'regression_loss', 'target_mean_train', 'target_std_train', 'train_size', 'validation_size', 'best_validation_metrics', 'test_evaluated']


ValueError: Could not identify target mean/std in metadata.json. Inspect the printed metadata keys and use the exact scaling values from training before continuing.

In [ ]:
# Use the exact keys from your metadata.json
target_mean = float(metadata["target_mean_train"])
target_std = float(metadata["target_std_train"])

print("Target mean:", target_mean)
print("Target std:", target_std)

pairs = [
    {
        "case": "Identical",
        "original": "The agreement starts on 1 January.",
        "simplified": "The agreement starts on 1 January.",
        "expected": "Very high"
    },
    {
        "case": "Minor paraphrase",
        "original": "The user must notify the company within 30 days.",
        "simplified": "The user is required to notify the company within 30 days.",
        "expected": "Very high"
    },
    {
        "case": "Negation reversal",
        "original": "The user may disclose confidential information.",
        "simplified": "The user may not disclose confidential information.",
        "expected": "Lower"
    },
    {
        "case": "Unrelated sentences",
        "original": "The tenant must pay rent by the fifth day of each month.",
        "simplified": "Photosynthesis converts light energy into chemical energy in plants.",
        "expected": "Very low"
    },
]

results = []

for pair in pairs:
    inputs = tokenizer(
        pair["original"],
        pair["simplified"],
        truncation=True,
        max_length=256,
        padding="max_length",
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        scaled_pred = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        ).item()

    score = scaled_pred * target_std + target_mean
    score = max(1.0, min(10.0, score))

    results.append({
        "case": pair["case"],
        "original": pair["original"],
        "simplified": pair["simplified"],
        "expected_behavior": pair["expected"],
        "predicted_LMP_score": round(score, 3)
    })

results_df = pd.DataFrame(results)
display(results_df)

output_path = os.path.join(MODEL_DIR, "sanity_check_results.csv")
results_df.to_csv(output_path, index=False)
print("Saved:", output_path)

Target mean: 8.592105263157896
Target std: 1.381370076107515


,case,original,simplified,expected_behavior,predicted_LMP_score
0,Identical,The agreement starts on 1 January.,The agreement starts on 1 January.,Very high,9.207
1,Minor paraphrase,The user must notify the company within 30 days.,The user is required to notify the company wit...,Very high,9.233
2,Negation reversal,The user may disclose confidential information.,The user may not disclose confidential informa...,Lower,9.457
3,Unrelated sentences,The tenant must pay rent by the fifth day of e...,Photosynthesis converts light energy into chem...,Very low,8.401


Saved: /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate/sanity_check_results.csv


In [ ]:
import pandas as pd
import os

path = (
    "/content/drive/MyDrive/LegalDelta_LMP/"
    "LegalDelta_improved_candidate/15_high_error_cases.csv"
)

errors = pd.read_csv(path)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

display(
    errors[
        [
            "ID",
            "Language",
            "Domain",
            "Original",
            "Simplified",
            "LMP score",
            "Predicted LMP score",
            "Absolute error",
        ]
    ].sort_values("Absolute error", ascending=False)
)

,ID,Language,Domain,Original,Simplified,LMP score,Predicted LMP score,Absolute error
0,en_rental_057,en,rental,"Within 30 days of the termination of this lease (and Tenant vacating the Property), Landlord will provide the Tenants with an itemized statement showing any retention of any portions of the Security Deposit amount for the cost of repair or cleaning above and beyond the ordinary wear and tear on the Property or any other sums due to Landlord under this Residential Lease Agreement.","Within 30 days after the lease ends and Tenant moves out, Landlord will give Tenants a list showing any part of the Security Deposit kept for repairs or cleaning beyond normal wear and tear or other amounts owed under this agreement.",4,7.2687,3.2687
1,hi_rental_047,hi,rental,अचल संपति को किराये पर देने की बाबत किसी प्रकार की कोई रूकावट किसी विभाग या किसी न्यायालय की नहीं है।,अचल संपत्ति को किराये पर देने की कोई रूकावट नहीं है।,6,8.3213,2.3213
2,en_tos_091,en,tos_privacy,"For the purpose of determining the taxes that will be charged over your subscription fee, we will use the IP address that is used when you sign up to the Netflix service.","For tax purposes, we will use the IP address you provide when signing up for Netflix to determine your subscription fee tax.",6,8.1577,2.1577
3,hi_rental_044,hi,rental,यह है कि जब भी किसी पक्ष को उपरोक्त अचल सम्पति को खाली करना या कराना हो तो वह दूसरे पक्ष को दो महिने पहले नोटिस देगा।,यदि किसी पक्ष को उपरोक्त अचल सम्पत्ति को खाली करना हो तो वह दूसरे पक्ष को दो महिने पहले नोटिस देगा।,10,7.9679,2.0321
4,en_rental_050,en,rental,The rental term begins on [Date] (“Start Date”) and continues as follows:— OPTION 1 Month-to-month. The rental tenancy of the Property will continue on a month-to-month basis until terminated under the terms of this agreement.,"The rental term starts on [Date] (""Start Date"") and continues month-to-month until terminated as per this agreement.",6,7.8933,1.8933
5,en_tos_083,en,tos_privacy,"This includes all personalization, features and functionalities, recommendations and reviews, related websites, applications and user interfaces, as well as all content and software associated with the service.","This includes all personalization, features, functionalities, recommendations, reviews, related websites, apps, user interfaces, and associated content and software.",8,9.8049,1.8049
6,en_tos_096,en,tos_privacy,"In addition, various aspects of the service are continually tested, including the websites, user interfaces and promotional features.","The service, including websites, user interfaces, and promotional features, is regularly tested.",7,8.7843,1.7843
7,hi_rental_051,hi,rental,प्रथम पक्ष ने अपनी उक्त राशि ————— रुपये प्रति मास किराये पर द्वितीय पक्ष को निम्नलिखित शर्तों पर दी है।,प्रथम पक्ष ने द्वितीय पक्ष को रुपये ————— प्रति मास किराये पर निम्नलिखित शर्तों पर दी है।,8,9.7630,1.7630
8,en_tos_099,en,tos_privacy,"Subject to these Terms and only to the extent you are expressly authorised, we grant you a limited, non-exclusive, non-transferable right to access the Netflix service.","Subject to these Terms, we grant you a limited, non-exclusive, non-transferable right to access the Netflix service only as expressly authorized.",8,9.6619,1.6619
9,en_tos_081,en,tos_privacy,These Terms of Use govern your use of the Netflix service.,These Terms govern your use of Netflix.,8,9.4069,1.4069


In [ ]:
import os

BASE = "/content/drive/MyDrive/LegalDelta_LMP"
MODEL = os.path.join(BASE, "LegalDelta_improved_candidate")

files = [
    os.path.join(BASE, "source_grouped_split_audit.json"),
    os.path.join(MODEL, "metadata.json"),
    os.path.join(MODEL, "test_metrics.json"),
    os.path.join(MODEL, "test_predictions.csv"),
    os.path.join(MODEL, "language_wise_test_metrics.csv"),
    os.path.join(MODEL, "domain_wise_test_metrics.csv"),
    os.path.join(MODEL, "sanity_check_results.csv"),
    os.path.join(MODEL, "15_high_error_cases.csv"),
]

for path in files:
    print(("OK      " if os.path.isfile(path) else "MISSING "), path)

OK       /content/drive/MyDrive/LegalDelta_LMP/source_grouped_split_audit.json
OK       /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate/metadata.json
OK       /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate/test_metrics.json
OK       /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate/test_predictions.csv
OK       /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate/language_wise_test_metrics.csv
OK       /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate/domain_wise_test_metrics.csv
OK       /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate/sanity_check_results.csv
OK       /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_improved_candidate/15_high_error_cases.csv


In [ ]:
from pathlib import Path
import json
import pandas as pd

BASE = Path("/content/drive/MyDrive/LegalDelta_LMP")
MODEL = BASE / "LegalDelta_improved_candidate"
OUT = BASE / "LegalDelta_Final_Results_Report.md"

# Load existing results only
with open(MODEL / "test_metrics.json") as f:
    test = json.load(f)

with open(MODEL / "metadata.json") as f:
    meta = json.load(f)

lang = pd.read_csv(MODEL / "language_wise_test_metrics.csv")
domain = pd.read_csv(MODEL / "domain_wise_test_metrics.csv")
sanity = pd.read_csv(MODEL / "sanity_check_results.csv")
errors = pd.read_csv(MODEL / "15_high_error_cases.csv")

# Previously recorded baseline metrics
baseline = pd.DataFrame([
    ["TF-IDF cosine + linear calibration", 0.6143, 0.6766, 1.0315, 0.6830],
    ["Multilingual MiniLM cosine + calibration", 0.4502, 0.5648, 1.1530, 0.9236],
    ["XLM-R paired-text regression",
     test["pearson"], test["spearman"], test["rmse"], test["mae"]],
], columns=["Model", "Pearson", "Spearman", "RMSE", "MAE"])

report = f"""# LegalDelta: Legal Meaning Preservation (LMP)

## 1. Objective

This project predicts how well a simplified legal sentence preserves
the meaning of its original version. It uses English and Hindi examples
from rental and terms-of-service/privacy contexts.

The proposed XLM-R paired-text regression model is inspired by
JUDGEBERT; it is not an exact reproduction of JUDGEBERT.

## 2. Dataset and evaluation

- Training examples: {meta.get('train_size', 'N/A')}
- Validation examples: {meta.get('validation_size', 'N/A')}
- Held-out test examples: {test.get('test_size', 79)}
- Languages: English and Hindi
- Split: source-disjoint train, validation, and test sets
- Human target: LMP score on a 1–10 scale

The reported test results are from saved artifacts. No new test
predictions or training runs were performed to create this report.

## 3. Overall model comparison

{baseline.round(4).to_markdown(index=False)}

The proposed XLM-R model improves on the recorded TF-IDF baseline in
RMSE, MAE, and Pearson correlation, but not Spearman correlation.
It outperforms the recorded MiniLM baseline on these four metrics.
The baseline figures are previously recorded results, not rerun here.

## 4. Language-wise results

{lang.to_markdown(index=False)}

These results are descriptive. The subsets are small and their score
distributions differ, so they do not establish that the model is better
at understanding one language.

## 5. Domain-wise results

{domain.to_markdown(index=False)}

This analysis covers only the represented domains and has limited
sample sizes. It is an initial generalization check, not proof of
generalization across all legal domains.

## 6. Sanity checks

{sanity.to_markdown(index=False)}

The model scores identical and mildly paraphrased pairs highly, as
expected. However, it assigns a high score to a pair where negation
reverses the legal meaning and to an unrelated pair. These are
qualitative probes, not formal benchmark metrics with human gold labels.

## 7. Difficult-case analysis

The following 15 examples were selected for high absolute error.
They are useful for diagnosing errors but are not a representative
random sample.

{errors[['ID', 'Language', 'Domain', 'LMP score',
         'Predicted LMP score', 'Absolute error']].to_markdown(index=False)}

Review the full original and simplified clauses when describing each
case. Potential error categories include omitted conditions, deadlines,
scope restrictions, permission changes, and over- or underestimation.
These classifications should not be treated as formal legal opinions.

## 8. Limitations

1. The dataset is modest in size for multilingual legal regression.
2. The available artifacts do not establish professional legal review
   or inter-annotator agreement for human scores.
3. The model does not reliably detect negation reversal or unrelated
   text in the sanity checks.
4. Language and domain breakdowns have limited sample sizes.
5. Historical baseline metrics should be identified as previously
   recorded results.
6. The model should not be used for autonomous legal decisions.

## 9. Conclusion

The XLM-R paired-text model is a promising prototype for legal meaning
preservation scoring in English and Hindi. It improves on the recorded
TF-IDF baseline in several error and correlation metrics, but not in
Spearman correlation. Sanity checks reveal important weaknesses in
distinguishing legal meaning preservation from surface similarity.

## 10. Future work

- Add human-annotated contrastive pairs for negation, obligations,
  permissions, deadlines, exceptions, and scope.
- Expand the dataset and include additional unseen legal sources.
- Establish independent human review and annotation agreement.
- Investigate contradiction-sensitive or ordinal objectives under a
  properly separated evaluation protocol.
"""

OUT.write_text(report, encoding="utf-8")
print("Report saved to:", OUT)
print("Size:", OUT.stat().st_size, "bytes")

Report saved to: /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_Final_Results_Report.md
Size: 7834 bytes


In [ ]:
from pathlib import Path

report_path = Path(
    "/content/drive/MyDrive/LegalDelta_LMP/LegalDelta_Final_Results_Report.md"
)

text = report_path.read_text(encoding="utf-8")

required_sections = [
    "Objective",
    "Dataset and evaluation",
    "Overall model comparison",
    "Language-wise results",
    "Domain-wise results",
    "Sanity checks",
    "Difficult-case analysis",
    "Limitations",
    "Conclusion",
    "Future work",
]

for section in required_sections:
    print(("OK" if section in text else "MISSING"), section)

print("\nReport size:", len(text), "characters")

OK Objective
OK Dataset and evaluation
OK Overall model comparison
OK Language-wise results
OK Domain-wise results
OK Sanity checks
OK Difficult-case analysis
OK Limitations
OK Conclusion
OK Future work

Report size: 7832 characters


In [ ]:
!pip -q install python-docx

In [ ]:
from pathlib import Path
from docx import Document
from docx.shared import Pt

base = Path("/content/drive/MyDrive/LegalDelta_LMP")
source = base / "LegalDelta_Final_Results_Report.md"
output = base / "LegalDelta_Final_Results_Report.docx"

lines = source.read_text(encoding="utf-8").splitlines()

doc = Document()
doc.styles["Normal"].font.name = "Arial"
doc.styles["Normal"].font.size = Pt(10)

for line in lines:
    line = line.strip()

    if not line:
        continue
    elif line.startswith("# "):
        doc.add_heading(line[2:], level=0)
    elif line.startswith("## "):
        doc.add_heading(line[3:], level=1)
    elif line.startswith("|"):
        # Keep Markdown tables readable as text for now.
        doc.add_paragraph(line)
    elif line.startswith("- "):
        doc.add_paragraph(line[2:], style="List Bullet")
    else:
        doc.add_paragraph(line)

doc.save(output)

print("Word report saved:", output)
print("File size:", output.stat().st_size, "bytes")

Word report saved: /content/drive/MyDrive/LegalDelta_LMP/LegalDelta_Final_Results_Report.docx
File size: 39459 bytes


In [ ]:

import os

drive = "/content/drive/MyDrive/LegalDelta_LMP"
artifacts = os.path.join(drive, "LegalDelta_improved_candidate")

print("=== SAVED MODEL ARTIFACTS ===")
for root, dirs, files in os.walk(artifacts):
    for name in files:
        path = os.path.join(root, name)
        print(f"{os.path.relpath(path, drive)}  ({os.path.getsize(path) / (1024**2):.1f} MB)")

print("\n=== COLAB NOTEBOOKS AND PYTHON FILES ===")
for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in {
        ".git", "site-packages", "__pycache__", ".cache"
    }]
    for name in files:
        if name.endswith((".ipynb", ".py")) and (
            "legal" in name.lower()
            or "lmp" in name.lower()
            or "train" in name.lower()
            or "model" in name.lower()
        ):
            print(os.path.join(root, name))


=== SAVED MODEL ARTIFACTS ===
LegalDelta_improved_candidate/best_model_state.pt  (1061.5 MB)
LegalDelta_improved_candidate/training_history.csv  (0.0 MB)
LegalDelta_improved_candidate/metadata.json  (0.0 MB)
LegalDelta_improved_candidate/test_predictions.csv  (0.0 MB)
LegalDelta_improved_candidate/test_metrics.json  (0.0 MB)
LegalDelta_improved_candidate/language_wise_test_metrics.csv  (0.0 MB)
LegalDelta_improved_candidate/domain_wise_test_metrics.csv  (0.0 MB)
LegalDelta_improved_candidate/15_high_error_cases.csv  (0.0 MB)
LegalDelta_improved_candidate/sanity_check_results.csv  (0.0 MB)
LegalDelta_improved_candidate/tokenizer/tokenizer_config.json  (0.0 MB)
LegalDelta_improved_candidate/tokenizer/tokenizer.json  (16.3 MB)
LegalDelta_improved_candidate/encoder_config/config.json  (0.0 MB)

=== COLAB NOTEBOOKS AND PYTHON FILES ===
/content/multilingual-legal-meaning-preservation/notebooks/LMP_Baseline_Experiments.ipynb
/content/multilingual-legal-meaning-preservation/notebooks/LMP_resu